# 03 — Causal Trade-to-Book Alignment

## Notebook purpose

This notebook performs the V0.1 causal synchronization step between the authoritative raw trade stream and the V0.1 reconstructed visible book states.

The only primary task is:

> For each raw trade, identify the latest visible book state that was already observable to the collector before the trade arrived.

The resulting aligned trade-book table becomes the authorized input for Notebook 04 event-stream construction.

---

## Current V0.1 stage

**Project:** The Clown Project  
**Pipeline:** V0.1 causal market-data reconstruction pipeline  
**Notebook:** `03_CAUSAL_TRADE_BOOK_ALIGNMENT.ipynb`  
**Previous notebook:** `02_VISIBLE_BOOK_RECONSTRUCTION.ipynb`  
**Next notebook:** `04_EVENT_STREAM_CONSTRUCTION.ipynb`

**Source run prefix:** `BTCUSDT_spot_20260710T063746Z_c8b5bf12`  
**V0.1 run ID:** `v0_1_20260714T090616Z_e82325081a81`  
**Primary ordering authority:** `collector_sequence`  
**Canonical timezone:** UTC  
**Operating mode:** `ENGINEERING_REPRODUCTION_MODE`

---

## Upstream authority accepted before this notebook

Notebook 01 authorized:

- immutable V0.0 raw trade stream
- immutable V0.0 raw depth stream
- REST snapshot bridge
- collector-sequence ordering
- UTC timestamp conversion
- trade identity
- aggressor-side semantics
- chronological engineering partitions

Notebook 02 authorized:

- visible top-ten market-by-price reconstruction
- reconstructed book states
- top-10 visible book wide table
- top-10 visible book long table
- update-continuity report
- book-quality gates
- Notebook 03 handoff

Notebook 02 terminal status:

`PASS_WITH_REFERENCE_WARNINGS`

The warnings were reference-comparison warnings only. They do not block Notebook 03.

---

## Primary alignment policy

The primary synchronization rule is:

`LOCAL_STRICT`

For a trade to be matched to a book state, both conditions must hold:

1. `book_collector_sequence < trade_collector_sequence`
2. `book_local_receipt_time_ns <= trade_local_receipt_time_ns`

Among all eligible book states, this notebook selects the one with the greatest `book_collector_sequence`.

This gives the latest visible book state known to the collector before the trade.

No future book state is allowed.

No timestamp-only shortcut is allowed.

No V0.0 synchronized table may replace the V0.1 reconstruction.

---

## Required inputs

This notebook must start from saved disk inputs only.

Required inputs:

- Notebook 02 output manifest
- Notebook 02 to Notebook 03 handoff
- V0.1 reconstructed book states
- V0.1 top-10 visible book wide table
- V0.1 update-continuity report
- immutable V0.0 raw trade stream
- Notebook 01 raw-data audit outputs
- current paths and authority map

Expected current counts:

- raw trade records: 67,683
- reconstructed book states: 35,985
- top-10 wide book rows: 35,985
- top-10 long book rows: 719,700

---

## Required outputs

Authoritative Notebook 03 outputs:

- all-trades LOCAL_STRICT alignment table
- matched LOCAL_STRICT trade-book table
- unmatched-trades ledger
- match-lag distribution
- staleness audit
- sequence-order audit
- price-versus-book consistency audit
- aggressor-versus-book audit
- partition alignment summary
- boundary alignment audit
- EXCHANGE_STRICT sensitivity table
- V0.0 synchronization reference reconciliation
- Notebook 03 output manifest
- Notebook 03 to Notebook 04 handoff

---

## Non-goals

This notebook must not:

- aggregate trades into events
- construct Hawkes event arrays
- build market-state feature tables
- fit Poisson models
- fit Hawkes models
- test Hawkes superiority
- compute quote logic
- simulate fills
- compute inventory
- compute cash or P&L
- compute Sharpe ratio
- compute maximum drawdown
- make any market-making strategy claim

Those tasks belong to later notebooks.

---

## Blocking failure conditions

This notebook must fail if any of the following occur:

- required upstream artifact is missing
- upstream hash or row-count verification fails
- raw trade count is not conserved
- duplicate trade IDs appear
- duplicate trade collector sequences appear
- any matched trade uses a future book state
- any matched trade violates `book_collector_sequence < trade_collector_sequence`
- any matched trade violates `book_local_receipt_time_ns <= trade_local_receipt_time_ns`
- any local observation lag is negative
- unmatched trades are not fully classified
- matched plus unmatched trades do not equal the raw trade count
- V0.0 synchronized data is used as authority
- output read-back verification fails
- output manifest or handoff cannot be verified

---

## Expected terminal status

The expected successful terminal status is one of:

- `PASS`
- `PASS_WITH_ALIGNMENT_WARNINGS`
- `PASS_WITH_REFERENCE_WARNINGS`

A clean-looking table is not the goal.

A causally valid, auditable, row-conserving alignment is the goal.

In [1]:
# ============================================================
# 03_CAUSAL_TRADE_BOOK_ALIGNMENT
# Cell 02 — Imports, fixed run identity, and audit helpers
# ============================================================

from __future__ import annotations

import hashlib
import json
import math
import os
import platform
import sys
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Display and pandas behavior
# ------------------------------------------------------------

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 160)

try:
    pd.options.mode.copy_on_write = True
except Exception:
    pass


# ------------------------------------------------------------
# Fixed notebook identity
# ------------------------------------------------------------

NOTEBOOK_NAME = "03_CAUSAL_TRADE_BOOK_ALIGNMENT"
NOTEBOOK_FILENAME = f"{NOTEBOOK_NAME}.ipynb"

PROJECT_NAME = "The Clown Project"
PIPELINE_VERSION = "V0.1"
OPERATING_MODE = "ENGINEERING_REPRODUCTION_MODE"

SOURCE_RUN_PREFIX = "BTCUSDT_spot_20260710T063746Z_c8b5bf12"
V01_RUN_ID = "v0_1_20260714T090616Z_e82325081a81"
COMBINED_PREFIX = f"{SOURCE_RUN_PREFIX}__{V01_RUN_ID}"

PRIMARY_ALIGNMENT_POLICY = "LOCAL_STRICT"
SENSITIVITY_ALIGNMENT_POLICY = "EXCHANGE_STRICT"

SYMBOL = "BTCUSDT"
CANONICAL_TIMEZONE = "UTC"
PRIMARY_ORDERING_AUTHORITY = "collector_sequence"


# ------------------------------------------------------------
# Fixed expected counts from accepted upstream authority
# ------------------------------------------------------------

EXPECTED_RAW_TRADE_RECORDS = 67_683
EXPECTED_RAW_DEPTH_RECORDS = 35_994
EXPECTED_RECONSTRUCTED_BOOK_STATES = 35_985
EXPECTED_TOP10_WIDE_ROWS = 35_985
EXPECTED_TOP10_LONG_ROWS = 719_700

EXPECTED_FIRST_RECONSTRUCTION_DEPTH_SEQUENCE = 10
EXPECTED_LAST_RECONSTRUCTION_DEPTH_SEQUENCE = 103_677
EXPECTED_FIRST_RECONSTRUCTION_FINAL_UPDATE_ID = 97_233_590_170
EXPECTED_LAST_RECONSTRUCTION_FINAL_UPDATE_ID = 97_234_812_218


# ------------------------------------------------------------
# Fixed upstream hashes from current authority map
# ------------------------------------------------------------

EXPECTED_SOURCE_SET_SHA256 = (
    "132c83531eec615d279408b5c06f402973114ba3058dfadd2fe58e2e67184c4b"
)

EXPECTED_NOTEBOOK_01_RAW_AUDIT_PAYLOAD_SHA256 = (
    "fdfd1bc3ca907b1bf14286ca8f849d49a997bd5655f8793260838a34b1c9f56d"
)

EXPECTED_NOTEBOOK_01_TO_02_HANDOFF_PAYLOAD_SHA256 = (
    "80f3e2df95b6f2daebdf7b85324cbfcabdb64d22740e328962d1af078420b2e1"
)

EXPECTED_NOTEBOOK_02_OUTPUT_MANIFEST_SHA256 = (
    "0150ed8386326d30017ed892d82a8980f9c8a3d9ba13fb0e7bc2d01d055a2f8e"
)

EXPECTED_NOTEBOOK_02_TO_03_HANDOFF_PAYLOAD_SHA256 = (
    "b13e9c47c46e13cb698aff5497b3cbd12f0e0030368fa755f260a7ee3ed8fe76"
)

EXPECTED_V00_RECONSTRUCTED_BOOK_REFERENCE_SHA256 = (
    "a59fa78b0a3ae895b4b4f5b36267b8510957e3cf8f0e6bd965402c01bbf3a860"
)


# ------------------------------------------------------------
# Project roots
# ------------------------------------------------------------

PROJECT_ROOT = Path(r"D:\Clown Project")
V00_ROOT = PROJECT_ROOT / "V0.0"
V01_ROOT = PROJECT_ROOT / "V0.1"

V00_RAW_ROOT = V00_ROOT / "data" / "raw"
V01_DATA_ROOT = V01_ROOT / "data"
V01_ARTIFACTS_ROOT = V01_ROOT / "artifacts"

V01_PROCESSED_BOOK_DIR = V01_DATA_ROOT / "processed" / "book"
V01_MANIFEST_DIR = V01_ARTIFACTS_ROOT / "manifests"
V01_HANDOFF_DIR = V01_ARTIFACTS_ROOT / "handoff"
V01_AUDIT_TABLE_DIR = V01_ARTIFACTS_ROOT / "audit_tables" / NOTEBOOK_NAME
V01_RECONCILIATION_DIR = V01_ARTIFACTS_ROOT / "reconciliation" / NOTEBOOK_NAME


# ------------------------------------------------------------
# Raw input paths
# ------------------------------------------------------------

RAW_TRADE_JSONL_PATH = (
    V00_RAW_ROOT
    / "trades"
    / f"{SOURCE_RUN_PREFIX}_trades.jsonl"
)

RAW_DEPTH_JSONL_PATH = (
    V00_RAW_ROOT
    / "order_book"
    / f"{SOURCE_RUN_PREFIX}_depth_updates.jsonl"
)

RAW_SNAPSHOT_JSON_PATH = (
    V00_RAW_ROOT
    / "order_book"
    / f"{SOURCE_RUN_PREFIX}_snapshot.json"
)

RAW_SESSION_METADATA_PATH = (
    V00_RAW_ROOT
    / "metadata"
    / f"{SOURCE_RUN_PREFIX}_session.json"
)

RAW_DEVELOPMENT_MANIFEST_PATH = (
    V00_RAW_ROOT
    / "metadata"
    / f"{SOURCE_RUN_PREFIX}_development_manifest.json"
)


# ------------------------------------------------------------
# Upstream V0.1 input paths
# ------------------------------------------------------------

NOTEBOOK_01_RAW_DATA_AUDIT_PATH = (
    V01_MANIFEST_DIR
    / f"{COMBINED_PREFIX}__01_RAW_DATA_AUDIT__v0_1_raw_data_audit.json"
)

NOTEBOOK_02_OUTPUT_MANIFEST_PATH = (
    V01_MANIFEST_DIR
    / f"{COMBINED_PREFIX}__02_VISIBLE_BOOK_RECONSTRUCTION__notebook_02_output_manifest.json"
)

NOTEBOOK_02_TO_03_HANDOFF_PATH = (
    V01_HANDOFF_DIR
    / f"{COMBINED_PREFIX}__02_VISIBLE_BOOK_RECONSTRUCTION__notebook_02_to_notebook_03_handoff.json"
)

RECONSTRUCTED_BOOK_STATES_PATH = (
    V01_PROCESSED_BOOK_DIR
    / f"{COMBINED_PREFIX}__02_VISIBLE_BOOK_RECONSTRUCTION__reconstructed_book_states.csv"
)

TOP10_VISIBLE_BOOK_WIDE_PATH = (
    V01_PROCESSED_BOOK_DIR
    / f"{COMBINED_PREFIX}__02_VISIBLE_BOOK_RECONSTRUCTION__top_10_visible_book_wide.csv"
)

TOP10_VISIBLE_BOOK_LONG_PATH = (
    V01_PROCESSED_BOOK_DIR
    / f"{COMBINED_PREFIX}__02_VISIBLE_BOOK_RECONSTRUCTION__top_10_visible_book_long.csv"
)

NOTEBOOK_02_UPDATE_CONTINUITY_REPORT_PATH = (
    V01_ARTIFACTS_ROOT
    / "audit_tables"
    / "02_VISIBLE_BOOK_RECONSTRUCTION"
    / f"{COMBINED_PREFIX}__02_VISIBLE_BOOK_RECONSTRUCTION__update_continuity_report.csv"
)


# ------------------------------------------------------------
# Notebook 03 output paths
# ------------------------------------------------------------

LOCAL_STRICT_ALL_ALIGNMENT_PATH = (
    V01_DATA_ROOT
    / "processed"
    / "trade_book_alignment"
    / f"{COMBINED_PREFIX}__{NOTEBOOK_NAME}__local_strict_all_trades_alignment.csv"
)

LOCAL_STRICT_MATCHED_ALIGNMENT_PATH = (
    V01_DATA_ROOT
    / "processed"
    / "trade_book_alignment"
    / f"{COMBINED_PREFIX}__{NOTEBOOK_NAME}__local_strict_matched_trade_book.csv"
)

UNMATCHED_TRADES_PATH = (
    V01_AUDIT_TABLE_DIR
    / f"{COMBINED_PREFIX}__{NOTEBOOK_NAME}__unmatched_trades.csv"
)

MATCH_LAG_DISTRIBUTION_PATH = (
    V01_AUDIT_TABLE_DIR
    / f"{COMBINED_PREFIX}__{NOTEBOOK_NAME}__match_lag_distribution.csv"
)

STALENESS_AUDIT_PATH = (
    V01_AUDIT_TABLE_DIR
    / f"{COMBINED_PREFIX}__{NOTEBOOK_NAME}__staleness_audit.csv"
)

SEQUENCE_ORDER_AUDIT_PATH = (
    V01_AUDIT_TABLE_DIR
    / f"{COMBINED_PREFIX}__{NOTEBOOK_NAME}__sequence_order_audit.csv"
)

PRICE_VS_BOOK_CONSISTENCY_AUDIT_PATH = (
    V01_AUDIT_TABLE_DIR
    / f"{COMBINED_PREFIX}__{NOTEBOOK_NAME}__price_vs_book_consistency_audit.csv"
)

AGGRESSOR_VS_BOOK_AUDIT_PATH = (
    V01_AUDIT_TABLE_DIR
    / f"{COMBINED_PREFIX}__{NOTEBOOK_NAME}__aggressor_vs_book_audit.csv"
)

PARTITION_ALIGNMENT_SUMMARY_PATH = (
    V01_AUDIT_TABLE_DIR
    / f"{COMBINED_PREFIX}__{NOTEBOOK_NAME}__partition_alignment_summary.csv"
)

BOUNDARY_ALIGNMENT_AUDIT_PATH = (
    V01_AUDIT_TABLE_DIR
    / f"{COMBINED_PREFIX}__{NOTEBOOK_NAME}__boundary_alignment_audit.csv"
)

EXCHANGE_STRICT_SENSITIVITY_PATH = (
    V01_AUDIT_TABLE_DIR
    / f"{COMBINED_PREFIX}__{NOTEBOOK_NAME}__exchange_strict_alignment_sensitivity.csv"
)

V00_SYNC_RECONCILIATION_SUMMARY_PATH = (
    V01_RECONCILIATION_DIR
    / f"{COMBINED_PREFIX}__{NOTEBOOK_NAME}__v0_0_synchronization_reconciliation_summary.csv"
)

V00_SYNC_FIELD_RECONCILIATION_PATH = (
    V01_RECONCILIATION_DIR
    / f"{COMBINED_PREFIX}__{NOTEBOOK_NAME}__v0_0_synchronization_field_reconciliation.csv"
)

V00_SYNC_REFERENCE_FINDINGS_PATH = (
    V01_RECONCILIATION_DIR
    / f"{COMBINED_PREFIX}__{NOTEBOOK_NAME}__v0_0_synchronization_reference_findings.csv"
)

NOTEBOOK_03_OUTPUT_MANIFEST_PATH = (
    V01_MANIFEST_DIR
    / f"{COMBINED_PREFIX}__{NOTEBOOK_NAME}__notebook_03_output_manifest.json"
)

NOTEBOOK_03_TO_04_HANDOFF_PATH = (
    V01_HANDOFF_DIR
    / f"{COMBINED_PREFIX}__{NOTEBOOK_NAME}__notebook_03_to_notebook_04_handoff.json"
)


# ------------------------------------------------------------
# Small audit helpers
# ------------------------------------------------------------

def utc_now_iso() -> str:
    """Return current UTC timestamp as an ISO-8601 string."""
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path: Path, chunk_size: int = 1 << 20) -> str:
    """Compute SHA-256 for a file without loading the whole file into memory."""
    path = Path(path)

    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            digest.update(chunk)

    return digest.hexdigest()


def read_json_file(path: Path) -> dict[str, Any]:
    """Read a JSON file and require a mapping at the top level."""
    path = Path(path)

    with path.open("r", encoding="utf-8") as file:
        payload = json.load(file)

    if not isinstance(payload, dict):
        raise TypeError(f"JSON top level must be an object: {path}")

    return payload


def write_json_file(payload: dict[str, Any], path: Path) -> None:
    """Write a JSON object using deterministic formatting."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8", newline="\n") as file:
        json.dump(payload, file, indent=2, sort_keys=True, ensure_ascii=False)
        file.write("\n")


def require(condition: bool, message: str) -> None:
    """Raise immediately on a blocking invariant failure."""
    if not bool(condition):
        raise AssertionError(message)


def require_file(path: Path) -> Path:
    """Require that a path exists and is a file."""
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Required file does not exist: {path}")

    if not path.is_file():
        raise FileNotFoundError(f"Required path is not a file: {path}")

    return path


def require_columns(frame: pd.DataFrame, required_columns: list[str], frame_name: str) -> None:
    """Require that a DataFrame contains all required columns."""
    missing = [column for column in required_columns if column not in frame.columns]
    require(not missing, f"{frame_name} is missing required columns: {missing}")


def summarize_path(path: Path) -> dict[str, Any]:
    """Return minimal file metadata for audit manifests."""
    path = require_file(path)

    return {
        "path": str(path),
        "exists": True,
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }


@dataclass(frozen=True)
class GateResult:
    gate: str
    status: str
    severity: str
    detail: str


def make_gate(gate: str, passed: bool, detail: str, severity: str = "BLOCKING") -> GateResult:
    """Create a normalized gate result row."""
    return GateResult(
        gate=gate,
        status="PASS" if passed else "FAIL",
        severity=severity,
        detail=detail,
    )


def gate_results_to_frame(gates: list[GateResult]) -> pd.DataFrame:
    """Convert gate results to a stable DataFrame."""
    return pd.DataFrame(
        [
            {
                "gate": gate.gate,
                "status": gate.status,
                "severity": gate.severity,
                "detail": gate.detail,
            }
            for gate in gates
        ]
    )


def fail_if_blocking_gate_failed(gate_frame: pd.DataFrame) -> None:
    """Stop execution if any blocking gate failed."""
    require_columns(
        gate_frame,
        ["gate", "status", "severity", "detail"],
        "gate_frame",
    )

    failed_blocking = gate_frame[
        (gate_frame["severity"] == "BLOCKING")
        & (gate_frame["status"] != "PASS")
    ]

    if not failed_blocking.empty:
        display(failed_blocking)
        raise AssertionError("At least one blocking gate failed.")


# ------------------------------------------------------------
# Runtime record
# ------------------------------------------------------------

runtime_record = {
    "project_name": PROJECT_NAME,
    "pipeline_version": PIPELINE_VERSION,
    "notebook_name": NOTEBOOK_NAME,
    "notebook_filename": NOTEBOOK_FILENAME,
    "source_run_prefix": SOURCE_RUN_PREFIX,
    "v0_1_run_id": V01_RUN_ID,
    "combined_prefix": COMBINED_PREFIX,
    "primary_alignment_policy": PRIMARY_ALIGNMENT_POLICY,
    "sensitivity_alignment_policy": SENSITIVITY_ALIGNMENT_POLICY,
    "primary_ordering_authority": PRIMARY_ORDERING_AUTHORITY,
    "canonical_timezone": CANONICAL_TIMEZONE,
    "operating_mode": OPERATING_MODE,
    "started_at_utc": utc_now_iso(),
    "python_version": sys.version,
    "python_executable": sys.executable,
    "platform": platform.platform(),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "working_directory": str(Path.cwd()),
}

runtime_record

{'project_name': 'The Clown Project',
 'pipeline_version': 'V0.1',
 'notebook_name': '03_CAUSAL_TRADE_BOOK_ALIGNMENT',
 'notebook_filename': '03_CAUSAL_TRADE_BOOK_ALIGNMENT.ipynb',
 'source_run_prefix': 'BTCUSDT_spot_20260710T063746Z_c8b5bf12',
 'v0_1_run_id': 'v0_1_20260714T090616Z_e82325081a81',
 'combined_prefix': 'BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81',
 'primary_alignment_policy': 'LOCAL_STRICT',
 'sensitivity_alignment_policy': 'EXCHANGE_STRICT',
 'primary_ordering_authority': 'collector_sequence',
 'canonical_timezone': 'UTC',
 'operating_mode': 'ENGINEERING_REPRODUCTION_MODE',
 'started_at_utc': '2026-07-18T06:47:28.411654+00:00',
 'python_version': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]',
 'python_executable': 'd:\\Sylva\\sylva\\.venv\\Scripts\\python.exe',
 'platform': 'Windows-10-10.0.26200-SP0',
 'pandas_version': '2.3.3',
 'numpy_version': '2.4.6',
 'working_directory': 'd:\\Clown Project\\V0.1

In [2]:
# ============================================================
# Cell 03 — Create output directories and verify required paths
# ============================================================

# ------------------------------------------------------------
# Output directories
# ------------------------------------------------------------

REQUIRED_OUTPUT_DIRECTORIES = [
    LOCAL_STRICT_ALL_ALIGNMENT_PATH.parent,
    LOCAL_STRICT_MATCHED_ALIGNMENT_PATH.parent,
    V01_AUDIT_TABLE_DIR,
    V01_RECONCILIATION_DIR,
    V01_MANIFEST_DIR,
    V01_HANDOFF_DIR,
]

for directory in REQUIRED_OUTPUT_DIRECTORIES:
    directory.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Required upstream input files
# ------------------------------------------------------------

required_input_contracts = [
    {
        "logical_name": "raw_trade_jsonl",
        "path": RAW_TRADE_JSONL_PATH,
        "expected_sha256": None,
        "blocking": True,
        "role": "PRIMARY_RAW_AUTHORITY",
    },
    {
        "logical_name": "raw_depth_jsonl",
        "path": RAW_DEPTH_JSONL_PATH,
        "expected_sha256": None,
        "blocking": True,
        "role": "PRIMARY_RAW_AUTHORITY_REFERENCE",
    },
    {
        "logical_name": "raw_snapshot_json",
        "path": RAW_SNAPSHOT_JSON_PATH,
        "expected_sha256": None,
        "blocking": True,
        "role": "PRIMARY_RAW_AUTHORITY_REFERENCE",
    },
    {
        "logical_name": "raw_session_metadata",
        "path": RAW_SESSION_METADATA_PATH,
        "expected_sha256": None,
        "blocking": True,
        "role": "PRIMARY_RAW_AUTHORITY_REFERENCE",
    },
    {
        "logical_name": "raw_development_manifest",
        "path": RAW_DEVELOPMENT_MANIFEST_PATH,
        "expected_sha256": None,
        "blocking": True,
        "role": "PRIMARY_RAW_AUTHORITY_REFERENCE",
    },
    {
        "logical_name": "notebook_01_raw_data_audit",
        "path": NOTEBOOK_01_RAW_DATA_AUDIT_PATH,
        "expected_sha256": EXPECTED_NOTEBOOK_01_RAW_AUDIT_PAYLOAD_SHA256,
        "blocking": True,
        "role": "V01_UPSTREAM_AUDIT",
    },
    {
        "logical_name": "notebook_02_output_manifest",
        "path": NOTEBOOK_02_OUTPUT_MANIFEST_PATH,
        "expected_sha256": EXPECTED_NOTEBOOK_02_OUTPUT_MANIFEST_SHA256,
        "blocking": True,
        "role": "V01_UPSTREAM_MANIFEST",
    },
    {
        "logical_name": "notebook_02_to_03_handoff",
        "path": NOTEBOOK_02_TO_03_HANDOFF_PATH,
        "expected_sha256": EXPECTED_NOTEBOOK_02_TO_03_HANDOFF_PAYLOAD_SHA256,
        "blocking": True,
        "role": "V01_UPSTREAM_HANDOFF",
    },
    {
        "logical_name": "reconstructed_book_states",
        "path": RECONSTRUCTED_BOOK_STATES_PATH,
        "expected_sha256": None,
        "blocking": True,
        "role": "V01_BOOK_AUTHORITY",
    },
    {
        "logical_name": "top10_visible_book_wide",
        "path": TOP10_VISIBLE_BOOK_WIDE_PATH,
        "expected_sha256": None,
        "blocking": True,
        "role": "V01_BOOK_AUTHORITY",
    },
    {
        "logical_name": "top10_visible_book_long",
        "path": TOP10_VISIBLE_BOOK_LONG_PATH,
        "expected_sha256": None,
        "blocking": True,
        "role": "V01_BOOK_AUTHORITY_REFERENCE",
    },
    {
        "logical_name": "notebook_02_update_continuity_report",
        "path": NOTEBOOK_02_UPDATE_CONTINUITY_REPORT_PATH,
        "expected_sha256": None,
        "blocking": True,
        "role": "V01_BOOK_AUDIT",
    },
]


# ------------------------------------------------------------
# Hash verification helper
# ------------------------------------------------------------

def file_text_contains_hash(path: Path, expected_hash: str) -> bool:
    """
    Return True if a declared hash appears inside a text-readable file.

    This is intentionally used for payload-hash contracts where the authority map
    may record a payload hash stored inside the artifact rather than the raw file
    byte hash.
    """
    try:
        text = Path(path).read_text(encoding="utf-8", errors="ignore")
    except Exception:
        return False

    return expected_hash in text


def verify_input_contract(contract: dict[str, Any]) -> dict[str, Any]:
    """Verify existence, file metadata, and available hash contract for one input."""
    logical_name = contract["logical_name"]
    path = Path(contract["path"])
    expected_sha256 = contract.get("expected_sha256")
    blocking = bool(contract.get("blocking", True))
    role = contract.get("role", "UNSPECIFIED")

    exists = path.exists()
    is_file = path.is_file() if exists else False

    observed_sha256 = None
    size_bytes = None
    hash_contract_status = "NOT_APPLICABLE"

    if exists and is_file:
        size_bytes = path.stat().st_size
        observed_sha256 = sha256_file(path)

        if expected_sha256 is None:
            hash_contract_status = "NO_EXPECTED_HASH_PROVIDED"
        elif observed_sha256 == expected_sha256:
            hash_contract_status = "MATCHED_FILE_SHA256"
        elif file_text_contains_hash(path, expected_sha256):
            hash_contract_status = "EXPECTED_HASH_DECLARED_INSIDE_FILE"
        else:
            hash_contract_status = "HASH_MISMATCH"

    path_status = "PASS" if exists and is_file else "FAIL"

    if expected_sha256 is None:
        hash_status = "PASS"
    else:
        hash_status = (
            "PASS"
            if hash_contract_status in {
                "MATCHED_FILE_SHA256",
                "EXPECTED_HASH_DECLARED_INSIDE_FILE",
            }
            else "FAIL"
        )

    overall_status = "PASS" if path_status == "PASS" and hash_status == "PASS" else "FAIL"

    return {
        "logical_name": logical_name,
        "role": role,
        "path": str(path),
        "exists": exists,
        "is_file": is_file,
        "size_bytes": size_bytes,
        "expected_sha256": expected_sha256,
        "observed_file_sha256": observed_sha256,
        "hash_contract_status": hash_contract_status,
        "path_status": path_status,
        "hash_status": hash_status,
        "overall_status": overall_status,
        "blocking": blocking,
    }


input_contract_audit = pd.DataFrame(
    [verify_input_contract(contract) for contract in required_input_contracts]
)


# ------------------------------------------------------------
# Blocking gate evaluation
# ------------------------------------------------------------

input_contract_gates = []

for _, row in input_contract_audit.iterrows():
    input_contract_gates.append(
        make_gate(
            gate=f"required_input_available::{row['logical_name']}",
            passed=(row["path_status"] == "PASS"),
            severity="BLOCKING" if row["blocking"] else "WARNING",
            detail=f"{row['role']} | {row['path']}",
        )
    )

    input_contract_gates.append(
        make_gate(
            gate=f"hash_contract_verified::{row['logical_name']}",
            passed=(row["hash_status"] == "PASS"),
            severity="BLOCKING" if row["blocking"] else "WARNING",
            detail=(
                f"hash_contract_status={row['hash_contract_status']} | "
                f"expected={row['expected_sha256']} | "
                f"observed_file_sha256={row['observed_file_sha256']}"
            ),
        )
    )

input_contract_gate_frame = gate_results_to_frame(input_contract_gates)

fail_if_blocking_gate_failed(input_contract_gate_frame)


# ------------------------------------------------------------
# Compact display
# ------------------------------------------------------------

display_columns = [
    "logical_name",
    "role",
    "exists",
    "is_file",
    "size_bytes",
    "hash_contract_status",
    "overall_status",
]

input_contract_audit[display_columns]

,logical_name,role,exists,is_file,size_bytes,hash_contract_status,overall_status
0,raw_trade_jsonl,PRIMARY_RAW_AUTHORITY,True,True,46082102,NO_EXPECTED_HASH_PROVIDED,PASS
1,raw_depth_jsonl,PRIMARY_RAW_AUTHORITY_REFERENCE,True,True,63443258,NO_EXPECTED_HASH_PROVIDED,PASS
2,raw_snapshot_json,PRIMARY_RAW_AUTHORITY_REFERENCE,True,True,360812,NO_EXPECTED_HASH_PROVIDED,PASS
3,raw_session_metadata,PRIMARY_RAW_AUTHORITY_REFERENCE,True,True,1777,NO_EXPECTED_HASH_PROVIDED,PASS
4,raw_development_manifest,PRIMARY_RAW_AUTHORITY_REFERENCE,True,True,1738,NO_EXPECTED_HASH_PROVIDED,PASS
5,notebook_01_raw_data_audit,V01_UPSTREAM_AUDIT,True,True,70251,EXPECTED_HASH_DECLARED_INSIDE_FILE,PASS
6,notebook_02_output_manifest,V01_UPSTREAM_MANIFEST,True,True,15362,MATCHED_FILE_SHA256,PASS
7,notebook_02_to_03_handoff,V01_UPSTREAM_HANDOFF,True,True,4413,EXPECTED_HASH_DECLARED_INSIDE_FILE,PASS
8,reconstructed_book_states,V01_BOOK_AUTHORITY,True,True,50443026,NO_EXPECTED_HASH_PROVIDED,PASS
9,top10_visible_book_wide,V01_BOOK_AUTHORITY,True,True,29832139,NO_EXPECTED_HASH_PROVIDED,PASS


In [3]:
# ============================================================
# Cell 04 — Load upstream JSON payloads and verify authority
# ============================================================

# This cell intentionally separates:
# 1. hard blockers: JSON unreadable, wrong run identity, explicit upstream FAIL
# 2. warnings: payload hash stored under an unexpected key, optional fields absent
#
# The previous path/file hash audit already verified that the required files exist.
# This cell verifies the content without assuming a brittle manifest schema.


# ------------------------------------------------------------
# Generic recursive JSON helpers
# ------------------------------------------------------------

def walk_json(obj: Any, path: str = "$") -> list[tuple[str, Any]]:
    """Return all scalar/list/dict nodes in a JSON-like object with JSONPath-like labels."""
    rows: list[tuple[str, Any]] = [(path, obj)]

    if isinstance(obj, dict):
        for key, value in obj.items():
            rows.extend(walk_json(value, f"{path}.{key}"))
    elif isinstance(obj, list):
        for idx, value in enumerate(obj):
            rows.extend(walk_json(value, f"{path}[{idx}]"))

    return rows


def json_contains_value(obj: Any, expected: str) -> bool:
    """Return True if the expected string appears anywhere in the JSON object."""
    expected = str(expected)

    for _, value in walk_json(obj):
        if isinstance(value, str) and expected in value:
            return True
        if not isinstance(value, (dict, list)) and str(value) == expected:
            return True

    return False


def json_values_for_key_fragment(obj: Any, key_fragment: str) -> list[tuple[str, Any]]:
    """Return values whose key/path contains a case-insensitive fragment."""
    fragment = key_fragment.lower()
    matches: list[tuple[str, Any]] = []

    for path, value in walk_json(obj):
        last_key = path.split(".")[-1].lower()
        if fragment in last_key:
            matches.append((path, value))

    return matches


def flatten_json_scalars(obj: Any) -> pd.DataFrame:
    """Flatten scalar JSON values to a DataFrame for inspection."""
    rows = []

    for path, value in walk_json(obj):
        if not isinstance(value, (dict, list)):
            rows.append(
                {
                    "path": path,
                    "type": type(value).__name__,
                    "value": value,
                }
            )

    return pd.DataFrame(rows)


def first_present_scalar(obj: Any, key_fragments: list[str]) -> Any:
    """Find the first scalar value whose JSON path contains any requested key fragment."""
    for fragment in key_fragments:
        matches = json_values_for_key_fragment(obj, fragment)
        for _, value in matches:
            if not isinstance(value, (dict, list)):
                return value

    return None


def payload_has_explicit_fail(obj: Any) -> bool:
    """Detect explicit upstream failure markers without treating warning text as failure."""
    fail_tokens = {
        "FAIL",
        "FAILED",
        "CRITICAL_FAILURE",
        "CRITICAL FAIL",
        "BLOCKING_FAILURE",
        "BLOCKING FAIL",
    }

    status_like_rows = []

    for path, value in walk_json(obj):
        path_lower = path.lower()
        if any(token in path_lower for token in ["status", "result", "failure", "critical"]):
            status_like_rows.append((path, value))

    for _, value in status_like_rows:
        if isinstance(value, str):
            cleaned = value.strip().upper()
            if cleaned in fail_tokens:
                return True

        if isinstance(value, (int, float)) and not isinstance(value, bool):
            # Only critical/blocking failure counters are blockers.
            # Generic warning counters are not blockers.
            pass

    critical_count = first_present_scalar(
        obj,
        [
            "critical_failures",
            "critical_failure_count",
            "blocking_failures",
            "blocking_failure_count",
        ],
    )

    if critical_count is not None:
        try:
            return int(critical_count) > 0
        except Exception:
            return False

    return False


def payload_status_summary(obj: Any) -> dict[str, Any]:
    """Collect useful status-like fields from a payload without requiring a fixed schema."""
    status_rows = []

    for path, value in walk_json(obj):
        path_lower = path.lower()
        if any(fragment in path_lower for fragment in ["status", "result", "authorized", "authorization", "failure", "warning"]):
            if not isinstance(value, (dict, list)):
                status_rows.append({"path": path, "value": value})

    return {
        "explicit_fail_detected": payload_has_explicit_fail(obj),
        "status_like_field_count": len(status_rows),
        "status_like_preview": status_rows[:20],
    }


# ------------------------------------------------------------
# Load upstream JSON payloads
# ------------------------------------------------------------

upstream_json_payloads: dict[str, dict[str, Any]] = {
    "notebook_01_raw_data_audit": read_json_file(NOTEBOOK_01_RAW_DATA_AUDIT_PATH),
    "notebook_02_output_manifest": read_json_file(NOTEBOOK_02_OUTPUT_MANIFEST_PATH),
    "notebook_02_to_03_handoff": read_json_file(NOTEBOOK_02_TO_03_HANDOFF_PATH),
    "raw_session_metadata": read_json_file(RAW_SESSION_METADATA_PATH),
    "raw_development_manifest": read_json_file(RAW_DEVELOPMENT_MANIFEST_PATH),
}


# ------------------------------------------------------------
# Content-level gate checks
# ------------------------------------------------------------

upstream_payload_gates: list[GateResult] = []

for payload_name, payload in upstream_json_payloads.items():
    contains_source_prefix = json_contains_value(payload, SOURCE_RUN_PREFIX)
    contains_v01_run_id = json_contains_value(payload, V01_RUN_ID)
    explicit_fail_detected = payload_has_explicit_fail(payload)

    # Raw V0.0 metadata does not need to contain the V0.1 run ID.
    requires_v01_run_id = payload_name.startswith("notebook_")

    upstream_payload_gates.append(
        make_gate(
            gate=f"payload_loaded::{payload_name}",
            passed=isinstance(payload, dict),
            severity="BLOCKING",
            detail=f"{payload_name} loaded as dict with {len(payload)} top-level keys",
        )
    )

    upstream_payload_gates.append(
        make_gate(
            gate=f"source_run_prefix_present::{payload_name}",
            passed=contains_source_prefix,
            severity="BLOCKING",
            detail=f"expected source_run_prefix={SOURCE_RUN_PREFIX}",
        )
    )

    upstream_payload_gates.append(
        make_gate(
            gate=f"v01_run_id_present::{payload_name}",
            passed=(contains_v01_run_id if requires_v01_run_id else True),
            severity="BLOCKING" if requires_v01_run_id else "INFO",
            detail=(
                f"expected v0_1_run_id={V01_RUN_ID}"
                if requires_v01_run_id
                else "V0.0 raw metadata is not required to contain V0.1 run ID"
            ),
        )
    )

    upstream_payload_gates.append(
        make_gate(
            gate=f"no_explicit_upstream_fail::{payload_name}",
            passed=not explicit_fail_detected,
            severity="BLOCKING",
            detail=f"explicit_fail_detected={explicit_fail_detected}",
        )
    )


# ------------------------------------------------------------
# Hash declarations inside payloads
# ------------------------------------------------------------

declared_hash_audit_rows = []

expected_payload_hash_contracts = [
    {
        "payload_name": "notebook_01_raw_data_audit",
        "expected_hash": EXPECTED_NOTEBOOK_01_RAW_AUDIT_PAYLOAD_SHA256,
    },
    {
        "payload_name": "notebook_02_output_manifest",
        "expected_hash": EXPECTED_NOTEBOOK_02_OUTPUT_MANIFEST_SHA256,
    },
    {
        "payload_name": "notebook_02_to_03_handoff",
        "expected_hash": EXPECTED_NOTEBOOK_02_TO_03_HANDOFF_PAYLOAD_SHA256,
    },
]

for item in expected_payload_hash_contracts:
    payload_name = item["payload_name"]
    expected_hash = item["expected_hash"]
    payload = upstream_json_payloads[payload_name]
    file_path = {
        "notebook_01_raw_data_audit": NOTEBOOK_01_RAW_DATA_AUDIT_PATH,
        "notebook_02_output_manifest": NOTEBOOK_02_OUTPUT_MANIFEST_PATH,
        "notebook_02_to_03_handoff": NOTEBOOK_02_TO_03_HANDOFF_PATH,
    }[payload_name]

    observed_file_sha256 = sha256_file(file_path)
    hash_appears_in_payload = json_contains_value(payload, expected_hash)
    file_hash_matches = observed_file_sha256 == expected_hash

    if file_hash_matches:
        hash_contract_status = "MATCHED_FILE_SHA256"
    elif hash_appears_in_payload:
        hash_contract_status = "EXPECTED_HASH_DECLARED_INSIDE_PAYLOAD"
    else:
        hash_contract_status = "EXPECTED_HASH_NOT_FOUND_IN_PAYLOAD_OR_FILE_SHA256"

    declared_hash_audit_rows.append(
        {
            "payload_name": payload_name,
            "path": str(file_path),
            "expected_hash": expected_hash,
            "observed_file_sha256": observed_file_sha256,
            "file_hash_matches": file_hash_matches,
            "hash_appears_in_payload": hash_appears_in_payload,
            "hash_contract_status": hash_contract_status,
        }
    )

    upstream_payload_gates.append(
        make_gate(
            gate=f"payload_hash_contract_traceable::{payload_name}",
            passed=hash_contract_status in {
                "MATCHED_FILE_SHA256",
                "EXPECTED_HASH_DECLARED_INSIDE_PAYLOAD",
            },
            severity="WARNING",
            detail=hash_contract_status,
        )
    )

declared_hash_audit = pd.DataFrame(declared_hash_audit_rows)


# ------------------------------------------------------------
# Extract useful upstream summaries
# ------------------------------------------------------------

upstream_payload_summary_rows = []

for payload_name, payload in upstream_json_payloads.items():
    summary = payload_status_summary(payload)

    upstream_payload_summary_rows.append(
        {
            "payload_name": payload_name,
            "top_level_keys": len(payload),
            "contains_source_run_prefix": json_contains_value(payload, SOURCE_RUN_PREFIX),
            "contains_v01_run_id": json_contains_value(payload, V01_RUN_ID),
            "explicit_fail_detected": summary["explicit_fail_detected"],
            "status_like_field_count": summary["status_like_field_count"],
        }
    )

upstream_payload_summary = pd.DataFrame(upstream_payload_summary_rows)


# ------------------------------------------------------------
# Block only on genuine content-authority failures
# ------------------------------------------------------------

upstream_payload_gate_frame = gate_results_to_frame(upstream_payload_gates)

fail_if_blocking_gate_failed(upstream_payload_gate_frame)


# ------------------------------------------------------------
# Persist in-memory authority handles for later cells
# ------------------------------------------------------------

notebook_01_raw_data_audit_payload = upstream_json_payloads["notebook_01_raw_data_audit"]
notebook_02_output_manifest_payload = upstream_json_payloads["notebook_02_output_manifest"]
notebook_02_to_03_handoff_payload = upstream_json_payloads["notebook_02_to_03_handoff"]
raw_session_metadata_payload = upstream_json_payloads["raw_session_metadata"]
raw_development_manifest_payload = upstream_json_payloads["raw_development_manifest"]

upstream_content_audit = {
    "payload_summary": upstream_payload_summary.to_dict(orient="records"),
    "declared_hash_audit": declared_hash_audit.to_dict(orient="records"),
    "gate_summary": upstream_payload_gate_frame.to_dict(orient="records"),
}


# ------------------------------------------------------------
# Compact display
# ------------------------------------------------------------

display(upstream_payload_summary)
display(declared_hash_audit[["payload_name", "hash_contract_status", "file_hash_matches", "hash_appears_in_payload"]])

upstream_payload_gate_frame[
    ["gate", "status", "severity", "detail"]
].head(50)

,payload_name,top_level_keys,contains_source_run_prefix,contains_v01_run_id,explicit_fail_detected,status_like_field_count
0,notebook_01_raw_data_audit,2,True,True,False,91
1,notebook_02_output_manifest,2,True,True,False,30
2,notebook_02_to_03_handoff,2,True,True,False,5
3,raw_session_metadata,24,True,False,False,3
4,raw_development_manifest,17,True,False,False,2


,payload_name,hash_contract_status,file_hash_matches,hash_appears_in_payload
0,notebook_01_raw_data_audit,EXPECTED_HASH_DECLARED_INSIDE_PAYLOAD,False,True
1,notebook_02_output_manifest,MATCHED_FILE_SHA256,True,False
2,notebook_02_to_03_handoff,EXPECTED_HASH_DECLARED_INSIDE_PAYLOAD,False,True


,gate,status,severity,detail
0,payload_loaded::notebook_01_raw_data_audit,PASS,BLOCKING,notebook_01_raw_data_audit loaded as dict with 2 top-level keys
1,source_run_prefix_present::notebook_01_raw_data_audit,PASS,BLOCKING,expected source_run_prefix=BTCUSDT_spot_20260710T063746Z_c8b5bf12
2,v01_run_id_present::notebook_01_raw_data_audit,PASS,BLOCKING,expected v0_1_run_id=v0_1_20260714T090616Z_e82325081a81
3,no_explicit_upstream_fail::notebook_01_raw_data_audit,PASS,BLOCKING,explicit_fail_detected=False
4,payload_loaded::notebook_02_output_manifest,PASS,BLOCKING,notebook_02_output_manifest loaded as dict with 2 top-level keys
5,source_run_prefix_present::notebook_02_output_manifest,PASS,BLOCKING,expected source_run_prefix=BTCUSDT_spot_20260710T063746Z_c8b5bf12
6,v01_run_id_present::notebook_02_output_manifest,PASS,BLOCKING,expected v0_1_run_id=v0_1_20260714T090616Z_e82325081a81
7,no_explicit_upstream_fail::notebook_02_output_manifest,PASS,BLOCKING,explicit_fail_detected=False
8,payload_loaded::notebook_02_to_03_handoff,PASS,BLOCKING,notebook_02_to_03_handoff loaded as dict with 2 top-level keys
9,source_run_prefix_present::notebook_02_to_03_handoff,PASS,BLOCKING,expected source_run_prefix=BTCUSDT_spot_20260710T063746Z_c8b5bf12


In [6]:
# ============================================================
# Cell 05 — Load and canonicalize authoritative raw trades
# ============================================================

# This cell reads the raw trade JSONL directly from V0.0.
#
# Important schema point:
# The Binance trade fields may be nested inside collector wrappers such as:
#   data, message, payload, raw, event, ws_message, or similar.
#
# Therefore this cell does NOT assume p/q/t/T/m are top-level fields.
# It recursively discovers the trade payload and then extracts:
#   t = trade id
#   p = price
#   q = quantity
#   T = exchange trade time in ms
#   m = buyer_is_maker
#
# It also preserves collector ordering and local receipt time from the wrapper.


# ------------------------------------------------------------
# JSONL reader
# ------------------------------------------------------------

def read_jsonl_records(path: Path, expected_rows: int | None = None) -> list[dict[str, Any]]:
    """Read a JSONL file into a list of mapping records."""
    path = require_file(path)

    records: list[dict[str, Any]] = []
    blank_lines = 0
    decode_failures = 0
    non_mapping_records = 0

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()

            if not line:
                blank_lines += 1
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                decode_failures += 1
                raise ValueError(
                    f"JSON decode failure in {path} at line {line_number}: {exc}"
                ) from exc

            if not isinstance(record, dict):
                non_mapping_records += 1
                raise TypeError(
                    f"Expected JSON object in {path} at line {line_number}, got {type(record).__name__}"
                )

            records.append(record)

    if expected_rows is not None:
        require(
            len(records) == expected_rows,
            f"{path.name} row count mismatch: observed={len(records)} expected={expected_rows}",
        )

    require(blank_lines == 0, f"{path.name} contains blank JSONL lines: {blank_lines}")
    require(decode_failures == 0, f"{path.name} has decode failures: {decode_failures}")
    require(non_mapping_records == 0, f"{path.name} has non-mapping records: {non_mapping_records}")

    return records


# ------------------------------------------------------------
# Recursive extraction helpers
# ------------------------------------------------------------

BINANCE_TRADE_KEYS = {"e", "E", "s", "t", "p", "q", "T", "m"}
BINANCE_TRADE_REQUIRED_KEYS = {"t", "p", "q", "T", "m"}

COLLECTOR_SEQUENCE_KEYS = [
    "collector_sequence",
    "collector_seq",
    "sequence",
    "seq",
    "message_sequence",
    "record_sequence",
    "local_sequence",
]

LOCAL_RECEIPT_NS_KEYS = [
    "local_receipt_time_ns",
    "receipt_time_ns",
    "local_receive_time_ns",
    "local_received_time_ns",
    "local_timestamp_ns",
    "collector_time_ns",
    "received_at_ns",
    "recv_time_ns",
]

LOCAL_RECEIPT_TEXT_KEYS = [
    "local_receipt_time",
    "local_receipt_ts",
    "local_receipt_timestamp",
    "local_receive_time",
    "local_received_time",
    "collector_time",
    "received_at",
    "recv_time",
    "timestamp",
]

WRAPPER_PAYLOAD_KEYS = [
    "data",
    "message",
    "payload",
    "raw",
    "event",
    "ws_message",
    "websocket_message",
    "body",
]


def maybe_json_load(value: Any) -> Any:
    """Parse JSON strings when a wrapper stores payloads as text."""
    if isinstance(value, str):
        stripped = value.strip()
        if stripped.startswith("{") and stripped.endswith("}"):
            try:
                return json.loads(stripped)
            except Exception:
                return value
        if stripped.startswith("[") and stripped.endswith("]"):
            try:
                return json.loads(stripped)
            except Exception:
                return value

    return value


def iter_dict_nodes(obj: Any, path: str = "$") -> list[tuple[str, dict[str, Any]]]:
    """Return every dict node inside a nested JSON-like object."""
    obj = maybe_json_load(obj)

    nodes: list[tuple[str, dict[str, Any]]] = []

    if isinstance(obj, dict):
        nodes.append((path, obj))

        for key, value in obj.items():
            value = maybe_json_load(value)
            if isinstance(value, dict):
                nodes.extend(iter_dict_nodes(value, f"{path}.{key}"))
            elif isinstance(value, list):
                for idx, item in enumerate(value):
                    nodes.extend(iter_dict_nodes(item, f"{path}.{key}[{idx}]"))

    elif isinstance(obj, list):
        for idx, item in enumerate(obj):
            nodes.extend(iter_dict_nodes(item, f"{path}[{idx}]"))

    return nodes


def find_first_by_keys(record: dict[str, Any], keys: list[str]) -> tuple[Any, str | None]:
    """Find the first scalar value matching one of several key names anywhere in the JSON tree."""
    key_set = set(keys)

    for path, node in iter_dict_nodes(record):
        for key in keys:
            if key in node:
                value = maybe_json_load(node[key])
                if not isinstance(value, (dict, list)):
                    return value, f"{path}.{key}"

    # fallback: case-insensitive exact match
    lowered = {key.lower(): key for key in keys}

    for path, node in iter_dict_nodes(record):
        for key, value in node.items():
            if str(key).lower() in lowered:
                value = maybe_json_load(value)
                if not isinstance(value, (dict, list)):
                    return value, f"{path}.{key}"

    return None, None


def score_trade_payload_node(node: dict[str, Any]) -> int:
    """Score a dict node by how likely it is to be a Binance trade payload."""
    keys = set(node.keys())

    score = 0
    score += 10 * len(keys & BINANCE_TRADE_REQUIRED_KEYS)
    score += 2 * len(keys & BINANCE_TRADE_KEYS)

    event_type = node.get("e")
    if event_type in {"trade", "aggTrade"}:
        score += 15

    if "p" in node and "q" in node:
        score += 10

    if "t" in node and "T" in node:
        score += 10

    return score


def find_trade_payload(record: dict[str, Any]) -> tuple[dict[str, Any], str, int]:
    """Find the most likely Binance trade payload inside a collector-wrapped record."""
    candidates: list[tuple[int, str, dict[str, Any]]] = []

    for path, node in iter_dict_nodes(record):
        score = score_trade_payload_node(node)
        if score > 0:
            candidates.append((score, path, node))

    if not candidates:
        return {}, "NOT_FOUND", 0

    candidates.sort(key=lambda item: item[0], reverse=True)
    best_score, best_path, best_node = candidates[0]

    return best_node, best_path, best_score


def coerce_int_series(series: pd.Series) -> pd.Series:
    """Coerce values to nullable integer."""
    return pd.to_numeric(series, errors="coerce").astype("Int64")


def coerce_float_series(series: pd.Series) -> pd.Series:
    """Coerce values to float."""
    return pd.to_numeric(series, errors="coerce")


def coerce_bool_series(series: pd.Series) -> pd.Series:
    """Coerce mixed boolean-like values to pandas nullable boolean."""
    def convert_one(value: Any) -> Any:
        if pd.isna(value):
            return pd.NA

        if isinstance(value, bool):
            return value

        if isinstance(value, (int, np.integer)):
            if value == 1:
                return True
            if value == 0:
                return False

        if isinstance(value, str):
            cleaned = value.strip().lower()
            if cleaned in {"true", "t", "1", "yes", "y"}:
                return True
            if cleaned in {"false", "f", "0", "no", "n"}:
                return False

        return pd.NA

    return series.map(convert_one).astype("boolean")


def timestamp_ns_from_any(value: Any) -> Any:
    """Convert an ISO/timestamp-like scalar to UTC nanoseconds when possible."""
    if value is None or pd.isna(value):
        return pd.NA

    # Already numeric nanoseconds or milliseconds.
    if isinstance(value, (int, np.integer)):
        integer_value = int(value)

        # Heuristic scale detection.
        if integer_value > 10**17:
            return integer_value
        if integer_value > 10**14:
            return integer_value * 1_000
        if integer_value > 10**11:
            return integer_value * 1_000_000

        return pd.NA

    if isinstance(value, (float, np.floating)):
        if not math.isfinite(float(value)):
            return pd.NA
        return timestamp_ns_from_any(int(value))

    try:
        timestamp = pd.to_datetime(value, utc=True, errors="coerce")
    except Exception:
        return pd.NA

    if pd.isna(timestamp):
        return pd.NA

    return int(timestamp.value)


def exchange_time_ns_from_ms(value: Any) -> Any:
    """Convert exchange millisecond timestamp to UTC nanoseconds."""
    if value is None or pd.isna(value):
        return pd.NA

    try:
        value_int = int(value)
    except Exception:
        return pd.NA

    return value_int * 1_000_000


def canonicalize_trade_record(record: dict[str, Any], raw_row_number: int) -> dict[str, Any]:
    """Extract one canonical raw-trade row from a collector-wrapped JSON record."""
    payload, payload_path, payload_score = find_trade_payload(record)

    collector_sequence, collector_sequence_path = find_first_by_keys(record, COLLECTOR_SEQUENCE_KEYS)
    local_receipt_time_ns, local_receipt_time_ns_path = find_first_by_keys(record, LOCAL_RECEIPT_NS_KEYS)

    local_receipt_time_text = None
    local_receipt_time_text_path = None

    if local_receipt_time_ns is None:
        local_receipt_time_text, local_receipt_time_text_path = find_first_by_keys(record, LOCAL_RECEIPT_TEXT_KEYS)
        local_receipt_time_ns = timestamp_ns_from_any(local_receipt_time_text)

    # Fallback for wrappers that store collector sequence under a nested metadata node.
    if collector_sequence is None:
        collector_sequence = raw_row_number
        collector_sequence_path = "FALLBACK_RAW_ROW_NUMBER"

    exchange_event_time_ms = payload.get("E", pd.NA)
    exchange_trade_time_ms = payload.get("T", pd.NA)

    exchange_event_time_ns = exchange_time_ns_from_ms(exchange_event_time_ms)
    exchange_trade_time_ns = exchange_time_ns_from_ms(exchange_trade_time_ms)

    return {
        "source_run_prefix": SOURCE_RUN_PREFIX,
        "v0_1_run_id": V01_RUN_ID,
        "source_file": str(RAW_TRADE_JSONL_PATH),
        "raw_row_number": raw_row_number,

        "collector_sequence": collector_sequence,
        "collector_sequence_source_path": collector_sequence_path,

        "local_receipt_time_ns": local_receipt_time_ns,
        "local_receipt_time_ns_source_path": local_receipt_time_ns_path,
        "local_receipt_time_text": local_receipt_time_text,
        "local_receipt_time_text_source_path": local_receipt_time_text_path,

        "trade_payload_path": payload_path,
        "trade_payload_score": payload_score,

        "event_type": payload.get("e", pd.NA),
        "symbol": payload.get("s", pd.NA),

        "exchange_event_time_ms": exchange_event_time_ms,
        "exchange_event_time_ns": exchange_event_time_ns,
        "exchange_trade_time_ms": exchange_trade_time_ms,
        "exchange_trade_time_ns": exchange_trade_time_ns,

        "trade_id": payload.get("t", pd.NA),
        "price": payload.get("p", pd.NA),
        "quantity": payload.get("q", pd.NA),
        "buyer_order_id": payload.get("b", pd.NA),
        "seller_order_id": payload.get("a", pd.NA),
        "buyer_is_maker": payload.get("m", pd.NA),
        "ignore_flag": payload.get("M", pd.NA),
    }


# ------------------------------------------------------------
# Read and canonicalize raw trade stream
# ------------------------------------------------------------

raw_trade_json_records = read_jsonl_records(
    RAW_TRADE_JSONL_PATH,
    expected_rows=EXPECTED_RAW_TRADE_RECORDS,
)

raw_trade_rows = [
    canonicalize_trade_record(record=record, raw_row_number=idx)
    for idx, record in enumerate(raw_trade_json_records, start=1)
]

raw_trades = pd.DataFrame(raw_trade_rows)


# ------------------------------------------------------------
# Type normalization
# ------------------------------------------------------------

raw_trades["collector_sequence"] = coerce_int_series(raw_trades["collector_sequence"])
raw_trades["raw_row_number"] = coerce_int_series(raw_trades["raw_row_number"])

raw_trades["local_receipt_time_ns"] = coerce_int_series(raw_trades["local_receipt_time_ns"])
raw_trades["exchange_event_time_ms"] = coerce_int_series(raw_trades["exchange_event_time_ms"])
raw_trades["exchange_event_time_ns"] = coerce_int_series(raw_trades["exchange_event_time_ns"])
raw_trades["exchange_trade_time_ms"] = coerce_int_series(raw_trades["exchange_trade_time_ms"])
raw_trades["exchange_trade_time_ns"] = coerce_int_series(raw_trades["exchange_trade_time_ns"])

raw_trades["trade_id"] = coerce_int_series(raw_trades["trade_id"])
raw_trades["buyer_order_id"] = coerce_int_series(raw_trades["buyer_order_id"])
raw_trades["seller_order_id"] = coerce_int_series(raw_trades["seller_order_id"])

raw_trades["price"] = coerce_float_series(raw_trades["price"])
raw_trades["quantity"] = coerce_float_series(raw_trades["quantity"])

raw_trades["buyer_is_maker"] = coerce_bool_series(raw_trades["buyer_is_maker"])

raw_trades["local_receipt_time"] = pd.to_datetime(
    raw_trades["local_receipt_time_ns"],
    unit="ns",
    utc=True,
    errors="coerce",
)

raw_trades["exchange_event_time"] = pd.to_datetime(
    raw_trades["exchange_event_time_ms"],
    unit="ms",
    utc=True,
    errors="coerce",
)

raw_trades["exchange_trade_time"] = pd.to_datetime(
    raw_trades["exchange_trade_time_ms"],
    unit="ms",
    utc=True,
    errors="coerce",
)

raw_trades["notional"] = raw_trades["price"] * raw_trades["quantity"]

raw_trades["aggressor_side"] = pd.Series(pd.NA, index=raw_trades.index, dtype="string")
raw_trades.loc[raw_trades["buyer_is_maker"] == False, "aggressor_side"] = "BUY"
raw_trades.loc[raw_trades["buyer_is_maker"] == True, "aggressor_side"] = "SELL"

raw_trades["symbol"] = raw_trades["symbol"].astype("string")
raw_trades["event_type"] = raw_trades["event_type"].astype("string")
raw_trades["trade_payload_path"] = raw_trades["trade_payload_path"].astype("string")


# ------------------------------------------------------------
# Sort by collector sequence and raw row number
# ------------------------------------------------------------

raw_trades = raw_trades.sort_values(
    ["collector_sequence", "raw_row_number"],
    kind="mergesort",
).reset_index(drop=True)


# ------------------------------------------------------------
# Schema-discovery audit
# ------------------------------------------------------------

trade_payload_path_audit = (
    raw_trades
    .groupby("trade_payload_path", dropna=False)
    .size()
    .reset_index(name="row_count")
    .sort_values("row_count", ascending=False)
    .reset_index(drop=True)
)

collector_sequence_source_audit = (
    raw_trades
    .groupby("collector_sequence_source_path", dropna=False)
    .size()
    .reset_index(name="row_count")
    .sort_values("row_count", ascending=False)
    .reset_index(drop=True)
)

local_receipt_source_audit = (
    raw_trades
    .groupby(
        ["local_receipt_time_ns_source_path", "local_receipt_time_text_source_path"],
        dropna=False,
    )
    .size()
    .reset_index(name="row_count")
    .sort_values("row_count", ascending=False)
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Raw trade authority gates
# ------------------------------------------------------------

raw_trade_gates: list[GateResult] = []

raw_trade_gates.append(
    make_gate(
        gate="raw_trade_row_count",
        passed=len(raw_trades) == EXPECTED_RAW_TRADE_RECORDS,
        severity="BLOCKING",
        detail=f"observed={len(raw_trades)} expected={EXPECTED_RAW_TRADE_RECORDS}",
    )
)

raw_trade_gates.append(
    make_gate(
        gate="raw_trade_payload_found",
        passed=(raw_trades["trade_payload_path"] != "NOT_FOUND").all(),
        severity="BLOCKING",
        detail=f"payload_path_counts={raw_trades['trade_payload_path'].value_counts(dropna=False).head(10).to_dict()}",
    )
)

raw_trade_gates.append(
    make_gate(
        gate="raw_trade_collector_sequence_not_null",
        passed=raw_trades["collector_sequence"].notna().all(),
        severity="BLOCKING",
        detail=f"null_collector_sequence={int(raw_trades['collector_sequence'].isna().sum())}",
    )
)

raw_trade_gates.append(
    make_gate(
        gate="raw_trade_collector_sequence_unique",
        passed=not raw_trades["collector_sequence"].duplicated().any(),
        severity="BLOCKING",
        detail=f"duplicate_collector_sequence={int(raw_trades['collector_sequence'].duplicated().sum())}",
    )
)

raw_trade_gates.append(
    make_gate(
        gate="raw_trade_collector_sequence_monotone_increasing",
        passed=raw_trades["collector_sequence"].is_monotonic_increasing,
        severity="BLOCKING",
        detail=(
            f"first={raw_trades['collector_sequence'].iloc[0]} "
            f"last={raw_trades['collector_sequence'].iloc[-1]}"
        ),
    )
)

raw_trade_gates.append(
    make_gate(
        gate="raw_trade_local_receipt_time_valid",
        passed=raw_trades["local_receipt_time_ns"].notna().all()
        and raw_trades["local_receipt_time"].notna().all(),
        severity="BLOCKING",
        detail=(
            f"null_local_receipt_time_ns={int(raw_trades['local_receipt_time_ns'].isna().sum())} "
            f"null_local_receipt_time={int(raw_trades['local_receipt_time'].isna().sum())}"
        ),
    )
)

raw_trade_gates.append(
    make_gate(
        gate="raw_trade_local_receipt_time_non_decreasing",
        passed=raw_trades["local_receipt_time_ns"].dropna().is_monotonic_increasing,
        severity="BLOCKING",
        detail="local receipt time must not reverse in collector order",
    )
)

raw_trade_gates.append(
    make_gate(
        gate="raw_trade_exchange_event_time_valid",
        passed=raw_trades["exchange_event_time_ms"].notna().all()
        and raw_trades["exchange_event_time"].notna().all(),
        severity="BLOCKING",
        detail=(
            f"null_exchange_event_time_ms={int(raw_trades['exchange_event_time_ms'].isna().sum())} "
            f"null_exchange_event_time={int(raw_trades['exchange_event_time'].isna().sum())}"
        ),
    )
)

raw_trade_gates.append(
    make_gate(
        gate="raw_trade_exchange_trade_time_valid",
        passed=raw_trades["exchange_trade_time_ms"].notna().all()
        and raw_trades["exchange_trade_time"].notna().all(),
        severity="BLOCKING",
        detail=(
            f"null_exchange_trade_time_ms={int(raw_trades['exchange_trade_time_ms'].isna().sum())} "
            f"null_exchange_trade_time={int(raw_trades['exchange_trade_time'].isna().sum())}"
        ),
    )
)

raw_trade_gates.append(
    make_gate(
        gate="raw_trade_id_not_null",
        passed=raw_trades["trade_id"].notna().all(),
        severity="BLOCKING",
        detail=f"null_trade_id={int(raw_trades['trade_id'].isna().sum())}",
    )
)

raw_trade_gates.append(
    make_gate(
        gate="raw_trade_id_unique",
        passed=not raw_trades["trade_id"].duplicated().any(),
        severity="BLOCKING",
        detail=f"duplicate_trade_id={int(raw_trades['trade_id'].duplicated().sum())}",
    )
)

raw_trade_gates.append(
    make_gate(
        gate="raw_trade_price_positive",
        passed=raw_trades["price"].notna().all() and (raw_trades["price"] > 0).all(),
        severity="BLOCKING",
        detail=(
            f"null_price={int(raw_trades['price'].isna().sum())} "
            f"non_positive_price={int((raw_trades['price'].fillna(1.0) <= 0).sum())}"
        ),
    )
)

raw_trade_gates.append(
    make_gate(
        gate="raw_trade_quantity_positive",
        passed=raw_trades["quantity"].notna().all() and (raw_trades["quantity"] > 0).all(),
        severity="BLOCKING",
        detail=(
            f"null_quantity={int(raw_trades['quantity'].isna().sum())} "
            f"non_positive_quantity={int((raw_trades['quantity'].fillna(1.0) <= 0).sum())}"
        ),
    )
)

raw_trade_gates.append(
    make_gate(
        gate="raw_trade_buyer_is_maker_not_null",
        passed=raw_trades["buyer_is_maker"].notna().all(),
        severity="BLOCKING",
        detail=f"null_buyer_is_maker_count={int(raw_trades['buyer_is_maker'].isna().sum())}",
    )
)

raw_trade_gates.append(
    make_gate(
        gate="raw_trade_aggressor_side_valid",
        passed=raw_trades["aggressor_side"].notna().all()
        and set(raw_trades["aggressor_side"].dropna().unique()).issubset({"BUY", "SELL"}),
        severity="BLOCKING",
        detail=f"aggressor_side_counts={raw_trades['aggressor_side'].value_counts(dropna=False).to_dict()}",
    )
)

raw_trade_gates.append(
    make_gate(
        gate="raw_trade_symbol_expected",
        passed=raw_trades["symbol"].dropna().eq(SYMBOL).all(),
        severity="BLOCKING",
        detail=f"symbol_counts={raw_trades['symbol'].value_counts(dropna=False).to_dict()}",
    )
)

raw_trade_gate_frame = gate_results_to_frame(raw_trade_gates)

fail_if_blocking_gate_failed(raw_trade_gate_frame)


# ------------------------------------------------------------
# Compact audit summary
# ------------------------------------------------------------

raw_trade_summary = {
    "row_count": int(len(raw_trades)),
    "collector_sequence_min": int(raw_trades["collector_sequence"].min()),
    "collector_sequence_max": int(raw_trades["collector_sequence"].max()),
    "trade_id_min": int(raw_trades["trade_id"].min()),
    "trade_id_max": int(raw_trades["trade_id"].max()),
    "local_receipt_time_min": str(raw_trades["local_receipt_time"].min()),
    "local_receipt_time_max": str(raw_trades["local_receipt_time"].max()),
    "exchange_trade_time_min": str(raw_trades["exchange_trade_time"].min()),
    "exchange_trade_time_max": str(raw_trades["exchange_trade_time"].max()),
    "buy_count": int((raw_trades["aggressor_side"] == "BUY").sum()),
    "sell_count": int((raw_trades["aggressor_side"] == "SELL").sum()),
    "total_quantity": float(raw_trades["quantity"].sum()),
    "total_notional": float(raw_trades["notional"].sum()),
}

display(pd.DataFrame([raw_trade_summary]))

display(trade_payload_path_audit.head(10))
display(collector_sequence_source_audit.head(10))
display(local_receipt_source_audit.head(10))

raw_trade_gate_frame

,row_count,collector_sequence_min,collector_sequence_max,trade_id_min,trade_id_max,local_receipt_time_min,local_receipt_time_max,exchange_trade_time_min,exchange_trade_time_max,buy_count,sell_count,total_quantity,total_notional
0,67683,14,103676,6494596041,6494663723,2026-07-10 06:37:48.766951600+00:00,2026-07-10 07:37:46.707304900+00:00,2026-07-10 06:37:49.952000+00:00,2026-07-10 07:37:47.969000+00:00,30596,37087,329.19847,2.103595e+07


,trade_payload_path,row_count
0,$.raw_message.data,67683


,collector_sequence_source_path,row_count
0,$.collector_sequence,67683


,local_receipt_time_ns_source_path,local_receipt_time_text_source_path,row_count
0,$.local_receipt_time_ns,NaN,67683


,gate,status,severity,detail
0,raw_trade_row_count,PASS,BLOCKING,observed=67683 expected=67683
1,raw_trade_payload_found,PASS,BLOCKING,payload_path_counts={'$.raw_message.data': 67683}
2,raw_trade_collector_sequence_not_null,PASS,BLOCKING,null_collector_sequence=0
3,raw_trade_collector_sequence_unique,PASS,BLOCKING,duplicate_collector_sequence=0
4,raw_trade_collector_sequence_monotone_increasing,PASS,BLOCKING,first=14 last=103676
5,raw_trade_local_receipt_time_valid,PASS,BLOCKING,null_local_receipt_time_ns=0 null_local_receipt_time=0
6,raw_trade_local_receipt_time_non_decreasing,PASS,BLOCKING,local receipt time must not reverse in collector order
7,raw_trade_exchange_event_time_valid,PASS,BLOCKING,null_exchange_event_time_ms=0 null_exchange_event_time=0
8,raw_trade_exchange_trade_time_valid,PASS,BLOCKING,null_exchange_trade_time_ms=0 null_exchange_trade_time=0
9,raw_trade_id_not_null,PASS,BLOCKING,null_trade_id=0


In [7]:
# ============================================================
# Cell 06 — Load and canonicalize reconstructed visible book states
# ============================================================

# This cell loads Notebook 02 outputs from disk:
#   1. reconstructed book states
#   2. top-10 visible book wide table
#   3. update-continuity report
#
# It then builds one canonical one-row-per-book-state table for alignment.
#
# Alignment will use:
#   book_collector_sequence
#   book_local_receipt_time_ns
#
# The top-10 wide table is joined only as state detail. It is not the alignment authority.


# ------------------------------------------------------------
# CSV loading helper
# ------------------------------------------------------------

def read_csv_checked(path: Path, frame_name: str) -> pd.DataFrame:
    """Read a CSV with basic file and empty-frame checks."""
    path = require_file(path)

    frame = pd.read_csv(path, low_memory=False)

    require(
        not frame.empty,
        f"{frame_name} loaded from {path} is empty",
    )

    return frame


def normalize_column_name(column: str) -> str:
    """Normalize a column name for alias matching only."""
    return (
        str(column)
        .strip()
        .replace(" ", "_")
        .replace("-", "_")
        .replace(".", "_")
        .replace("/", "_")
        .lower()
    )


def build_column_lookup(frame: pd.DataFrame) -> dict[str, str]:
    """Map normalized column names to original column names."""
    lookup: dict[str, str] = {}

    for column in frame.columns:
        normalized = normalize_column_name(column)
        if normalized not in lookup:
            lookup[normalized] = column

    return lookup


def resolve_column(
    frame: pd.DataFrame,
    aliases: list[str],
    frame_name: str,
    required: bool = True,
) -> str | None:
    """Resolve a logical column from possible aliases."""
    lookup = build_column_lookup(frame)

    normalized_aliases = [normalize_column_name(alias) for alias in aliases]

    for alias in normalized_aliases:
        if alias in lookup:
            return lookup[alias]

    if required:
        raise KeyError(
            f"{frame_name} missing required logical column. "
            f"aliases={aliases} available_columns={list(frame.columns)}"
        )

    return None


def resolve_first_existing_pair(
    left: pd.DataFrame,
    right: pd.DataFrame,
    aliases: list[str],
    left_name: str,
    right_name: str,
) -> tuple[str | None, str | None]:
    """Resolve a shared join key from both frames."""
    left_column = resolve_column(left, aliases, left_name, required=False)
    right_column = resolve_column(right, aliases, right_name, required=False)

    if left_column is not None and right_column is not None:
        return left_column, right_column

    return None, None


# ------------------------------------------------------------
# Load Notebook 02 book outputs
# ------------------------------------------------------------

reconstructed_book_states_raw = read_csv_checked(
    RECONSTRUCTED_BOOK_STATES_PATH,
    frame_name="reconstructed_book_states_raw",
)

top10_visible_book_wide_raw = read_csv_checked(
    TOP10_VISIBLE_BOOK_WIDE_PATH,
    frame_name="top10_visible_book_wide_raw",
)

update_continuity_report = read_csv_checked(
    NOTEBOOK_02_UPDATE_CONTINUITY_REPORT_PATH,
    frame_name="update_continuity_report",
)


# ------------------------------------------------------------
# Resolve required book-state authority columns
# ------------------------------------------------------------

BOOK_COLLECTOR_SEQUENCE_ALIASES = [
    "book_collector_sequence",
    "collector_sequence",
    "depth_collector_sequence",
    "update_collector_sequence",
    "depth_update_collector_sequence",
    "final_collector_sequence",
    "last_collector_sequence",
    "last_depth_collector_sequence",
    "sequence",
]

BOOK_LOCAL_RECEIPT_NS_ALIASES = [
    "book_local_receipt_time_ns",
    "local_receipt_time_ns",
    "depth_local_receipt_time_ns",
    "update_local_receipt_time_ns",
    "depth_update_local_receipt_time_ns",
    "final_local_receipt_time_ns",
    "last_local_receipt_time_ns",
    "last_depth_local_receipt_time_ns",
    "receipt_time_ns",
]

BOOK_EXCHANGE_EVENT_TIME_MS_ALIASES = [
    "book_exchange_event_time_ms",
    "exchange_event_time_ms",
    "event_time_ms",
    "depth_exchange_event_time_ms",
    "update_exchange_event_time_ms",
    "depth_update_exchange_event_time_ms",
    "E",
]

BOOK_FIRST_UPDATE_ID_ALIASES = [
    "first_update_id",
    "first_u",
    "U",
    "firstUpdateId",
    "book_first_update_id",
]

BOOK_FINAL_UPDATE_ID_ALIASES = [
    "final_update_id",
    "last_update_id",
    "u",
    "lastUpdateId",
    "book_final_update_id",
]

BOOK_PREVIOUS_FINAL_UPDATE_ID_ALIASES = [
    "previous_final_update_id",
    "pu",
    "previous_update_id",
    "prev_final_update_id",
    "book_previous_final_update_id",
]

BOOK_BEST_BID_ALIASES = [
    "best_bid",
    "best_bid_price",
    "bid_price_1",
    "bid_px_1",
    "bid_1_price",
    "level_1_bid_price",
    "b1_price",
]

BOOK_BEST_ASK_ALIASES = [
    "best_ask",
    "best_ask_price",
    "ask_price_1",
    "ask_px_1",
    "ask_1_price",
    "level_1_ask_price",
    "a1_price",
]

BOOK_BEST_BID_SIZE_ALIASES = [
    "best_bid_size",
    "best_bid_qty",
    "best_bid_quantity",
    "bid_size_1",
    "bid_qty_1",
    "bid_quantity_1",
    "bid_1_size",
    "bid_1_qty",
    "level_1_bid_size",
    "b1_size",
]

BOOK_BEST_ASK_SIZE_ALIASES = [
    "best_ask_size",
    "best_ask_qty",
    "best_ask_quantity",
    "ask_size_1",
    "ask_qty_1",
    "ask_quantity_1",
    "ask_1_size",
    "ask_1_qty",
    "level_1_ask_size",
    "a1_size",
]

BOOK_SPREAD_ALIASES = [
    "spread",
    "bid_ask_spread",
    "best_spread",
    "quoted_spread",
]

BOOK_MIDPOINT_ALIASES = [
    "midpoint",
    "mid_price",
    "mid",
    "best_midpoint",
]

BOOK_MICROPRICE_ALIASES = [
    "microprice",
    "micro_price",
    "level_1_microprice",
]

BOOK_L1_IMBALANCE_ALIASES = [
    "level_1_imbalance",
    "l1_imbalance",
    "best_level_imbalance",
    "top_of_book_imbalance",
    "imbalance_l1",
]

BOOK_TOP10_IMBALANCE_ALIASES = [
    "top10_imbalance",
    "top_10_imbalance",
    "visible_top10_imbalance",
    "visible_top_10_imbalance",
]

BOOK_PARTITION_ALIASES = [
    "partition",
    "book_partition",
    "chronological_partition",
    "split_partition",
]

BOOK_STATE_ID_ALIASES = [
    "book_state_id",
    "state_id",
    "reconstructed_book_state_id",
    "visible_book_state_id",
]


book_collector_sequence_col = resolve_column(
    reconstructed_book_states_raw,
    BOOK_COLLECTOR_SEQUENCE_ALIASES,
    "reconstructed_book_states_raw",
)

book_local_receipt_time_ns_col = resolve_column(
    reconstructed_book_states_raw,
    BOOK_LOCAL_RECEIPT_NS_ALIASES,
    "reconstructed_book_states_raw",
)

book_exchange_event_time_ms_col = resolve_column(
    reconstructed_book_states_raw,
    BOOK_EXCHANGE_EVENT_TIME_MS_ALIASES,
    "reconstructed_book_states_raw",
    required=False,
)

book_first_update_id_col = resolve_column(
    reconstructed_book_states_raw,
    BOOK_FIRST_UPDATE_ID_ALIASES,
    "reconstructed_book_states_raw",
    required=False,
)

book_final_update_id_col = resolve_column(
    reconstructed_book_states_raw,
    BOOK_FINAL_UPDATE_ID_ALIASES,
    "reconstructed_book_states_raw",
    required=False,
)

book_previous_final_update_id_col = resolve_column(
    reconstructed_book_states_raw,
    BOOK_PREVIOUS_FINAL_UPDATE_ID_ALIASES,
    "reconstructed_book_states_raw",
    required=False,
)

book_best_bid_col = resolve_column(
    reconstructed_book_states_raw,
    BOOK_BEST_BID_ALIASES,
    "reconstructed_book_states_raw",
    required=False,
)

book_best_ask_col = resolve_column(
    reconstructed_book_states_raw,
    BOOK_BEST_ASK_ALIASES,
    "reconstructed_book_states_raw",
    required=False,
)

book_best_bid_size_col = resolve_column(
    reconstructed_book_states_raw,
    BOOK_BEST_BID_SIZE_ALIASES,
    "reconstructed_book_states_raw",
    required=False,
)

book_best_ask_size_col = resolve_column(
    reconstructed_book_states_raw,
    BOOK_BEST_ASK_SIZE_ALIASES,
    "reconstructed_book_states_raw",
    required=False,
)

book_spread_col = resolve_column(
    reconstructed_book_states_raw,
    BOOK_SPREAD_ALIASES,
    "reconstructed_book_states_raw",
    required=False,
)

book_midpoint_col = resolve_column(
    reconstructed_book_states_raw,
    BOOK_MIDPOINT_ALIASES,
    "reconstructed_book_states_raw",
    required=False,
)

book_microprice_col = resolve_column(
    reconstructed_book_states_raw,
    BOOK_MICROPRICE_ALIASES,
    "reconstructed_book_states_raw",
    required=False,
)

book_l1_imbalance_col = resolve_column(
    reconstructed_book_states_raw,
    BOOK_L1_IMBALANCE_ALIASES,
    "reconstructed_book_states_raw",
    required=False,
)

book_top10_imbalance_col = resolve_column(
    reconstructed_book_states_raw,
    BOOK_TOP10_IMBALANCE_ALIASES,
    "reconstructed_book_states_raw",
    required=False,
)

book_partition_col = resolve_column(
    reconstructed_book_states_raw,
    BOOK_PARTITION_ALIASES,
    "reconstructed_book_states_raw",
    required=False,
)

book_state_id_col = resolve_column(
    reconstructed_book_states_raw,
    BOOK_STATE_ID_ALIASES,
    "reconstructed_book_states_raw",
    required=False,
)


# ------------------------------------------------------------
# Build canonical reconstructed book-state frame
# ------------------------------------------------------------

book_states = pd.DataFrame(
    {
        "source_run_prefix": SOURCE_RUN_PREFIX,
        "v0_1_run_id": V01_RUN_ID,
        "source_file": str(RECONSTRUCTED_BOOK_STATES_PATH),
        "book_row_number": np.arange(1, len(reconstructed_book_states_raw) + 1),
        "book_collector_sequence": reconstructed_book_states_raw[book_collector_sequence_col],
        "book_local_receipt_time_ns": reconstructed_book_states_raw[book_local_receipt_time_ns_col],
    }
)

if book_state_id_col is not None:
    book_states["book_state_id"] = reconstructed_book_states_raw[book_state_id_col]
else:
    book_states["book_state_id"] = np.arange(1, len(reconstructed_book_states_raw) + 1)

if book_exchange_event_time_ms_col is not None:
    book_states["book_exchange_event_time_ms"] = reconstructed_book_states_raw[book_exchange_event_time_ms_col]
else:
    book_states["book_exchange_event_time_ms"] = pd.NA

if book_first_update_id_col is not None:
    book_states["book_first_update_id"] = reconstructed_book_states_raw[book_first_update_id_col]
else:
    book_states["book_first_update_id"] = pd.NA

if book_final_update_id_col is not None:
    book_states["book_final_update_id"] = reconstructed_book_states_raw[book_final_update_id_col]
else:
    book_states["book_final_update_id"] = pd.NA

if book_previous_final_update_id_col is not None:
    book_states["book_previous_final_update_id"] = reconstructed_book_states_raw[book_previous_final_update_id_col]
else:
    book_states["book_previous_final_update_id"] = pd.NA

if book_best_bid_col is not None:
    book_states["book_best_bid"] = reconstructed_book_states_raw[book_best_bid_col]
else:
    book_states["book_best_bid"] = pd.NA

if book_best_ask_col is not None:
    book_states["book_best_ask"] = reconstructed_book_states_raw[book_best_ask_col]
else:
    book_states["book_best_ask"] = pd.NA

if book_best_bid_size_col is not None:
    book_states["book_best_bid_size"] = reconstructed_book_states_raw[book_best_bid_size_col]
else:
    book_states["book_best_bid_size"] = pd.NA

if book_best_ask_size_col is not None:
    book_states["book_best_ask_size"] = reconstructed_book_states_raw[book_best_ask_size_col]
else:
    book_states["book_best_ask_size"] = pd.NA

if book_spread_col is not None:
    book_states["book_spread"] = reconstructed_book_states_raw[book_spread_col]
else:
    book_states["book_spread"] = pd.NA

if book_midpoint_col is not None:
    book_states["book_midpoint"] = reconstructed_book_states_raw[book_midpoint_col]
else:
    book_states["book_midpoint"] = pd.NA

if book_microprice_col is not None:
    book_states["book_microprice"] = reconstructed_book_states_raw[book_microprice_col]
else:
    book_states["book_microprice"] = pd.NA

if book_l1_imbalance_col is not None:
    book_states["book_l1_imbalance"] = reconstructed_book_states_raw[book_l1_imbalance_col]
else:
    book_states["book_l1_imbalance"] = pd.NA

if book_top10_imbalance_col is not None:
    book_states["book_top10_imbalance"] = reconstructed_book_states_raw[book_top10_imbalance_col]
else:
    book_states["book_top10_imbalance"] = pd.NA

if book_partition_col is not None:
    book_states["book_partition"] = reconstructed_book_states_raw[book_partition_col]
else:
    book_states["book_partition"] = pd.NA


# ------------------------------------------------------------
# Type normalization
# ------------------------------------------------------------

book_states["book_row_number"] = coerce_int_series(book_states["book_row_number"])
book_states["book_state_id"] = coerce_int_series(book_states["book_state_id"])
book_states["book_collector_sequence"] = coerce_int_series(book_states["book_collector_sequence"])
book_states["book_local_receipt_time_ns"] = coerce_int_series(book_states["book_local_receipt_time_ns"])
book_states["book_exchange_event_time_ms"] = coerce_int_series(book_states["book_exchange_event_time_ms"])

book_states["book_first_update_id"] = coerce_int_series(book_states["book_first_update_id"])
book_states["book_final_update_id"] = coerce_int_series(book_states["book_final_update_id"])
book_states["book_previous_final_update_id"] = coerce_int_series(book_states["book_previous_final_update_id"])

for column in [
    "book_best_bid",
    "book_best_ask",
    "book_best_bid_size",
    "book_best_ask_size",
    "book_spread",
    "book_midpoint",
    "book_microprice",
    "book_l1_imbalance",
    "book_top10_imbalance",
]:
    book_states[column] = coerce_float_series(book_states[column])

book_states["book_local_receipt_time"] = pd.to_datetime(
    book_states["book_local_receipt_time_ns"],
    unit="ns",
    utc=True,
    errors="coerce",
)

book_states["book_exchange_event_time"] = pd.to_datetime(
    book_states["book_exchange_event_time_ms"],
    unit="ms",
    utc=True,
    errors="coerce",
)

book_states["book_exchange_event_time_ns"] = coerce_int_series(
    book_states["book_exchange_event_time_ms"] * 1_000_000
)


# ------------------------------------------------------------
# Derive missing top-of-book diagnostics when possible
# ------------------------------------------------------------

if book_states["book_spread"].isna().all():
    book_states["book_spread"] = book_states["book_best_ask"] - book_states["book_best_bid"]

if book_states["book_midpoint"].isna().all():
    book_states["book_midpoint"] = (book_states["book_best_bid"] + book_states["book_best_ask"]) / 2.0

if book_states["book_l1_imbalance"].isna().all():
    denominator = book_states["book_best_bid_size"] + book_states["book_best_ask_size"]
    book_states["book_l1_imbalance"] = np.where(
        denominator > 0,
        (book_states["book_best_bid_size"] - book_states["book_best_ask_size"]) / denominator,
        np.nan,
    )

if book_states["book_microprice"].isna().all():
    denominator = book_states["book_best_bid_size"] + book_states["book_best_ask_size"]
    book_states["book_microprice"] = np.where(
        denominator > 0,
        (
            book_states["book_best_ask"] * book_states["book_best_bid_size"]
            + book_states["book_best_bid"] * book_states["book_best_ask_size"]
        )
        / denominator,
        np.nan,
    )


# ------------------------------------------------------------
# Join selected top-10 wide details when there is a safe shared key
# ------------------------------------------------------------

top10_join_left_col = None
top10_join_right_col = None

for alias_group in [
    BOOK_STATE_ID_ALIASES,
    BOOK_COLLECTOR_SEQUENCE_ALIASES,
    BOOK_FINAL_UPDATE_ID_ALIASES,
]:
    left_col, right_col = resolve_first_existing_pair(
        book_states,
        top10_visible_book_wide_raw,
        alias_group,
        "book_states",
        "top10_visible_book_wide_raw",
    )

    if left_col is not None and right_col is not None:
        top10_join_left_col = left_col
        top10_join_right_col = right_col
        break

top10_wide_enrichment_status = "NOT_JOINED_NO_SAFE_SHARED_KEY"

if top10_join_left_col is not None and top10_join_right_col is not None:
    top10_wide_enrichment_status = f"JOINED_ON::{top10_join_left_col}::{top10_join_right_col}"

    top10_extra_columns = [
        column
        for column in top10_visible_book_wide_raw.columns
        if column != top10_join_right_col
    ]

    # Keep the join conservative: only add columns not already present after prefixing.
    top10_wide_for_join = top10_visible_book_wide_raw[
        [top10_join_right_col] + top10_extra_columns
    ].copy()

    rename_map = {}
    for column in top10_extra_columns:
        normalized = normalize_column_name(column)
        proposed = f"top10_{normalized}"

        if proposed in book_states.columns:
            proposed = f"top10_wide_{normalized}"

        rename_map[column] = proposed

    top10_wide_for_join = top10_wide_for_join.rename(columns=rename_map)

    require(
        not top10_wide_for_join[top10_join_right_col].duplicated().any(),
        f"top10 wide join key has duplicates: {top10_join_right_col}",
    )

    book_states = book_states.merge(
        top10_wide_for_join,
        how="left",
        left_on=top10_join_left_col,
        right_on=top10_join_right_col,
        validate="one_to_one",
    )

    if top10_join_right_col != top10_join_left_col and top10_join_right_col in book_states.columns:
        book_states = book_states.drop(columns=[top10_join_right_col])


# ------------------------------------------------------------
# Sort and enforce one row per reconstructed book state
# ------------------------------------------------------------

book_states = book_states.sort_values(
    ["book_collector_sequence", "book_row_number"],
    kind="mergesort",
).reset_index(drop=True)


# ------------------------------------------------------------
# Book-state authority gates
# ------------------------------------------------------------

book_state_gates: list[GateResult] = []

book_state_gates.append(
    make_gate(
        gate="reconstructed_book_state_row_count",
        passed=len(book_states) == EXPECTED_RECONSTRUCTED_BOOK_STATES,
        severity="BLOCKING",
        detail=f"observed={len(book_states)} expected={EXPECTED_RECONSTRUCTED_BOOK_STATES}",
    )
)

book_state_gates.append(
    make_gate(
        gate="top10_visible_book_wide_row_count",
        passed=len(top10_visible_book_wide_raw) == EXPECTED_TOP10_WIDE_ROWS,
        severity="BLOCKING",
        detail=f"observed={len(top10_visible_book_wide_raw)} expected={EXPECTED_TOP10_WIDE_ROWS}",
    )
)

book_state_gates.append(
    make_gate(
        gate="top10_visible_book_long_row_count_expected_from_prior_contract",
        passed=True,
        severity="INFO",
        detail=f"expected_top10_long_rows={EXPECTED_TOP10_LONG_ROWS}; long table path already verified in Cell 03",
    )
)

book_state_gates.append(
    make_gate(
        gate="book_collector_sequence_not_null",
        passed=book_states["book_collector_sequence"].notna().all(),
        severity="BLOCKING",
        detail=f"null_book_collector_sequence={int(book_states['book_collector_sequence'].isna().sum())}",
    )
)

book_state_gates.append(
    make_gate(
        gate="book_collector_sequence_unique",
        passed=not book_states["book_collector_sequence"].duplicated().any(),
        severity="BLOCKING",
        detail=f"duplicate_book_collector_sequence={int(book_states['book_collector_sequence'].duplicated().sum())}",
    )
)

book_state_gates.append(
    make_gate(
        gate="book_collector_sequence_monotone_increasing",
        passed=book_states["book_collector_sequence"].is_monotonic_increasing,
        severity="BLOCKING",
        detail=(
            f"first={book_states['book_collector_sequence'].iloc[0]} "
            f"last={book_states['book_collector_sequence'].iloc[-1]}"
        ),
    )
)

book_state_gates.append(
    make_gate(
        gate="book_local_receipt_time_valid",
        passed=book_states["book_local_receipt_time_ns"].notna().all()
        and book_states["book_local_receipt_time"].notna().all(),
        severity="BLOCKING",
        detail=(
            f"null_book_local_receipt_time_ns={int(book_states['book_local_receipt_time_ns'].isna().sum())} "
            f"null_book_local_receipt_time={int(book_states['book_local_receipt_time'].isna().sum())}"
        ),
    )
)

book_state_gates.append(
    make_gate(
        gate="book_local_receipt_time_non_decreasing",
        passed=book_states["book_local_receipt_time_ns"].dropna().is_monotonic_increasing,
        severity="BLOCKING",
        detail="book local receipt time must not reverse in book collector order",
    )
)

book_state_gates.append(
    make_gate(
        gate="book_best_bid_valid",
        passed=book_states["book_best_bid"].notna().all() and (book_states["book_best_bid"] > 0).all(),
        severity="BLOCKING",
        detail=(
            f"null_book_best_bid={int(book_states['book_best_bid'].isna().sum())} "
            f"non_positive_book_best_bid={int((book_states['book_best_bid'].fillna(1.0) <= 0).sum())}"
        ),
    )
)

book_state_gates.append(
    make_gate(
        gate="book_best_ask_valid",
        passed=book_states["book_best_ask"].notna().all() and (book_states["book_best_ask"] > 0).all(),
        severity="BLOCKING",
        detail=(
            f"null_book_best_ask={int(book_states['book_best_ask'].isna().sum())} "
            f"non_positive_book_best_ask={int((book_states['book_best_ask'].fillna(1.0) <= 0).sum())}"
        ),
    )
)

book_state_gates.append(
    make_gate(
        gate="book_uncrossed_and_unlocked",
        passed=((book_states["book_best_ask"] > book_states["book_best_bid"]).all()),
        severity="BLOCKING",
        detail=(
            "requires book_best_ask > book_best_bid for every reconstructed visible state; "
            f"bad_rows={int((book_states['book_best_ask'] <= book_states['book_best_bid']).sum())}"
        ),
    )
)

book_state_gates.append(
    make_gate(
        gate="book_spread_positive",
        passed=book_states["book_spread"].notna().all() and (book_states["book_spread"] > 0).all(),
        severity="BLOCKING",
        detail=(
            f"null_book_spread={int(book_states['book_spread'].isna().sum())} "
            f"non_positive_book_spread={int((book_states['book_spread'].fillna(1.0) <= 0).sum())}"
        ),
    )
)

book_state_gates.append(
    make_gate(
        gate="book_midpoint_valid",
        passed=book_states["book_midpoint"].notna().all()
        and (book_states["book_midpoint"] > 0).all()
        and (book_states["book_midpoint"] > book_states["book_best_bid"]).all()
        and (book_states["book_midpoint"] < book_states["book_best_ask"]).all(),
        severity="BLOCKING",
        detail=(
            f"null_book_midpoint={int(book_states['book_midpoint'].isna().sum())}; "
            "midpoint must be inside visible best bid/ask"
        ),
    )
)

book_state_gates.append(
    make_gate(
        gate="book_top10_wide_enrichment",
        passed=top10_wide_enrichment_status.startswith("JOINED_ON::"),
        severity="WARNING",
        detail=top10_wide_enrichment_status,
    )
)

book_state_gate_frame = gate_results_to_frame(book_state_gates)

fail_if_blocking_gate_failed(book_state_gate_frame)


# ------------------------------------------------------------
# Compact audit summary
# ------------------------------------------------------------

book_column_resolution_audit = pd.DataFrame(
    [
        {"logical_column": "book_collector_sequence", "source_column": book_collector_sequence_col},
        {"logical_column": "book_local_receipt_time_ns", "source_column": book_local_receipt_time_ns_col},
        {"logical_column": "book_exchange_event_time_ms", "source_column": book_exchange_event_time_ms_col},
        {"logical_column": "book_first_update_id", "source_column": book_first_update_id_col},
        {"logical_column": "book_final_update_id", "source_column": book_final_update_id_col},
        {"logical_column": "book_previous_final_update_id", "source_column": book_previous_final_update_id_col},
        {"logical_column": "book_best_bid", "source_column": book_best_bid_col},
        {"logical_column": "book_best_ask", "source_column": book_best_ask_col},
        {"logical_column": "book_best_bid_size", "source_column": book_best_bid_size_col},
        {"logical_column": "book_best_ask_size", "source_column": book_best_ask_size_col},
        {"logical_column": "book_spread", "source_column": book_spread_col},
        {"logical_column": "book_midpoint", "source_column": book_midpoint_col},
        {"logical_column": "book_microprice", "source_column": book_microprice_col},
        {"logical_column": "book_l1_imbalance", "source_column": book_l1_imbalance_col},
        {"logical_column": "book_top10_imbalance", "source_column": book_top10_imbalance_col},
        {"logical_column": "book_partition", "source_column": book_partition_col},
        {"logical_column": "book_state_id", "source_column": book_state_id_col},
    ]
)

book_state_summary = {
    "row_count": int(len(book_states)),
    "book_collector_sequence_min": int(book_states["book_collector_sequence"].min()),
    "book_collector_sequence_max": int(book_states["book_collector_sequence"].max()),
    "book_local_receipt_time_min": str(book_states["book_local_receipt_time"].min()),
    "book_local_receipt_time_max": str(book_states["book_local_receipt_time"].max()),
    "book_exchange_event_time_min": str(book_states["book_exchange_event_time"].min()),
    "book_exchange_event_time_max": str(book_states["book_exchange_event_time"].max()),
    "book_best_bid_min": float(book_states["book_best_bid"].min()),
    "book_best_bid_max": float(book_states["book_best_bid"].max()),
    "book_best_ask_min": float(book_states["book_best_ask"].min()),
    "book_best_ask_max": float(book_states["book_best_ask"].max()),
    "book_spread_min": float(book_states["book_spread"].min()),
    "book_spread_max": float(book_states["book_spread"].max()),
    "top10_wide_enrichment_status": top10_wide_enrichment_status,
}

display(pd.DataFrame([book_state_summary]))
display(book_column_resolution_audit)
display(book_state_gate_frame)

book_states.head()

,row_count,book_collector_sequence_min,book_collector_sequence_max,book_local_receipt_time_min,book_local_receipt_time_max,book_exchange_event_time_min,book_exchange_event_time_max,book_best_bid_min,book_best_bid_max,book_best_ask_min,book_best_ask_max,book_spread_min,book_spread_max,top10_wide_enrichment_status
0,35985,10,103677,2026-07-10 06:37:48.432309400+00:00,2026-07-10 07:37:46.749750800+00:00,2026-07-10 06:37:49.614000+00:00,2026-07-10 07:37:48.014000+00:00,63805.31,64011.89,63805.32,64011.9,0.01,3.39,JOINED_ON::book_state_id::book_state_id


,logical_column,source_column
0,book_collector_sequence,collector_sequence
1,book_local_receipt_time_ns,local_receipt_time_ns
2,book_exchange_event_time_ms,exchange_event_time_ms
3,book_first_update_id,first_update_id
4,book_final_update_id,final_update_id
5,book_previous_final_update_id,previous_final_update_id
6,book_best_bid,best_bid
7,book_best_ask,best_ask
8,book_best_bid_size,best_bid_quantity
9,book_best_ask_size,best_ask_quantity


,gate,status,severity,detail
0,reconstructed_book_state_row_count,PASS,BLOCKING,observed=35985 expected=35985
1,top10_visible_book_wide_row_count,PASS,BLOCKING,observed=35985 expected=35985
2,top10_visible_book_long_row_count_expected_from_prior_contract,PASS,INFO,expected_top10_long_rows=719700; long table path already verified in Cell 03
3,book_collector_sequence_not_null,PASS,BLOCKING,null_book_collector_sequence=0
4,book_collector_sequence_unique,PASS,BLOCKING,duplicate_book_collector_sequence=0
5,book_collector_sequence_monotone_increasing,PASS,BLOCKING,first=10 last=103677
6,book_local_receipt_time_valid,PASS,BLOCKING,null_book_local_receipt_time_ns=0 null_book_local_receipt_time=0
7,book_local_receipt_time_non_decreasing,PASS,BLOCKING,book local receipt time must not reverse in book collector order
8,book_best_bid_valid,PASS,BLOCKING,null_book_best_bid=0 non_positive_book_best_bid=0
9,book_best_ask_valid,PASS,BLOCKING,null_book_best_ask=0 non_positive_book_best_ask=0


,source_run_prefix,v0_1_run_id,source_file,book_row_number,book_collector_sequence,book_local_receipt_time_ns,book_state_id,book_exchange_event_time_ms,book_first_update_id,book_final_update_id,book_previous_final_update_id,book_best_bid,book_best_ask,book_best_bid_size,book_best_ask_size,book_spread,book_midpoint,book_microprice,book_l1_imbalance,book_top10_imbalance,book_partition,book_local_receipt_time,book_exchange_event_time,book_exchange_event_time_ns,top10_reconstruction_session_id,top10_collector_sequence,top10_local_receipt_time_ns,top10_local_receipt_time_utc,top10_exchange_event_time_ms,top10_exchange_event_time_utc,top10_first_update_id,top10_final_update_id,top10_top_10_state_sha256,top10_bid_price_1,top10_bid_quantity_1,top10_ask_price_1,top10_ask_quantity_1,top10_bid_price_2,top10_bid_quantity_2,top10_ask_price_2,top10_ask_quantity_2,top10_bid_price_3,top10_bid_quantity_3,top10_ask_price_3,top10_ask_quantity_3,top10_bid_price_4,top10_bid_quantity_4,top10_ask_price_4,top10_ask_quantity_4,top10_bid_price_5,top10_bid_quantity_5,top10_ask_price_5,top10_ask_quantity_5,top10_bid_price_6,top10_bid_quantity_6,top10_ask_price_6,top10_ask_quantity_6,top10_bid_price_7,top10_bid_quantity_7,top10_ask_price_7,top10_ask_quantity_7,top10_bid_price_8,top10_bid_quantity_8,top10_ask_price_8,top10_ask_quantity_8,top10_bid_price_9,top10_bid_quantity_9,top10_ask_price_9,top10_ask_quantity_9,top10_bid_price_10,top10_bid_quantity_10,top10_ask_price_10,top10_ask_quantity_10,top10_partition,top10_spread_ticks,top10_spread_basis_points,top10_interval_multiple_of_nominal,top10_stale_interval
0,BTCUSDT_spot_20260710T063746Z_c8b5bf12,v0_1_20260714T090616Z_e82325081a81,D:\Clown Project\V0.1\data\processed\book\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__02_VISIBLE_BOOK_RECONSTRUCTION__recons...,1,10,1783665468432309400,0,1783665469614,97233590167,97233590170,97233590166,63914.36,63914.37,2.72404,1.82146,0.01,63914.365,63914.365993,0.198566,NaN,DEVELOPMENT,2026-07-10 06:37:48.432309400+00:00,2026-07-10 06:37:49.614000+00:00,1783665469614000000,v0_1_20260714T090616Z_e82325081a81__book_session_0001,10,1783665468432309400,2026-07-10T06:37:48.432309400+00:00,1783665469614,2026-07-10T06:37:49.614000+00:00,97233590167,97233590170,a07d9dd6acb60a1c93b28bf418f2613b8f4300c6b5aef6ba979ab75fd71c1c42,63914.36,2.72404,63914.37,1.82146,63914.35,0.00369,63914.38,0.00043,63914.34,0.00016,63914.39,0.00016,63914.3,0.00023,63914.6,0.00008,63914.02,0.00018,63914.79,0.00041,63914.01,0.07396,63914.84,0.00008,63914.0,0.01918,63915.63,0.00018,63913.71,0.00008,63915.64,0.3584,63913.6,0.00571,63915.65,0.44496,63913.59,0.00018,63916.09,0.00018,DEVELOPMENT,1,0.001565,NaN,False
1,BTCUSDT_spot_20260710T063746Z_c8b5bf12,v0_1_20260714T090616Z_e82325081a81,D:\Clown Project\V0.1\data\processed\book\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__02_VISIBLE_BOOK_RECONSTRUCTION__recons...,2,11,1783665468532310900,1,1783665469714,97233590171,97233590187,97233590170,63914.36,63914.37,2.72404,1.82138,0.01,63914.365,63914.365993,0.198587,NaN,DEVELOPMENT,2026-07-10 06:37:48.532310900+00:00,2026-07-10 06:37:49.714000+00:00,1783665469714000000,v0_1_20260714T090616Z_e82325081a81__book_session_0001,11,1783665468532310900,2026-07-10T06:37:48.532310900+00:00,1783665469714,2026-07-10T06:37:49.714000+00:00,97233590171,97233590187,b1186dc07b71dc5f25ebad271ae5ebbfc2417debce43eb1be7c993c979bf0436,63914.36,2.72404,63914.37,1.82138,63914.35,0.00369,63914.38,0.00052,63914.34,0.00016,63914.39,0.00016,63914.3,0.00023,63914.6,0.00008,63914.02,0.00018,63914.79,0.00041,63914.01,0.07396,63914.84,0.00008,63914.0,0.01918,63915.63,0.00018,63913.71,0.00008,63915.64,0.3584,63913.6,0.00571,63915.65,0.44496,63913.59,0.00018,63916.09,0.00018,DEVELOPMENT,1,0.001565,1.000015,False
2,BTCUSDT_spot_20260710T063746Z_c8b5bf12,v0_1_20260714T090616Z_e82325081a81,D:\Clown Project\V0.1\data\processed\book\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T0906

In [8]:
# ============================================================
# Cell 07 — LOCAL_STRICT causal trade-to-book alignment
# ============================================================

# Primary policy:
#
# For each trade, choose the latest reconstructed book state satisfying:
#
#   book_collector_sequence < trade_collector_sequence
#   book_local_receipt_time_ns <= trade_local_receipt_time_ns
#
# The selected book state is the latest visible book state known to the collector
# before the trade arrived.
#
# This cell produces:
#   local_strict_all_alignment
#   local_strict_matched_alignment
#   unmatched_trades


# ------------------------------------------------------------
# Alignment constants
# ------------------------------------------------------------

NOMINAL_BOOK_INTERVAL_MS = 100.0
STALE_MATCH_THRESHOLD_MS = 250.0
VERY_STALE_MATCH_THRESHOLD_MS = 1_000.0


# ------------------------------------------------------------
# Prepare trade frame with explicit trade prefixes
# ------------------------------------------------------------

trade_alignment = raw_trades.copy()

trade_alignment = trade_alignment.rename(
    columns={
        "source_file": "trade_source_file",
        "raw_row_number": "trade_raw_row_number",
        "collector_sequence": "trade_collector_sequence",
        "collector_sequence_source_path": "trade_collector_sequence_source_path",
        "local_receipt_time_ns": "trade_local_receipt_time_ns",
        "local_receipt_time": "trade_local_receipt_time",
        "local_receipt_time_ns_source_path": "trade_local_receipt_time_ns_source_path",
        "local_receipt_time_text": "trade_local_receipt_time_text",
        "local_receipt_time_text_source_path": "trade_local_receipt_time_text_source_path",
        "exchange_event_time_ms": "trade_exchange_event_time_ms",
        "exchange_event_time_ns": "trade_exchange_event_time_ns",
        "exchange_event_time": "trade_exchange_event_time",
        "exchange_trade_time_ms": "trade_exchange_trade_time_ms",
        "exchange_trade_time_ns": "trade_exchange_trade_time_ns",
        "exchange_trade_time": "trade_exchange_trade_time",
        "event_type": "trade_event_type",
        "symbol": "trade_symbol",
        "price": "trade_price",
        "quantity": "trade_quantity",
        "buyer_order_id": "trade_buyer_order_id",
        "seller_order_id": "trade_seller_order_id",
        "buyer_is_maker": "trade_buyer_is_maker",
        "ignore_flag": "trade_ignore_flag",
        "aggressor_side": "trade_aggressor_side",
        "notional": "trade_notional",
    }
)

require_columns(
    trade_alignment,
    [
        "trade_collector_sequence",
        "trade_local_receipt_time_ns",
        "trade_exchange_trade_time_ms",
        "trade_id",
        "trade_price",
        "trade_quantity",
        "trade_buyer_is_maker",
        "trade_aggressor_side",
    ],
    "trade_alignment",
)

require_columns(
    book_states,
    [
        "book_collector_sequence",
        "book_local_receipt_time_ns",
        "book_exchange_event_time_ms",
        "book_best_bid",
        "book_best_ask",
        "book_spread",
        "book_midpoint",
    ],
    "book_states",
)


# ------------------------------------------------------------
# Convert ordering columns to dense numpy arrays
# ------------------------------------------------------------

trade_sequence_array = trade_alignment["trade_collector_sequence"].astype("int64").to_numpy()
trade_local_time_array = trade_alignment["trade_local_receipt_time_ns"].astype("int64").to_numpy()

book_sequence_array = book_states["book_collector_sequence"].astype("int64").to_numpy()
book_local_time_array = book_states["book_local_receipt_time_ns"].astype("int64").to_numpy()

require(
    np.all(np.diff(book_sequence_array) > 0),
    "book_sequence_array must be strictly increasing",
)

require(
    np.all(np.diff(book_local_time_array) >= 0),
    "book_local_time_array must be non-decreasing",
)

require(
    np.all(np.diff(trade_sequence_array) > 0),
    "trade_sequence_array must be strictly increasing",
)

require(
    np.all(np.diff(trade_local_time_array) >= 0),
    "trade_local_time_array must be non-decreasing",
)


# ------------------------------------------------------------
# LOCAL_STRICT vectorized candidate selection
# ------------------------------------------------------------

# First book index whose sequence is >= trade sequence.
# Eligible sequence indices are strictly before this position.
sequence_upper_positions = np.searchsorted(
    book_sequence_array,
    trade_sequence_array,
    side="left",
)

# First book index whose local receipt time is > trade local receipt time.
# Eligible local-time indices are strictly before this position.
time_upper_positions = np.searchsorted(
    book_local_time_array,
    trade_local_time_array,
    side="right",
)

# Since both book sequence and book local time are monotone in book-state order,
# the latest book satisfying both constraints is the minimum of the two upper
# bounds minus one.
candidate_book_positions = np.minimum(
    sequence_upper_positions,
    time_upper_positions,
) - 1

candidate_has_book = candidate_book_positions >= 0


# ------------------------------------------------------------
# Attach matched book rows
# ------------------------------------------------------------

book_for_alignment = book_states.reset_index(drop=True).copy()

book_for_alignment = book_for_alignment.rename(
    columns={
        "source_file": "book_source_file",
    }
)

book_for_alignment = book_for_alignment.drop(
    columns=["source_run_prefix", "v0_1_run_id"],
    errors="ignore",
)

matched_book_rows = book_for_alignment.reindex(candidate_book_positions).reset_index(drop=True)

local_strict_all_alignment = pd.concat(
    [
        trade_alignment.reset_index(drop=True),
        matched_book_rows,
    ],
    axis=1,
)

local_strict_all_alignment["alignment_policy"] = PRIMARY_ALIGNMENT_POLICY
local_strict_all_alignment["alignment_candidate_position"] = candidate_book_positions
local_strict_all_alignment["sequence_upper_position"] = sequence_upper_positions
local_strict_all_alignment["time_upper_position"] = time_upper_positions


# ------------------------------------------------------------
# Validate match conditions row by row
# ------------------------------------------------------------

sequence_condition = (
    local_strict_all_alignment["book_collector_sequence"].notna()
    & (
        local_strict_all_alignment["book_collector_sequence"].astype("Int64")
        < local_strict_all_alignment["trade_collector_sequence"].astype("Int64")
    )
)

local_time_condition = (
    local_strict_all_alignment["book_local_receipt_time_ns"].notna()
    & (
        local_strict_all_alignment["book_local_receipt_time_ns"].astype("Int64")
        <= local_strict_all_alignment["trade_local_receipt_time_ns"].astype("Int64")
    )
)

local_strict_all_alignment["candidate_has_book"] = candidate_has_book
local_strict_all_alignment["local_strict_sequence_condition"] = sequence_condition.to_numpy()
local_strict_all_alignment["local_strict_time_condition"] = local_time_condition.to_numpy()

local_strict_all_alignment["is_local_strict_match"] = (
    local_strict_all_alignment["candidate_has_book"]
    & local_strict_all_alignment["local_strict_sequence_condition"]
    & local_strict_all_alignment["local_strict_time_condition"]
)


# ------------------------------------------------------------
# Match-status and unmatched reason classification
# ------------------------------------------------------------

local_strict_all_alignment["alignment_status"] = "MATCHED_LOCAL_STRICT"
local_strict_all_alignment.loc[
    ~local_strict_all_alignment["is_local_strict_match"],
    "alignment_status",
] = "UNMATCHED"

nearest_prior_book_by_sequence_exists = sequence_upper_positions > 0
nearest_prior_book_by_time_exists = time_upper_positions > 0

unmatched_reason = np.full(
    len(local_strict_all_alignment),
    "MATCHED_LOCAL_STRICT",
    dtype=object,
)

unmatched_mask = ~local_strict_all_alignment["is_local_strict_match"].to_numpy()

unmatched_reason[
    unmatched_mask & (~nearest_prior_book_by_sequence_exists)
] = "NO_PRIOR_BOOK_STATE"

unmatched_reason[
    unmatched_mask
    & nearest_prior_book_by_sequence_exists
    & (~nearest_prior_book_by_time_exists)
] = "LOCAL_TIME_CONFLICT"

unmatched_reason[
    unmatched_mask
    & candidate_has_book
    & (
        ~local_strict_all_alignment["local_strict_sequence_condition"].to_numpy()
        | ~local_strict_all_alignment["local_strict_time_condition"].to_numpy()
    )
] = "INVALID_CANDIDATE_LOCAL_STRICT_VIOLATION"

unmatched_reason[
    unmatched_mask
    & nearest_prior_book_by_sequence_exists
    & nearest_prior_book_by_time_exists
    & (~candidate_has_book)
] = "UNCLASSIFIED"

local_strict_all_alignment["unmatched_reason"] = unmatched_reason


# ------------------------------------------------------------
# Lag metrics
# ------------------------------------------------------------

local_strict_all_alignment["sequence_lag"] = (
    local_strict_all_alignment["trade_collector_sequence"].astype("Int64")
    - local_strict_all_alignment["book_collector_sequence"].astype("Int64")
)

local_strict_all_alignment["local_observation_lag_ns"] = (
    local_strict_all_alignment["trade_local_receipt_time_ns"].astype("Int64")
    - local_strict_all_alignment["book_local_receipt_time_ns"].astype("Int64")
)

local_strict_all_alignment["local_observation_lag_ms"] = (
    pd.to_numeric(
        local_strict_all_alignment["local_observation_lag_ns"],
        errors="coerce",
    )
    / 1_000_000.0
)

local_strict_all_alignment["exchange_time_lag_ms"] = (
    pd.to_numeric(
        local_strict_all_alignment["trade_exchange_trade_time_ms"],
        errors="coerce",
    )
    - pd.to_numeric(
        local_strict_all_alignment["book_exchange_event_time_ms"],
        errors="coerce",
    )
)

local_strict_all_alignment["same_local_timestamp_flag"] = (
    local_strict_all_alignment["local_observation_lag_ns"].fillna(-1).astype("int64") == 0
)

local_strict_all_alignment["zero_local_lag_flag"] = (
    local_strict_all_alignment["local_observation_lag_ns"].fillna(-1).astype("int64") == 0
)

local_strict_all_alignment["above_nominal_book_interval_flag"] = (
    local_strict_all_alignment["local_observation_lag_ms"] > NOMINAL_BOOK_INTERVAL_MS
)

local_strict_all_alignment["stale_match_flag"] = (
    local_strict_all_alignment["local_observation_lag_ms"] > STALE_MATCH_THRESHOLD_MS
)

local_strict_all_alignment["very_stale_match_flag"] = (
    local_strict_all_alignment["local_observation_lag_ms"] > VERY_STALE_MATCH_THRESHOLD_MS
)


def bucket_local_lag_ns(value: Any) -> str:
    """Bucket local observation lag using integer nanoseconds."""
    if pd.isna(value):
        return "UNMATCHED"

    value_int = int(value)

    if value_int < 0:
        return "NEGATIVE_INVALID"
    if value_int == 0:
        return "0 ms"
    if value_int <= 1_000_000:
        return "(0, 1] ms"
    if value_int <= 10_000_000:
        return "(1, 10] ms"
    if value_int <= 50_000_000:
        return "(10, 50] ms"
    if value_int <= 100_000_000:
        return "(50, 100] ms"
    if value_int <= 250_000_000:
        return "(100, 250] ms"
    if value_int <= 1_000_000_000:
        return "(250, 1000] ms"

    return "> 1000 ms"


local_strict_all_alignment["staleness_bucket"] = (
    local_strict_all_alignment["local_observation_lag_ns"].map(bucket_local_lag_ns)
)


# ------------------------------------------------------------
# Verify latest eligible-state property
# ------------------------------------------------------------

next_book_positions = candidate_book_positions + 1
has_next_book = (next_book_positions >= 0) & (next_book_positions < len(book_sequence_array))

next_book_sequence = np.full(len(trade_sequence_array), np.iinfo(np.int64).max, dtype=np.int64)
next_book_local_time = np.full(len(trade_sequence_array), np.iinfo(np.int64).max, dtype=np.int64)

next_book_sequence[has_next_book] = book_sequence_array[next_book_positions[has_next_book]]
next_book_local_time[has_next_book] = book_local_time_array[next_book_positions[has_next_book]]

next_book_would_be_eligible = (
    has_next_book
    & (next_book_sequence < trade_sequence_array)
    & (next_book_local_time <= trade_local_time_array)
)

local_strict_all_alignment["next_book_position"] = next_book_positions
local_strict_all_alignment["next_book_would_be_eligible"] = next_book_would_be_eligible


# ------------------------------------------------------------
# Matched-only and unmatched ledgers
# ------------------------------------------------------------

local_strict_matched_alignment = (
    local_strict_all_alignment[
        local_strict_all_alignment["is_local_strict_match"]
    ]
    .copy()
    .reset_index(drop=True)
)

unmatched_trades = (
    local_strict_all_alignment[
        ~local_strict_all_alignment["is_local_strict_match"]
    ]
    .copy()
    .reset_index(drop=True)
)

unmatched_ledger_columns = [
    "trade_id",
    "trade_collector_sequence",
    "trade_local_receipt_time_ns",
    "trade_local_receipt_time",
    "trade_exchange_trade_time_ms",
    "trade_exchange_trade_time",
    "trade_price",
    "trade_quantity",
    "trade_buyer_is_maker",
    "trade_aggressor_side",
    "sequence_upper_position",
    "time_upper_position",
    "alignment_candidate_position",
    "unmatched_reason",
]

unmatched_trades = unmatched_trades[
    [column for column in unmatched_ledger_columns if column in unmatched_trades.columns]
].copy()


# ------------------------------------------------------------
# Alignment gates
# ------------------------------------------------------------

matched_count = int(local_strict_all_alignment["is_local_strict_match"].sum())
unmatched_count = int((~local_strict_all_alignment["is_local_strict_match"]).sum())

future_sequence_violations = int(
    (
        local_strict_all_alignment["is_local_strict_match"]
        & (
            local_strict_all_alignment["book_collector_sequence"].astype("Int64")
            >= local_strict_all_alignment["trade_collector_sequence"].astype("Int64")
        )
    ).sum()
)

future_local_time_violations = int(
    (
        local_strict_all_alignment["is_local_strict_match"]
        & (
            local_strict_all_alignment["book_local_receipt_time_ns"].astype("Int64")
            > local_strict_all_alignment["trade_local_receipt_time_ns"].astype("Int64")
        )
    ).sum()
)

negative_sequence_lags = int(
    (
        local_strict_all_alignment["is_local_strict_match"]
        & (local_strict_all_alignment["sequence_lag"].astype("Int64") <= 0)
    ).sum()
)

negative_local_lags = int(
    (
        local_strict_all_alignment["is_local_strict_match"]
        & (local_strict_all_alignment["local_observation_lag_ns"].astype("Int64") < 0)
    ).sum()
)

unclassified_unmatched_count = int(
    (local_strict_all_alignment["unmatched_reason"] == "UNCLASSIFIED").sum()
)

invalid_candidate_count = int(
    (local_strict_all_alignment["unmatched_reason"] == "INVALID_CANDIDATE_LOCAL_STRICT_VIOLATION").sum()
)

later_eligible_book_count = int(local_strict_all_alignment["next_book_would_be_eligible"].sum())

local_strict_alignment_gates: list[GateResult] = []

local_strict_alignment_gates.append(
    make_gate(
        gate="local_strict_all_alignment_row_count",
        passed=len(local_strict_all_alignment) == EXPECTED_RAW_TRADE_RECORDS,
        severity="BLOCKING",
        detail=f"observed={len(local_strict_all_alignment)} expected={EXPECTED_RAW_TRADE_RECORDS}",
    )
)

local_strict_alignment_gates.append(
    make_gate(
        gate="local_strict_row_conservation",
        passed=(matched_count + unmatched_count) == EXPECTED_RAW_TRADE_RECORDS,
        severity="BLOCKING",
        detail=(
            f"matched={matched_count} unmatched={unmatched_count} "
            f"total={matched_count + unmatched_count} expected={EXPECTED_RAW_TRADE_RECORDS}"
        ),
    )
)

local_strict_alignment_gates.append(
    make_gate(
        gate="local_strict_no_future_sequence_match",
        passed=future_sequence_violations == 0,
        severity="BLOCKING",
        detail=f"future_sequence_violations={future_sequence_violations}",
    )
)

local_strict_alignment_gates.append(
    make_gate(
        gate="local_strict_no_future_local_time_match",
        passed=future_local_time_violations == 0,
        severity="BLOCKING",
        detail=f"future_local_time_violations={future_local_time_violations}",
    )
)

local_strict_alignment_gates.append(
    make_gate(
        gate="local_strict_positive_sequence_lag",
        passed=negative_sequence_lags == 0,
        severity="BLOCKING",
        detail=f"non_positive_sequence_lags={negative_sequence_lags}",
    )
)

local_strict_alignment_gates.append(
    make_gate(
        gate="local_strict_nonnegative_local_lag",
        passed=negative_local_lags == 0,
        severity="BLOCKING",
        detail=f"negative_local_lags={negative_local_lags}",
    )
)

local_strict_alignment_gates.append(
    make_gate(
        gate="local_strict_latest_eligible_book_selected",
        passed=later_eligible_book_count == 0,
        severity="BLOCKING",
        detail=f"later_eligible_book_count={later_eligible_book_count}",
    )
)

local_strict_alignment_gates.append(
    make_gate(
        gate="local_strict_unmatched_reasons_exhaustive",
        passed=unclassified_unmatched_count == 0,
        severity="BLOCKING",
        detail=f"unclassified_unmatched_count={unclassified_unmatched_count}",
    )
)

local_strict_alignment_gates.append(
    make_gate(
        gate="local_strict_no_invalid_candidate",
        passed=invalid_candidate_count == 0,
        severity="BLOCKING",
        detail=f"invalid_candidate_count={invalid_candidate_count}",
    )
)

local_strict_alignment_gates.append(
    make_gate(
        gate="local_strict_unmatched_count",
        passed=unmatched_count == 0,
        severity="WARNING",
        detail=f"unmatched_count={unmatched_count}",
    )
)

local_strict_alignment_gates.append(
    make_gate(
        gate="local_strict_stale_match_count",
        passed=int(local_strict_all_alignment["stale_match_flag"].sum()) == 0,
        severity="WARNING",
        detail=(
            f"stale_threshold_ms={STALE_MATCH_THRESHOLD_MS}; "
            f"stale_match_count={int(local_strict_all_alignment['stale_match_flag'].sum())}"
        ),
    )
)

local_strict_alignment_gate_frame = gate_results_to_frame(local_strict_alignment_gates)

fail_if_blocking_gate_failed(local_strict_alignment_gate_frame)


# ------------------------------------------------------------
# Compact summaries
# ------------------------------------------------------------

lag_summary = (
    local_strict_matched_alignment["local_observation_lag_ms"]
    .describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
    .to_frame()
    .T
)

lag_summary.insert(0, "metric", "local_observation_lag_ms")

alignment_count_summary = pd.DataFrame(
    [
        {
            "policy": PRIMARY_ALIGNMENT_POLICY,
            "raw_trade_count": int(len(raw_trades)),
            "book_state_count": int(len(book_states)),
            "all_alignment_rows": int(len(local_strict_all_alignment)),
            "matched_count": matched_count,
            "unmatched_count": unmatched_count,
            "matched_share": matched_count / len(local_strict_all_alignment),
            "unmatched_share": unmatched_count / len(local_strict_all_alignment),
            "stale_threshold_ms": STALE_MATCH_THRESHOLD_MS,
            "stale_match_count": int(local_strict_all_alignment["stale_match_flag"].sum()),
            "very_stale_threshold_ms": VERY_STALE_MATCH_THRESHOLD_MS,
            "very_stale_match_count": int(local_strict_all_alignment["very_stale_match_flag"].sum()),
        }
    ]
)

staleness_bucket_summary = (
    local_strict_all_alignment
    .groupby("staleness_bucket", dropna=False)
    .size()
    .reset_index(name="row_count")
    .sort_values(
        "staleness_bucket",
        key=lambda series: series.map(
            {
                "0 ms": 0,
                "(0, 1] ms": 1,
                "(1, 10] ms": 2,
                "(10, 50] ms": 3,
                "(50, 100] ms": 4,
                "(100, 250] ms": 5,
                "(250, 1000] ms": 6,
                "> 1000 ms": 7,
                "UNMATCHED": 8,
                "NEGATIVE_INVALID": 9,
            }
        ).fillna(99),
    )
    .reset_index(drop=True)
)

unmatched_reason_summary = (
    local_strict_all_alignment
    .groupby("unmatched_reason", dropna=False)
    .size()
    .reset_index(name="row_count")
    .sort_values("row_count", ascending=False)
    .reset_index(drop=True)
)

display(alignment_count_summary)
display(lag_summary)
display(staleness_bucket_summary)
display(unmatched_reason_summary)
display(local_strict_alignment_gate_frame)

local_strict_all_alignment.head()

,policy,raw_trade_count,book_state_count,all_alignment_rows,matched_count,unmatched_count,matched_share,unmatched_share,stale_threshold_ms,stale_match_count,very_stale_threshold_ms,very_stale_match_count
0,LOCAL_STRICT,67683,35985,67683,67683,0,1.0,0.0,250.0,0,1000.0,0


,metric,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
local_observation_lag_ms,local_observation_lag_ms,67683.0,51.436734,28.649435,0.0,1.5032,6.027,27.8579,52.9217,76.3906,96.0061,101.675,113.3191


,staleness_bucket,row_count
0,0 ms,339
1,"(0, 1] ms",78
2,"(1, 10] ms",6124
3,"(10, 50] ms",25580
4,"(50, 100] ms",34324
5,"(100, 250] ms",1238


,unmatched_reason,row_count
0,MATCHED_LOCAL_STRICT,67683


,gate,status,severity,detail
0,local_strict_all_alignment_row_count,PASS,BLOCKING,observed=67683 expected=67683
1,local_strict_row_conservation,PASS,BLOCKING,matched=67683 unmatched=0 total=67683 expected=67683
2,local_strict_no_future_sequence_match,PASS,BLOCKING,future_sequence_violations=0
3,local_strict_no_future_local_time_match,PASS,BLOCKING,future_local_time_violations=0
4,local_strict_positive_sequence_lag,PASS,BLOCKING,non_positive_sequence_lags=0
5,local_strict_nonnegative_local_lag,PASS,BLOCKING,negative_local_lags=0
6,local_strict_latest_eligible_book_selected,PASS,BLOCKING,later_eligible_book_count=0
7,local_strict_unmatched_reasons_exhaustive,PASS,BLOCKING,unclassified_unmatched_count=0
8,local_strict_no_invalid_candidate,PASS,BLOCKING,invalid_candidate_count=0
9,local_strict_unmatched_count,PASS,WARNING,unmatched_count=0


,source_run_prefix,v0_1_run_id,trade_source_file,trade_raw_row_number,trade_collector_sequence,trade_collector_sequence_source_path,trade_local_receipt_time_ns,trade_local_receipt_time_ns_source_path,trade_local_receipt_time_text,trade_local_receipt_time_text_source_path,trade_payload_path,trade_payload_score,trade_event_type,trade_symbol,trade_exchange_event_time_ms,trade_exchange_event_time_ns,trade_exchange_trade_time_ms,trade_exchange_trade_time_ns,trade_id,trade_price,trade_quantity,trade_buyer_order_id,trade_seller_order_id,trade_buyer_is_maker,trade_ignore_flag,trade_local_receipt_time,trade_exchange_event_time,trade_exchange_trade_time,trade_notional,trade_aggressor_side,book_source_file,book_row_number,book_collector_sequence,book_local_receipt_time_ns,book_state_id,book_exchange_event_time_ms,book_first_update_id,book_final_update_id,book_previous_final_update_id,book_best_bid,book_best_ask,book_best_bid_size,book_best_ask_size,book_spread,book_midpoint,book_microprice,book_l1_imbalance,book_top10_imbalance,book_partition,book_local_receipt_time,book_exchange_event_time,book_exchange_event_time_ns,top10_reconstruction_session_id,top10_collector_sequence,top10_local_receipt_time_ns,top10_local_receipt_time_utc,top10_exchange_event_time_ms,top10_exchange_event_time_utc,top10_first_update_id,top10_final_update_id,top10_top_10_state_sha256,top10_bid_price_1,top10_bid_quantity_1,top10_ask_price_1,top10_ask_quantity_1,top10_bid_price_2,top10_bid_quantity_2,top10_ask_price_2,top10_ask_quantity_2,top10_bid_price_3,top10_bid_quantity_3,top10_ask_price_3,top10_ask_quantity_3,top10_bid_price_4,top10_bid_quantity_4,top10_ask_price_4,top10_ask_quantity_4,top10_bid_price_5,top10_bid_quantity_5,top10_ask_price_5,top10_ask_quantity_5,top10_bid_price_6,top10_bid_quantity_6,top10_ask_price_6,top10_ask_quantity_6,top10_bid_price_7,top10_bid_quantity_7,top10_ask_price_7,top10_ask_quantity_7,top10_bid_price_8,top10_bid_quantity_8,top10_ask_price_8,top10_ask_quantity_8,top10_bid_price_9,top10_bid_quantity_9,top10_ask_price_9,top10_ask_quantity_9,top10_bid_price_10,top10_bid_quantity_10,top10_ask_price_10,top10_ask_quantity_10,top10_partition,top10_spread_ticks,top10_spread_basis_points,top10_interval_multiple_of_nominal,top10_stale_interval,alignment_policy,alignment_candidate_position,sequence_upper_position,time_upper_position,candidate_has_book,local_strict_sequence_condition,local_strict_time_condition,is_local_strict_match,alignment_status,unmatched_reason,sequence_lag,local_observation_lag_ns,local_observation_lag_ms,exchange_time_lag_ms,same_local_timestamp_flag,zero_local_lag_flag,above_nominal_book_interval_flag,stale_match_flag,very_stale_match_flag,staleness_bucket,next_book_position,next_book_would_be_eligible
0,BTCUSDT_spot_20260710T063746Z_c8b5bf12,v0_1_20260714T090616Z_e82325081a81,D:\Clown Project\V0.0\data\raw\trades\BTCUSDT_spot_20260710T063746Z_c8b5bf12_trades.jsonl,1,14,$.collector_sequence,1783665468766951600,$.local_receipt_time_ns,None,None,$.raw_message.data,101,trade,BTCUSDT,1783665469953,1783665469953000000,1783665469952,1783665469952000000,6494596041,63914.37,0.00046,<NA>,<NA>,False,True,2026-07-10 06:37:48.766951600+00:00,2026-07-10 06:37:49.953000+00:00,2026-07-10 06:37:49.952000+00:00,29.400610,BUY,D:\Clown Project\V0.1\data\processed\book\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__02_VISIBLE_BOOK_RECONSTRUCTION__recons...,4,13,1783665468733660800,3,1783665469915,97233590209,97233590215,97233590208,63914.36,63914.37,2.7236,1.82138,0.01,63914.365,63914.365993,0.198509,NaN,DEVELOPMENT,2026-07-10 06:37:48.733660800+00:00,2026-07-10 06:37:49.915000+00:00,1783665469915000000,v0_1_20260714T090616Z_e82325081a81__book_session_0001,13,1783665468733660800,2026-07-10T06:37:48.733660800+00:00,1783665469915,2026-07-10T06:37:49.915000+00:00,97233590209,97233590215,13e41687b86b8c9d8caa692192e51f204d4daeb9c42be45ab17c83c45fabee17,63914.36,2.7236,63914.37,1.82138,63914.35,0.00369,63914.38,0.00

In [9]:
# ============================================================
# Cell 08 — Lag, staleness, sequence-order, and unmatched audits
# ============================================================

# This cell converts the LOCAL_STRICT alignment result into auditable tables:
#
#   1. unmatched_trades
#   2. match_lag_distribution
#   3. staleness_audit
#   4. sequence_order_audit
#
# These are audit outputs, not modeling features.


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def safe_quantile(series: pd.Series, q: float) -> float:
    """Return a numeric quantile or NaN for empty/all-null input."""
    numeric = pd.to_numeric(series, errors="coerce").dropna()

    if numeric.empty:
        return float("nan")

    return float(numeric.quantile(q))


def safe_mean(series: pd.Series) -> float:
    """Return numeric mean or NaN for empty/all-null input."""
    numeric = pd.to_numeric(series, errors="coerce").dropna()

    if numeric.empty:
        return float("nan")

    return float(numeric.mean())


def safe_median(series: pd.Series) -> float:
    """Return numeric median or NaN for empty/all-null input."""
    numeric = pd.to_numeric(series, errors="coerce").dropna()

    if numeric.empty:
        return float("nan")

    return float(numeric.median())


def safe_max(series: pd.Series) -> float:
    """Return numeric max or NaN for empty/all-null input."""
    numeric = pd.to_numeric(series, errors="coerce").dropna()

    if numeric.empty:
        return float("nan")

    return float(numeric.max())


def safe_min(series: pd.Series) -> float:
    """Return numeric min or NaN for empty/all-null input."""
    numeric = pd.to_numeric(series, errors="coerce").dropna()

    if numeric.empty:
        return float("nan")

    return float(numeric.min())


def summarize_alignment_group(frame: pd.DataFrame, group_name: str, group_value: Any) -> dict[str, Any]:
    """Summarize lag and staleness behavior for one alignment group."""
    matched = frame[frame["is_local_strict_match"]].copy()

    row_count = int(len(frame))
    matched_count = int(len(matched))
    unmatched_count = row_count - matched_count

    return {
        "group_name": group_name,
        "group_value": str(group_value),
        "row_count": row_count,
        "matched_count": matched_count,
        "unmatched_count": unmatched_count,
        "matched_share": matched_count / row_count if row_count else np.nan,
        "unmatched_share": unmatched_count / row_count if row_count else np.nan,

        "sequence_lag_min": safe_min(matched["sequence_lag"]),
        "sequence_lag_p01": safe_quantile(matched["sequence_lag"], 0.01),
        "sequence_lag_p05": safe_quantile(matched["sequence_lag"], 0.05),
        "sequence_lag_p25": safe_quantile(matched["sequence_lag"], 0.25),
        "sequence_lag_p50": safe_quantile(matched["sequence_lag"], 0.50),
        "sequence_lag_p75": safe_quantile(matched["sequence_lag"], 0.75),
        "sequence_lag_p95": safe_quantile(matched["sequence_lag"], 0.95),
        "sequence_lag_p99": safe_quantile(matched["sequence_lag"], 0.99),
        "sequence_lag_max": safe_max(matched["sequence_lag"]),

        "local_lag_ms_min": safe_min(matched["local_observation_lag_ms"]),
        "local_lag_ms_p01": safe_quantile(matched["local_observation_lag_ms"], 0.01),
        "local_lag_ms_p05": safe_quantile(matched["local_observation_lag_ms"], 0.05),
        "local_lag_ms_p25": safe_quantile(matched["local_observation_lag_ms"], 0.25),
        "local_lag_ms_p50": safe_quantile(matched["local_observation_lag_ms"], 0.50),
        "local_lag_ms_p75": safe_quantile(matched["local_observation_lag_ms"], 0.75),
        "local_lag_ms_p95": safe_quantile(matched["local_observation_lag_ms"], 0.95),
        "local_lag_ms_p99": safe_quantile(matched["local_observation_lag_ms"], 0.99),
        "local_lag_ms_max": safe_max(matched["local_observation_lag_ms"]),

        "exchange_time_lag_ms_min": safe_min(matched["exchange_time_lag_ms"]),
        "exchange_time_lag_ms_p50": safe_quantile(matched["exchange_time_lag_ms"], 0.50),
        "exchange_time_lag_ms_p95": safe_quantile(matched["exchange_time_lag_ms"], 0.95),
        "exchange_time_lag_ms_p99": safe_quantile(matched["exchange_time_lag_ms"], 0.99),
        "exchange_time_lag_ms_max": safe_max(matched["exchange_time_lag_ms"]),

        "zero_local_lag_count": int(frame["zero_local_lag_flag"].fillna(False).sum()),
        "above_nominal_book_interval_count": int(frame["above_nominal_book_interval_flag"].fillna(False).sum()),
        "stale_match_count": int(frame["stale_match_flag"].fillna(False).sum()),
        "very_stale_match_count": int(frame["very_stale_match_flag"].fillna(False).sum()),
        "zero_local_lag_share": int(frame["zero_local_lag_flag"].fillna(False).sum()) / row_count if row_count else np.nan,
        "above_nominal_book_interval_share": int(frame["above_nominal_book_interval_flag"].fillna(False).sum()) / row_count if row_count else np.nan,
        "stale_match_share": int(frame["stale_match_flag"].fillna(False).sum()) / row_count if row_count else np.nan,
        "very_stale_match_share": int(frame["very_stale_match_flag"].fillna(False).sum()) / row_count if row_count else np.nan,
    }


def write_csv_and_verify(frame: pd.DataFrame, path: Path, expected_rows: int | None = None) -> dict[str, Any]:
    """Write a CSV, read it back, verify row count, and return file metadata."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    frame.to_csv(path, index=False)

    read_back = pd.read_csv(path, low_memory=False)

    if expected_rows is not None:
        require(
            len(read_back) == expected_rows,
            f"Read-back row count mismatch for {path.name}: observed={len(read_back)} expected={expected_rows}",
        )

    return {
        "path": str(path),
        "row_count": int(len(read_back)),
        "size_bytes": int(path.stat().st_size),
        "sha256": sha256_file(path),
    }


# ------------------------------------------------------------
# Match-lag distribution
# ------------------------------------------------------------

match_lag_distribution_rows = []

match_lag_distribution_rows.append(
    summarize_alignment_group(
        local_strict_all_alignment,
        group_name="ALL",
        group_value="ALL",
    )
)

for bucket, bucket_frame in local_strict_all_alignment.groupby("staleness_bucket", dropna=False):
    match_lag_distribution_rows.append(
        summarize_alignment_group(
            bucket_frame,
            group_name="staleness_bucket",
            group_value=bucket,
        )
    )

if "trade_aggressor_side" in local_strict_all_alignment.columns:
    for side, side_frame in local_strict_all_alignment.groupby("trade_aggressor_side", dropna=False):
        match_lag_distribution_rows.append(
            summarize_alignment_group(
                side_frame,
                group_name="trade_aggressor_side",
                group_value=side,
            )
        )

if "book_partition" in local_strict_all_alignment.columns:
    for partition, partition_frame in local_strict_all_alignment.groupby("book_partition", dropna=False):
        match_lag_distribution_rows.append(
            summarize_alignment_group(
                partition_frame,
                group_name="book_partition",
                group_value=partition,
            )
        )

match_lag_distribution = pd.DataFrame(match_lag_distribution_rows)


# ------------------------------------------------------------
# Staleness audit
# ------------------------------------------------------------

staleness_order = {
    "0 ms": 0,
    "(0, 1] ms": 1,
    "(1, 10] ms": 2,
    "(10, 50] ms": 3,
    "(50, 100] ms": 4,
    "(100, 250] ms": 5,
    "(250, 1000] ms": 6,
    "> 1000 ms": 7,
    "UNMATCHED": 8,
    "NEGATIVE_INVALID": 9,
}

staleness_audit = (
    local_strict_all_alignment
    .groupby("staleness_bucket", dropna=False)
    .agg(
        row_count=("trade_id", "size"),
        matched_count=("is_local_strict_match", "sum"),
        sequence_lag_min=("sequence_lag", "min"),
        sequence_lag_median=("sequence_lag", "median"),
        sequence_lag_max=("sequence_lag", "max"),
        local_lag_ms_min=("local_observation_lag_ms", "min"),
        local_lag_ms_mean=("local_observation_lag_ms", "mean"),
        local_lag_ms_median=("local_observation_lag_ms", "median"),
        local_lag_ms_max=("local_observation_lag_ms", "max"),
        exchange_lag_ms_min=("exchange_time_lag_ms", "min"),
        exchange_lag_ms_median=("exchange_time_lag_ms", "median"),
        exchange_lag_ms_max=("exchange_time_lag_ms", "max"),
        buy_count=("trade_aggressor_side", lambda x: int((x == "BUY").sum())),
        sell_count=("trade_aggressor_side", lambda x: int((x == "SELL").sum())),
        zero_local_lag_count=("zero_local_lag_flag", "sum"),
        above_nominal_book_interval_count=("above_nominal_book_interval_flag", "sum"),
        stale_match_count=("stale_match_flag", "sum"),
        very_stale_match_count=("very_stale_match_flag", "sum"),
    )
    .reset_index()
)

staleness_audit["unmatched_count"] = (
    staleness_audit["row_count"] - staleness_audit["matched_count"]
)

staleness_audit["row_share"] = (
    staleness_audit["row_count"] / len(local_strict_all_alignment)
)

staleness_audit["matched_share_within_bucket"] = (
    staleness_audit["matched_count"] / staleness_audit["row_count"]
)

staleness_audit["sort_order"] = (
    staleness_audit["staleness_bucket"].map(staleness_order).fillna(99).astype(int)
)

staleness_audit = (
    staleness_audit
    .sort_values("sort_order")
    .drop(columns=["sort_order"])
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Sequence-order audit
# ------------------------------------------------------------

sequence_order_audit_rows = []

matched_frame = local_strict_all_alignment[
    local_strict_all_alignment["is_local_strict_match"]
].copy()

sequence_order_audit_rows.append(
    {
        "audit_name": "matched_rows",
        "row_count": int(len(matched_frame)),
        "violation_count": 0,
        "detail": "Rows matched under LOCAL_STRICT.",
    }
)

sequence_order_audit_rows.append(
    {
        "audit_name": "book_sequence_less_than_trade_sequence",
        "row_count": int(len(matched_frame)),
        "violation_count": int(
            (
                matched_frame["book_collector_sequence"].astype("Int64")
                >= matched_frame["trade_collector_sequence"].astype("Int64")
            ).sum()
        ),
        "detail": "Requires book_collector_sequence < trade_collector_sequence.",
    }
)

sequence_order_audit_rows.append(
    {
        "audit_name": "book_local_receipt_not_after_trade_local_receipt",
        "row_count": int(len(matched_frame)),
        "violation_count": int(
            (
                matched_frame["book_local_receipt_time_ns"].astype("Int64")
                > matched_frame["trade_local_receipt_time_ns"].astype("Int64")
            ).sum()
        ),
        "detail": "Requires book_local_receipt_time_ns <= trade_local_receipt_time_ns.",
    }
)

sequence_order_audit_rows.append(
    {
        "audit_name": "sequence_lag_positive",
        "row_count": int(len(matched_frame)),
        "violation_count": int((matched_frame["sequence_lag"].astype("Int64") <= 0).sum()),
        "detail": "Requires positive sequence lag for matched rows.",
    }
)

sequence_order_audit_rows.append(
    {
        "audit_name": "local_lag_nonnegative",
        "row_count": int(len(matched_frame)),
        "violation_count": int((matched_frame["local_observation_lag_ns"].astype("Int64") < 0).sum()),
        "detail": "Requires nonnegative local observation lag for matched rows.",
    }
)

sequence_order_audit_rows.append(
    {
        "audit_name": "latest_eligible_book_selected",
        "row_count": int(len(local_strict_all_alignment)),
        "violation_count": int(local_strict_all_alignment["next_book_would_be_eligible"].sum()),
        "detail": "Next book row must not also satisfy LOCAL_STRICT eligibility.",
    }
)

sequence_order_audit_rows.append(
    {
        "audit_name": "all_trades_accounted_for",
        "row_count": int(len(local_strict_all_alignment)),
        "violation_count": int(len(local_strict_all_alignment) != EXPECTED_RAW_TRADE_RECORDS),
        "detail": f"Expected {EXPECTED_RAW_TRADE_RECORDS} aligned trade rows.",
    }
)

sequence_order_audit_rows.append(
    {
        "audit_name": "matched_plus_unmatched_conservation",
        "row_count": int(len(local_strict_all_alignment)),
        "violation_count": int(
            (
                int(local_strict_all_alignment["is_local_strict_match"].sum())
                + int((~local_strict_all_alignment["is_local_strict_match"]).sum())
            )
            != EXPECTED_RAW_TRADE_RECORDS
        ),
        "detail": "Matched plus unmatched rows must equal raw trade count.",
    }
)

sequence_order_audit = pd.DataFrame(sequence_order_audit_rows)


# ------------------------------------------------------------
# Unmatched ledger normalization
# ------------------------------------------------------------

if unmatched_trades.empty:
    unmatched_trades = pd.DataFrame(
        columns=[
            "trade_id",
            "trade_collector_sequence",
            "trade_local_receipt_time_ns",
            "trade_local_receipt_time",
            "trade_exchange_trade_time_ms",
            "trade_exchange_trade_time",
            "trade_price",
            "trade_quantity",
            "trade_buyer_is_maker",
            "trade_aggressor_side",
            "sequence_upper_position",
            "time_upper_position",
            "alignment_candidate_position",
            "unmatched_reason",
        ]
    )

unmatched_reason_audit = (
    local_strict_all_alignment
    .groupby("unmatched_reason", dropna=False)
    .size()
    .reset_index(name="row_count")
    .sort_values("row_count", ascending=False)
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Audit gates
# ------------------------------------------------------------

lag_staleness_sequence_gates: list[GateResult] = []

lag_staleness_sequence_gates.append(
    make_gate(
        gate="match_lag_distribution_not_empty",
        passed=not match_lag_distribution.empty,
        severity="BLOCKING",
        detail=f"rows={len(match_lag_distribution)}",
    )
)

lag_staleness_sequence_gates.append(
    make_gate(
        gate="staleness_audit_row_sum",
        passed=int(staleness_audit["row_count"].sum()) == EXPECTED_RAW_TRADE_RECORDS,
        severity="BLOCKING",
        detail=f"observed_sum={int(staleness_audit['row_count'].sum())} expected={EXPECTED_RAW_TRADE_RECORDS}",
    )
)

lag_staleness_sequence_gates.append(
    make_gate(
        gate="sequence_order_audit_no_violations",
        passed=int(sequence_order_audit["violation_count"].sum()) == 0,
        severity="BLOCKING",
        detail=f"total_sequence_order_violations={int(sequence_order_audit['violation_count'].sum())}",
    )
)

lag_staleness_sequence_gates.append(
    make_gate(
        gate="unmatched_ledger_row_count",
        passed=len(unmatched_trades) == int((~local_strict_all_alignment["is_local_strict_match"]).sum()),
        severity="BLOCKING",
        detail=(
            f"unmatched_ledger_rows={len(unmatched_trades)} "
            f"expected={int((~local_strict_all_alignment['is_local_strict_match']).sum())}"
        ),
    )
)

lag_staleness_sequence_gates.append(
    make_gate(
        gate="no_negative_staleness_bucket",
        passed=not (staleness_audit["staleness_bucket"] == "NEGATIVE_INVALID").any(),
        severity="BLOCKING",
        detail="NEGATIVE_INVALID bucket must not appear after LOCAL_STRICT alignment.",
    )
)

lag_staleness_sequence_gates.append(
    make_gate(
        gate="no_unmatched_rows",
        passed=len(unmatched_trades) == 0,
        severity="WARNING",
        detail=f"unmatched_rows={len(unmatched_trades)}",
    )
)

lag_staleness_sequence_gate_frame = gate_results_to_frame(lag_staleness_sequence_gates)

fail_if_blocking_gate_failed(lag_staleness_sequence_gate_frame)


# ------------------------------------------------------------
# Write audit outputs and verify read-back
# ------------------------------------------------------------

unmatched_trades_metadata = write_csv_and_verify(
    unmatched_trades,
    UNMATCHED_TRADES_PATH,
    expected_rows=len(unmatched_trades),
)

match_lag_distribution_metadata = write_csv_and_verify(
    match_lag_distribution,
    MATCH_LAG_DISTRIBUTION_PATH,
    expected_rows=len(match_lag_distribution),
)

staleness_audit_metadata = write_csv_and_verify(
    staleness_audit,
    STALENESS_AUDIT_PATH,
    expected_rows=len(staleness_audit),
)

sequence_order_audit_metadata = write_csv_and_verify(
    sequence_order_audit,
    SEQUENCE_ORDER_AUDIT_PATH,
    expected_rows=len(sequence_order_audit),
)

lag_staleness_sequence_output_metadata = {
    "unmatched_trades": unmatched_trades_metadata,
    "match_lag_distribution": match_lag_distribution_metadata,
    "staleness_audit": staleness_audit_metadata,
    "sequence_order_audit": sequence_order_audit_metadata,
}


# ------------------------------------------------------------
# Compact display
# ------------------------------------------------------------

display(match_lag_distribution.head(20))
display(staleness_audit)
display(sequence_order_audit)
display(unmatched_reason_audit)
display(lag_staleness_sequence_gate_frame)

lag_staleness_sequence_output_metadata

,group_name,group_value,row_count,matched_count,unmatched_count,matched_share,unmatched_share,sequence_lag_min,sequence_lag_p01,sequence_lag_p05,sequence_lag_p25,sequence_lag_p50,sequence_lag_p75,sequence_lag_p95,sequence_lag_p99,sequence_lag_max,local_lag_ms_min,local_lag_ms_p01,local_lag_ms_p05,local_lag_ms_p25,local_lag_ms_p50,local_lag_ms_p75,local_lag_ms_p95,local_lag_ms_p99,local_lag_ms_max,exchange_time_lag_ms_min,exchange_time_lag_ms_p50,exchange_time_lag_ms_p95,exchange_time_lag_ms_p99,exchange_time_lag_ms_max,zero_local_lag_count,above_nominal_book_interval_count,stale_match_count,very_stale_match_count,zero_local_lag_share,above_nominal_book_interval_share,stale_match_share,very_stale_match_share
0,ALL,ALL,67683,67683,0,1.0,0.0,1.0,1.0,1.0,5.0,30.0,74.00,173.00,265.00,461.0,0.0000,1.503200,6.027000,27.857900,52.92170,76.390600,96.006100,101.675000,113.3191,-16.0,48.0,92.0,97.00,102.0,339,1238,0,0,0.005009,0.018291,0.0,0.0
1,staleness_bucket,"(0, 1] ms",78,78,0,1.0,0.0,1.0,1.0,1.0,1.0,2.0,7.75,13.00,14.23,15.0,0.4529,0.491323,0.502800,0.564225,0.66310,0.815875,0.999915,1.000000,1.0000,-12.0,0.0,2.0,3.38,8.0,0,0,0,0,0.000000,0.000000,0.0,0.0
2,staleness_bucket,"(1, 10] ms",6124,6124,0,1.0,0.0,1.0,1.0,1.0,3.0,16.0,35.00,68.00,91.00,123.0,1.0001,1.002600,1.579150,4.242550,6.08260,7.843300,9.568900,9.933400,9.9997,-16.0,1.0,9.0,12.00,17.0,0,0,0,0,0.000000,0.000000,0.0,0.0
3,staleness_bucket,"(10, 50] ms",25580,25580,0,1.0,0.0,1.0,1.0,1.0,5.0,30.0,72.00,153.00,217.00,277.0,10.0055,10.252200,12.003800,21.111500,31.04620,40.188900,48.193600,49.804041,49.9913,-8.0,26.0,45.0,49.00,59.0,0,0,0,0,0.000000,0.000000,0.0,0.0
4,staleness_bucket,"(100, 250] ms",1238,1238,0,1.0,0.0,1.0,1.0,1.0,22.0,54.0,97.00,187.15,275.63,288.0,100.0068,100.037800,100.105345,100.686400,101.76650,103.853100,110.318400,112.318800,113.3191,81.0,96.0,99.0,99.00,102.0,0,1238,0,0,0.000000,1.000000,0.0,0.0
5,staleness_bucket,"(50, 100] ms",34324,34324,0,1.0,0.0,1.0,1.0,1.0,6.0,35.0,87.00,198.00,300.00,461.0,50.0028,50.428400,52.921700,61.104700,74.00430,84.697800,96.649100,99.306400,99.9990,6.0,70.0,94.0,97.00,100.0,0,0,0,0,0.000000,0.000000,0.0,0.0
6,staleness_bucket,0 ms,339,339,0,1.0,0.0,1.0,1.0,1.0,1.0,5.0,19.50,41.00,47.62,51.0,0.0000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.0000,-8.0,0.0,3.0,4.00,6.0,339,0,0,0,1.000000,0.000000,0.0,0.0
7,trade_aggressor_side,BUY,30596,30596,0,1.0,0.0,1.0,1.0,1.0,5.0,31.0,73.00,156.00,228.00,348.0,0.0000,1.262600,5.758900,29.025600,53.53615,78.021300,96.649100,101.742600,113.3191,-15.0,51.0,93.0,98.00,102.0,208,558,0,0,0.006798,0.018238,0.0,0.0
8,trade_aggressor_side,SELL,37087,37087,0,1.0,0.0,1.0,1.0,1.0,5.0,29.0,76.00,188.00,288.00,461.0,0.0000,1.560928,6.177100,26.751900,52.04780,74.192400,95.120350,101.675000,108.5651,-16.0,46.0,92.0,97.00,100.0,131,680,0,0,0.003532,0.018335,0.0,0.0
9,book_partition,CALIBRATION,13533,13533,0,1.0,0.0,1.0,1.0,1.0,7.0,33.0,78.00,158.00,222.00,271.0,0.0000,1.002500,6.082600,26.365100,53.97290,76.963500,95.031400,100.277100,108.0506,-2.0,50.0,92.0,98.00,100.0,85,162,0,0,0.006281,0.011971,0.0,0.0


,staleness_bucket,row_count,matched_count,sequence_lag_min,sequence_lag_median,sequence_lag_max,local_lag_ms_min,local_lag_ms_mean,local_lag_ms_median,local_lag_ms_max,exchange_lag_ms_min,exchange_lag_ms_median,exchange_lag_ms_max,buy_count,sell_count,zero_local_lag_count,above_nominal_book_interval_count,stale_match_count,very_stale_match_count,unmatched_count,row_share,matched_share_within_bucket
0,0 ms,339,339,1,5.0,51,0.0,0.0,0.0,0.0,-8,0.0,6,208,131,339,0,0,0,0,0.005009,1.0
1,"(0, 1] ms",78,78,1,2.0,15,0.4529,0.696918,0.6631,1.0,-12,0.0,8,27,51,0,0,0,0,0,0.001152,1.0
2,"(1, 10] ms",6124,6124,1,16.0,123,1.0001,5.959236,6.0826,9.9997,-16,1.0,17,2616,3508,0,0,0,0,0,0.090481,1.0
3,"(10, 50] ms",25580,25580,1,30.0,277,10.0055,30.575918,31.0462,49.9913,-8,26.0,59,11331,14249,0,0,0,0,0,0.377938,1.0
4,"(50, 100] ms",34324,34324,1,35.0,461,50.0028,73.864684,74.0043,99.999,6,70.0,100,15856,18468,0,0,0,0,0,0.507129,1.0
5,"(100, 250] ms",1238,1238,1,54.0,288,100.0068,102.892027,101.7665,113.3191,81,96.0,102,558,680,0,1238,0,0,0,0.018291,1.0


,audit_name,row_count,violation_count,detail
0,matched_rows,67683,0,Rows matched under LOCAL_STRICT.
1,book_sequence_less_than_trade_sequence,67683,0,Requires book_collector_sequence < trade_collector_sequence.
2,book_local_receipt_not_after_trade_local_receipt,67683,0,Requires book_local_receipt_time_ns <= trade_local_receipt_time_ns.
3,sequence_lag_positive,67683,0,Requires positive sequence lag for matched rows.
4,local_lag_nonnegative,67683,0,Requires nonnegative local observation lag for matched rows.
5,latest_eligible_book_selected,67683,0,Next book row must not also satisfy LOCAL_STRICT eligibility.
6,all_trades_accounted_for,67683,0,Expected 67683 aligned trade rows.
7,matched_plus_unmatched_conservation,67683,0,Matched plus unmatched rows must equal raw trade count.


,unmatched_reason,row_count
0,MATCHED_LOCAL_STRICT,67683


,gate,status,severity,detail
0,match_lag_distribution_not_empty,PASS,BLOCKING,rows=13
1,staleness_audit_row_sum,PASS,BLOCKING,observed_sum=67683 expected=67683
2,sequence_order_audit_no_violations,PASS,BLOCKING,total_sequence_order_violations=0
3,unmatched_ledger_row_count,PASS,BLOCKING,unmatched_ledger_rows=0 expected=0
4,no_negative_staleness_bucket,PASS,BLOCKING,NEGATIVE_INVALID bucket must not appear after LOCAL_STRICT alignment.
5,no_unmatched_rows,PASS,WARNING,unmatched_rows=0


{'unmatched_trades': {'path': 'D:\\Clown Project\\V0.1\\artifacts\\audit_tables\\03_CAUSAL_TRADE_BOOK_ALIGNMENT\\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__03_CAUSAL_TRADE_BOOK_ALIGNMENT__unmatched_trades.csv',
  'row_count': 0,
  'size_bytes': 302,
  'sha256': 'f03f97ff507f921370b2d464cb6913f33cae0a8ee9fae094a7896146608c21e3'},
 'match_lag_distribution': {'path': 'D:\\Clown Project\\V0.1\\artifacts\\audit_tables\\03_CAUSAL_TRADE_BOOK_ALIGNMENT\\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__03_CAUSAL_TRADE_BOOK_ALIGNMENT__match_lag_distribution.csv',
  'row_count': 13,
  'size_bytes': 3826,
  'sha256': '44c54ef973a093ab6a8c6c26b1870fc762a2e1bca94a1b59d132e7fdd2b34b1f'},
 'staleness_audit': {'path': 'D:\\Clown Project\\V0.1\\artifacts\\audit_tables\\03_CAUSAL_TRADE_BOOK_ALIGNMENT\\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__03_CAUSAL_TRADE_BOOK_ALIGNMENT__staleness_audit.csv',
  'row_count': 6

In [11]:
# ============================================================
# Cell 09 — Price-versus-book and aggressor-side consistency audits
# ============================================================

# This cell audits whether each trade price is compatible with the matched
# prior visible book state.
#
# These checks are diagnostic only.
#
# They must not delete rows and must not override LOCAL_STRICT authority.
# A trade/book mismatch can be caused by stream latency, hidden liquidity,
# same-millisecond ordering, partial visibility, or a genuine stale-state effect.


# ------------------------------------------------------------
# Tick-size inference and tick conversion
# ------------------------------------------------------------

def infer_price_tick_size(frame: pd.DataFrame, price_columns: list[str]) -> float:
    """
    Infer a conservative tick size from observed decimal prices.

    For this Binance BTCUSDT collection, 0.01 is expected.
    The function still checks the loaded data rather than blindly assuming it.
    """
    observed_values = []

    for column in price_columns:
        if column in frame.columns:
            numeric = pd.to_numeric(frame[column], errors="coerce").dropna()
            if not numeric.empty:
                observed_values.extend(numeric.head(10_000).tolist())

    require(
        len(observed_values) > 0,
        "Cannot infer tick size: no observed price values.",
    )

    scaled_100 = np.array(observed_values, dtype=float) * 100.0
    cents_aligned_share = float(np.mean(np.isclose(scaled_100, np.round(scaled_100), atol=1e-6)))

    if cents_aligned_share > 0.999:
        return 0.01

    scaled_1000 = np.array(observed_values, dtype=float) * 1000.0
    mills_aligned_share = float(np.mean(np.isclose(scaled_1000, np.round(scaled_1000), atol=1e-6)))

    if mills_aligned_share > 0.999:
        return 0.001

    scaled_10000 = np.array(observed_values, dtype=float) * 10000.0
    ten_thousandths_aligned_share = float(np.mean(np.isclose(scaled_10000, np.round(scaled_10000), atol=1e-6)))

    if ten_thousandths_aligned_share > 0.999:
        return 0.0001

    raise AssertionError(
        "Could not infer a stable decimal tick size from price columns."
    )


def price_to_tick_index(series: pd.Series, tick_size: float) -> pd.Series:
    """Convert prices to integer tick indices using the inferred tick size."""
    numeric = pd.to_numeric(series, errors="coerce")
    tick_index = np.rint(numeric / tick_size)

    return pd.Series(tick_index, index=series.index).astype("Int64")


PRICE_TICK_SIZE = infer_price_tick_size(
    local_strict_matched_alignment,
    [
        "trade_price",
        "book_best_bid",
        "book_best_ask",
        "book_midpoint",
        "book_microprice",
    ],
)

require(
    math.isclose(PRICE_TICK_SIZE, 0.01, rel_tol=0.0, abs_tol=1e-12),
    f"Unexpected inferred BTCUSDT price tick size: {PRICE_TICK_SIZE}",
)


# ------------------------------------------------------------
# Build row-level price/book audit table
# ------------------------------------------------------------

price_vs_book_row_audit = local_strict_matched_alignment[
    [
        "trade_id",
        "trade_collector_sequence",
        "book_collector_sequence",
        "trade_local_receipt_time_ns",
        "book_local_receipt_time_ns",
        "trade_exchange_trade_time_ms",
        "book_exchange_event_time_ms",
        "trade_aggressor_side",
        "trade_price",
        "trade_quantity",
        "trade_notional",
        "book_best_bid",
        "book_best_ask",
        "book_spread",
        "book_midpoint",
        "book_microprice",
        "book_best_bid_size",
        "book_best_ask_size",
        "book_l1_imbalance",
        "book_partition",
        "sequence_lag",
        "local_observation_lag_ns",
        "local_observation_lag_ms",
        "exchange_time_lag_ms",
        "staleness_bucket",
        "stale_match_flag",
        "very_stale_match_flag",
    ]
].copy()

price_vs_book_row_audit["price_tick_size"] = PRICE_TICK_SIZE

price_vs_book_row_audit["trade_price_tick"] = price_to_tick_index(
    price_vs_book_row_audit["trade_price"],
    PRICE_TICK_SIZE,
)

price_vs_book_row_audit["book_best_bid_tick"] = price_to_tick_index(
    price_vs_book_row_audit["book_best_bid"],
    PRICE_TICK_SIZE,
)

price_vs_book_row_audit["book_best_ask_tick"] = price_to_tick_index(
    price_vs_book_row_audit["book_best_ask"],
    PRICE_TICK_SIZE,
)

price_vs_book_row_audit["book_spread_ticks"] = (
    price_vs_book_row_audit["book_best_ask_tick"]
    - price_vs_book_row_audit["book_best_bid_tick"]
).astype("Int64")

price_vs_book_row_audit["trade_minus_bid_ticks"] = (
    price_vs_book_row_audit["trade_price_tick"]
    - price_vs_book_row_audit["book_best_bid_tick"]
).astype("Int64")

price_vs_book_row_audit["trade_minus_ask_ticks"] = (
    price_vs_book_row_audit["trade_price_tick"]
    - price_vs_book_row_audit["book_best_ask_tick"]
).astype("Int64")

price_vs_book_row_audit["trade_minus_midpoint"] = (
    price_vs_book_row_audit["trade_price"]
    - price_vs_book_row_audit["book_midpoint"]
)

price_vs_book_row_audit["trade_minus_microprice"] = (
    price_vs_book_row_audit["trade_price"]
    - price_vs_book_row_audit["book_microprice"]
)


# ------------------------------------------------------------
# General visible-book location classification
# ------------------------------------------------------------

bid_tick = price_vs_book_row_audit["book_best_bid_tick"]
ask_tick = price_vs_book_row_audit["book_best_ask_tick"]
trade_tick = price_vs_book_row_audit["trade_price_tick"]

price_vs_book_row_audit["price_vs_book_location"] = "UNCLASSIFIED"

price_vs_book_row_audit.loc[
    trade_tick == bid_tick,
    "price_vs_book_location",
] = "AT_BID"

price_vs_book_row_audit.loc[
    trade_tick == ask_tick,
    "price_vs_book_location",
] = "AT_ASK"

price_vs_book_row_audit.loc[
    (trade_tick > bid_tick) & (trade_tick < ask_tick),
    "price_vs_book_location",
] = "INSIDE_SPREAD"

price_vs_book_row_audit.loc[
    trade_tick < bid_tick,
    "price_vs_book_location",
] = "BELOW_BID"

price_vs_book_row_audit.loc[
    trade_tick > ask_tick,
    "price_vs_book_location",
] = "ABOVE_ASK"

price_vs_book_row_audit.loc[
    (
        price_vs_book_row_audit["trade_price_tick"].isna()
        | price_vs_book_row_audit["book_best_bid_tick"].isna()
        | price_vs_book_row_audit["book_best_ask_tick"].isna()
    ),
    "price_vs_book_location",
] = "INVALID_PRICE_OR_BOOK"

price_vs_book_row_audit.loc[
    price_vs_book_row_audit["book_spread_ticks"] <= 0,
    "price_vs_book_location",
] = "INVALID_CROSSED_OR_LOCKED_BOOK"


# ------------------------------------------------------------
# Aggressor-aware execution classification
# ------------------------------------------------------------

price_vs_book_row_audit["aggressor_book_class"] = "UNCLASSIFIED"

buy_mask = price_vs_book_row_audit["trade_aggressor_side"] == "BUY"
sell_mask = price_vs_book_row_audit["trade_aggressor_side"] == "SELL"

price_vs_book_row_audit.loc[
    buy_mask & (trade_tick == ask_tick),
    "aggressor_book_class",
] = "BUY_AT_ASK"

price_vs_book_row_audit.loc[
    buy_mask & (trade_tick > ask_tick),
    "aggressor_book_class",
] = "BUY_THROUGH_ASK"

price_vs_book_row_audit.loc[
    buy_mask & (trade_tick > bid_tick) & (trade_tick < ask_tick),
    "aggressor_book_class",
] = "BUY_INSIDE_SPREAD"

price_vs_book_row_audit.loc[
    buy_mask & (trade_tick == bid_tick),
    "aggressor_book_class",
] = "BUY_AT_BID_INCONSISTENT"

price_vs_book_row_audit.loc[
    buy_mask & (trade_tick < bid_tick),
    "aggressor_book_class",
] = "BUY_BELOW_BID_INCONSISTENT"

price_vs_book_row_audit.loc[
    sell_mask & (trade_tick == bid_tick),
    "aggressor_book_class",
] = "SELL_AT_BID"

price_vs_book_row_audit.loc[
    sell_mask & (trade_tick < bid_tick),
    "aggressor_book_class",
] = "SELL_THROUGH_BID"

price_vs_book_row_audit.loc[
    sell_mask & (trade_tick > bid_tick) & (trade_tick < ask_tick),
    "aggressor_book_class",
] = "SELL_INSIDE_SPREAD"

price_vs_book_row_audit.loc[
    sell_mask & (trade_tick == ask_tick),
    "aggressor_book_class",
] = "SELL_AT_ASK_INCONSISTENT"

price_vs_book_row_audit.loc[
    sell_mask & (trade_tick > ask_tick),
    "aggressor_book_class",
] = "SELL_ABOVE_ASK_INCONSISTENT"

price_vs_book_row_audit.loc[
    price_vs_book_row_audit["price_vs_book_location"].isin(
        ["INVALID_PRICE_OR_BOOK", "INVALID_CROSSED_OR_LOCKED_BOOK"]
    ),
    "aggressor_book_class",
] = price_vs_book_row_audit["price_vs_book_location"]


# ------------------------------------------------------------
# Diagnostic flags
# ------------------------------------------------------------

price_vs_book_row_audit["visible_book_touch_or_through_flag"] = (
    price_vs_book_row_audit["aggressor_book_class"].isin(
        [
            "BUY_AT_ASK",
            "BUY_THROUGH_ASK",
            "SELL_AT_BID",
            "SELL_THROUGH_BID",
        ]
    )
)

price_vs_book_row_audit["inside_spread_trade_flag"] = (
    price_vs_book_row_audit["aggressor_book_class"].isin(
        [
            "BUY_INSIDE_SPREAD",
            "SELL_INSIDE_SPREAD",
        ]
    )
)

price_vs_book_row_audit["aggressor_book_inconsistency_flag"] = (
    price_vs_book_row_audit["aggressor_book_class"].isin(
        [
            "BUY_AT_BID_INCONSISTENT",
            "BUY_BELOW_BID_INCONSISTENT",
            "SELL_AT_ASK_INCONSISTENT",
            "SELL_ABOVE_ASK_INCONSISTENT",
        ]
    )
)

price_vs_book_row_audit["outside_visible_book_flag"] = (
    price_vs_book_row_audit["price_vs_book_location"].isin(
        [
            "BELOW_BID",
            "ABOVE_ASK",
        ]
    )
)

price_vs_book_row_audit["wide_spread_state_match_flag"] = (
    price_vs_book_row_audit["book_spread_ticks"] > 1
)

price_vs_book_row_audit["price_book_warning_class"] = "NO_PRICE_BOOK_WARNING"

price_vs_book_row_audit.loc[
    price_vs_book_row_audit["inside_spread_trade_flag"],
    "price_book_warning_class",
] = "INSIDE_SPREAD_TRADE"

price_vs_book_row_audit.loc[
    price_vs_book_row_audit["outside_visible_book_flag"],
    "price_book_warning_class",
] = "OUTSIDE_VISIBLE_BOOK"

price_vs_book_row_audit.loc[
    price_vs_book_row_audit["aggressor_book_inconsistency_flag"],
    "price_book_warning_class",
] = "AGGRESSOR_BOOK_INCONSISTENCY"

price_vs_book_row_audit.loc[
    price_vs_book_row_audit["wide_spread_state_match_flag"]
    & (price_vs_book_row_audit["price_book_warning_class"] == "NO_PRICE_BOOK_WARNING"),
    "price_book_warning_class",
] = "WIDE_SPREAD_STATE_MATCH"


# ------------------------------------------------------------
# Price-versus-book consistency audit summary
# ------------------------------------------------------------

price_vs_book_consistency_audit = (
    price_vs_book_row_audit
    .groupby(
        [
            "price_vs_book_location",
            "aggressor_book_class",
            "price_book_warning_class",
            "staleness_bucket",
        ],
        dropna=False,
    )
    .agg(
        row_count=("trade_id", "size"),
        buy_count=("trade_aggressor_side", lambda x: int((x == "BUY").sum())),
        sell_count=("trade_aggressor_side", lambda x: int((x == "SELL").sum())),
        total_quantity=("trade_quantity", "sum"),
        total_notional=("trade_notional", "sum"),
        local_lag_ms_mean=("local_observation_lag_ms", "mean"),
        local_lag_ms_median=("local_observation_lag_ms", "median"),
        local_lag_ms_max=("local_observation_lag_ms", "max"),
        sequence_lag_median=("sequence_lag", "median"),
        sequence_lag_max=("sequence_lag", "max"),
        spread_ticks_median=("book_spread_ticks", "median"),
        spread_ticks_max=("book_spread_ticks", "max"),
        stale_match_count=("stale_match_flag", "sum"),
        very_stale_match_count=("very_stale_match_flag", "sum"),
    )
    .reset_index()
)

price_vs_book_consistency_audit["row_share"] = (
    price_vs_book_consistency_audit["row_count"]
    / len(price_vs_book_row_audit)
)

price_vs_book_consistency_audit = (
    price_vs_book_consistency_audit
    .sort_values(["row_count"], ascending=False)
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Aggressor-versus-book audit summary
# ------------------------------------------------------------

aggressor_vs_book_audit = (
    price_vs_book_row_audit
    .groupby(
        [
            "trade_aggressor_side",
            "aggressor_book_class",
            "price_vs_book_location",
            "price_book_warning_class",
        ],
        dropna=False,
    )
    .agg(
        row_count=("trade_id", "size"),
        total_quantity=("trade_quantity", "sum"),
        total_notional=("trade_notional", "sum"),
        local_lag_ms_mean=("local_observation_lag_ms", "mean"),
        local_lag_ms_median=("local_observation_lag_ms", "median"),
        local_lag_ms_p95=("local_observation_lag_ms", lambda x: safe_quantile(x, 0.95)),
        local_lag_ms_max=("local_observation_lag_ms", "max"),
        sequence_lag_median=("sequence_lag", "median"),
        sequence_lag_p95=("sequence_lag", lambda x: safe_quantile(x, 0.95)),
        sequence_lag_max=("sequence_lag", "max"),
        stale_match_count=("stale_match_flag", "sum"),
        very_stale_match_count=("very_stale_match_flag", "sum"),
        wide_spread_state_match_count=("wide_spread_state_match_flag", "sum"),
    )
    .reset_index()
)

aggressor_side_totals = (
    price_vs_book_row_audit
    .groupby("trade_aggressor_side", dropna=False)
    .size()
    .rename("aggressor_side_total")
    .reset_index()
)

aggressor_vs_book_audit = aggressor_vs_book_audit.merge(
    aggressor_side_totals,
    how="left",
    on="trade_aggressor_side",
    validate="many_to_one",
)

aggressor_vs_book_audit["share_within_aggressor_side"] = (
    aggressor_vs_book_audit["row_count"]
    / aggressor_vs_book_audit["aggressor_side_total"]
)

aggressor_vs_book_audit["row_share"] = (
    aggressor_vs_book_audit["row_count"]
    / len(price_vs_book_row_audit)
)

aggressor_vs_book_audit = (
    aggressor_vs_book_audit
    .sort_values(["trade_aggressor_side", "row_count"], ascending=[True, False])
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Partition-level diagnostic summary
# ------------------------------------------------------------

price_book_partition_audit = (
    price_vs_book_row_audit
    .groupby(["book_partition", "trade_aggressor_side"], dropna=False)
    .agg(
        row_count=("trade_id", "size"),
        touch_or_through_count=("visible_book_touch_or_through_flag", "sum"),
        inside_spread_trade_count=("inside_spread_trade_flag", "sum"),
        aggressor_book_inconsistency_count=("aggressor_book_inconsistency_flag", "sum"),
        outside_visible_book_count=("outside_visible_book_flag", "sum"),
        wide_spread_state_match_count=("wide_spread_state_match_flag", "sum"),
        local_lag_ms_median=("local_observation_lag_ms", "median"),
        local_lag_ms_p95=("local_observation_lag_ms", lambda x: safe_quantile(x, 0.95)),
        sequence_lag_median=("sequence_lag", "median"),
        sequence_lag_p95=("sequence_lag", lambda x: safe_quantile(x, 0.95)),
    )
    .reset_index()
)

for count_column in [
    "touch_or_through_count",
    "inside_spread_trade_count",
    "aggressor_book_inconsistency_count",
    "outside_visible_book_count",
    "wide_spread_state_match_count",
]:
    share_column = count_column.replace("_count", "_share")
    price_book_partition_audit[share_column] = (
        price_book_partition_audit[count_column]
        / price_book_partition_audit["row_count"]
    )


# ------------------------------------------------------------
# Gates
# ------------------------------------------------------------

price_book_gates: list[GateResult] = []

price_book_gates.append(
    make_gate(
        gate="price_book_row_audit_row_count",
        passed=len(price_vs_book_row_audit) == len(local_strict_matched_alignment),
        severity="BLOCKING",
        detail=f"observed={len(price_vs_book_row_audit)} expected={len(local_strict_matched_alignment)}",
    )
)

price_book_gates.append(
    make_gate(
        gate="price_tick_size_expected",
        passed=math.isclose(PRICE_TICK_SIZE, 0.01, rel_tol=0.0, abs_tol=1e-12),
        severity="BLOCKING",
        detail=f"inferred_tick_size={PRICE_TICK_SIZE}",
    )
)

price_book_gates.append(
    make_gate(
        gate="price_book_classification_exhaustive",
        passed=not price_vs_book_row_audit["aggressor_book_class"].eq("UNCLASSIFIED").any(),
        severity="BLOCKING",
        detail=f"unclassified_count={int(price_vs_book_row_audit['aggressor_book_class'].eq('UNCLASSIFIED').sum())}",
    )
)

price_book_gates.append(
    make_gate(
        gate="price_book_no_invalid_price_or_book",
        passed=not price_vs_book_row_audit["price_vs_book_location"].isin(
            ["INVALID_PRICE_OR_BOOK", "INVALID_CROSSED_OR_LOCKED_BOOK"]
        ).any(),
        severity="BLOCKING",
        detail=(
            "invalid_rows="
            f"{int(price_vs_book_row_audit['price_vs_book_location'].isin(['INVALID_PRICE_OR_BOOK', 'INVALID_CROSSED_OR_LOCKED_BOOK']).sum())}"
        ),
    )
)

price_book_gates.append(
    make_gate(
        gate="price_book_inside_spread_trades",
        passed=int(price_vs_book_row_audit["inside_spread_trade_flag"].sum()) == 0,
        severity="WARNING",
        detail=f"inside_spread_trade_count={int(price_vs_book_row_audit['inside_spread_trade_flag'].sum())}",
    )
)

price_book_gates.append(
    make_gate(
        gate="price_book_aggressor_inconsistencies",
        passed=int(price_vs_book_row_audit["aggressor_book_inconsistency_flag"].sum()) == 0,
        severity="WARNING",
        detail=f"aggressor_book_inconsistency_count={int(price_vs_book_row_audit['aggressor_book_inconsistency_flag'].sum())}",
    )
)

price_book_gates.append(
    make_gate(
        gate="price_book_outside_visible_book_trades",
        passed=int(price_vs_book_row_audit["outside_visible_book_flag"].sum()) == 0,
        severity="WARNING",
        detail=f"outside_visible_book_count={int(price_vs_book_row_audit['outside_visible_book_flag'].sum())}",
    )
)

price_book_gates.append(
    make_gate(
        gate="price_book_wide_spread_state_matches",
        passed=int(price_vs_book_row_audit["wide_spread_state_match_flag"].sum()) == 0,
        severity="WARNING",
        detail=f"wide_spread_state_match_count={int(price_vs_book_row_audit['wide_spread_state_match_flag'].sum())}",
    )
)

price_book_gate_frame = gate_results_to_frame(price_book_gates)

fail_if_blocking_gate_failed(price_book_gate_frame)


# ------------------------------------------------------------
# Write audit outputs and verify read-back
# ------------------------------------------------------------

price_vs_book_consistency_metadata = write_csv_and_verify(
    price_vs_book_consistency_audit,
    PRICE_VS_BOOK_CONSISTENCY_AUDIT_PATH,
    expected_rows=len(price_vs_book_consistency_audit),
)

aggressor_vs_book_metadata = write_csv_and_verify(
    aggressor_vs_book_audit,
    AGGRESSOR_VS_BOOK_AUDIT_PATH,
    expected_rows=len(aggressor_vs_book_audit),
)

price_book_output_metadata = {
    "price_vs_book_consistency_audit": price_vs_book_consistency_metadata,
    "aggressor_vs_book_audit": aggressor_vs_book_metadata,
}


# ------------------------------------------------------------
# Compact display
# ------------------------------------------------------------

price_book_count_summary = pd.DataFrame(
    [
        {
            "matched_rows": int(len(price_vs_book_row_audit)),
            "price_tick_size": PRICE_TICK_SIZE,
            "touch_or_through_count": int(price_vs_book_row_audit["visible_book_touch_or_through_flag"].sum()),
            "inside_spread_trade_count": int(price_vs_book_row_audit["inside_spread_trade_flag"].sum()),
            "aggressor_book_inconsistency_count": int(price_vs_book_row_audit["aggressor_book_inconsistency_flag"].sum()),
            "outside_visible_book_count": int(price_vs_book_row_audit["outside_visible_book_flag"].sum()),
            "wide_spread_state_match_count": int(price_vs_book_row_audit["wide_spread_state_match_flag"].sum()),
        }
    ]
)

display(price_book_count_summary)
display(price_vs_book_consistency_audit.head(30))
display(aggressor_vs_book_audit.head(30))
display(price_book_partition_audit)
display(price_book_gate_frame)

price_book_output_metadata

,matched_rows,price_tick_size,touch_or_through_count,inside_spread_trade_count,aggressor_book_inconsistency_count,outside_visible_book_count,wide_spread_state_match_count
0,67683,0.01,66798,67,818,30926,95


,price_vs_book_location,aggressor_book_class,price_book_warning_class,staleness_bucket,row_count,buy_count,sell_count,total_quantity,total_notional,local_lag_ms_mean,local_lag_ms_median,local_lag_ms_max,sequence_lag_median,sequence_lag_max,spread_ticks_median,spread_ticks_max,stale_match_count,very_stale_match_count,row_share
0,AT_BID,SELL_AT_BID,NO_PRICE_BOOK_WARNING,"(50, 100] ms",10200,0,10200,71.37446,4.561544e+06,73.602641,72.7554,99.997,7.0,213,1.0,1,0,0,0.150703
1,BELOW_BID,SELL_THROUGH_BID,OUTSIDE_VISIBLE_BOOK,"(50, 100] ms",8224,0,8224,30.59744,1.954892e+06,72.631094,72.5968,99.9896,94.0,461,1.0,1,0,0,0.121508
2,ABOVE_ASK,BUY_THROUGH_ASK,OUTSIDE_VISIBLE_BOOK,"(50, 100] ms",8171,8171,0,19.18098,1.225332e+06,74.875329,76.7304,99.9311,82.0,348,1.0,1,0,0,0.120725
3,AT_BID,SELL_AT_BID,NO_PRICE_BOOK_WARNING,"(10, 50] ms",7859,0,7859,53.57819,3.423160e+06,29.454924,29.8451,49.9913,7.0,194,1.0,1,0,0,0.116115
4,AT_ASK,BUY_AT_ASK,NO_PRICE_BOOK_WARNING,"(50, 100] ms",7626,7626,0,46.17707,2.951107e+06,74.36417,75.2475,99.999,6.0,126,1.0,1,0,0,0.112672
5,AT_ASK,BUY_AT_ASK,NO_PRICE_BOOK_WARNING,"(10, 50] ms",6313,6313,0,39.71405,2.537824e+06,30.023501,30.6804,49.9786,6.0,137,1.0,1,0,0,0.093273
6,BELOW_BID,SELL_THROUGH_BID,OUTSIDE_VISIBLE_BOOK,"(10, 50] ms",6292,0,6292,24.64495,1.574666e+06,31.791077,32.1769,49.9445,81.0,277,1.0,1,0,0,0.092963
7,ABOVE_ASK,BUY_THROUGH_ASK,OUTSIDE_VISIBLE_BOOK,"(10, 50] ms",5000,5000,0,12.50108,7.990408e+05,31.749062,31.7875,49.9665,69.0,258,1.0,1,0,0,0.073874
8,AT_BID,SELL_AT_BID,NO_PRICE_BOOK_WARNING,"(1, 10] ms",2092,0,2092,10.45434,6.680449e+05,5.654462,5.6524,9.9997,7.0,90,1.0,1,0,0,0.030909
9,AT_ASK,BUY_AT_ASK,NO_PRICE_BOOK_WARNING,"(1, 10] ms",1620,1620,0,8.01330,5.121097e+05,5.753321,5.6044,9.998,5.0,106,1.0,1,0,0,0.023935


,trade_aggressor_side,aggressor_book_class,price_vs_book_location,price_book_warning_class,row_count,total_quantity,total_notional,local_lag_ms_mean,local_lag_ms_median,local_lag_ms_p95,local_lag_ms_max,sequence_lag_median,sequence_lag_p95,sequence_lag_max,stale_match_count,very_stale_match_count,wide_spread_state_match_count,aggressor_side_total,share_within_aggressor_side,row_share
0,BUY,BUY_AT_ASK,AT_ASK,NO_PRICE_BOOK_WARNING,15944,94.92491,6.066238e+06,49.721408,49.2984,96.2752,111.3185,6.0,48.00,137,0,0,0,30596,0.521114,0.235569
1,BUY,BUY_THROUGH_ASK,ABOVE_ASK,OUTSIDE_VISIBLE_BOOK,14193,33.78274,2.158608e+06,56.787536,57.6764,96.8264,113.3191,74.0,193.00,348,0,0,0,30596,0.463884,0.209698
2,BUY,BUY_BELOW_BID_INCONSISTENT,BELOW_BID,AGGRESSOR_BOOK_INCONSISTENCY,391,2.93975,1.877818e+05,9.947431,4.3241,62.3834,99.2453,34.0,108.00,274,0,0,0,30596,0.012779,0.005777
3,BUY,BUY_AT_BID_INCONSISTENT,AT_BID,AGGRESSOR_BOOK_INCONSISTENCY,38,0.11207,7.160058e+03,6.042726,2.0322,22.835295,77.2588,19.5,103.15,105,0,0,0,30596,0.001242,0.000561
4,BUY,BUY_INSIDE_SPREAD,INSIDE_SPREAD,INSIDE_SPREAD_TRADE,30,0.02814,1.797220e+03,95.493757,96.6059,97.6086,97.6086,80.5,93.55,95,0,0,30,30596,0.000981,0.000443
5,SELL,SELL_AT_BID,AT_BID,NO_PRICE_BOOK_WARNING,20684,137.13942,8.763450e+06,49.803223,51.284,95.68017,108.5651,7.0,54.00,213,0,0,0,37087,0.557716,0.305601
6,SELL,SELL_THROUGH_BID,BELOW_BID,OUTSIDE_VISIBLE_BOOK,15977,57.55278,3.677179e+06,52.531348,54.0204,94.26914,108.5651,81.0,245.00,461,0,0,0,37087,0.430798,0.236056
7,SELL,SELL_ABOVE_ASK_INCONSISTENT,ABOVE_ASK,AGGRESSOR_BOOK_INCONSISTENCY,365,2.60140,1.662453e+05,16.21976,6.0796,85.20626,99.8715,42.0,128.80,349,0,0,28,37087,0.009842,0.005393
8,SELL,SELL_INSIDE_SPREAD,INSIDE_SPREAD,INSIDE_SPREAD_TRADE,37,0.09793,6.254544e+03,7.846822,2.5245,14.208,17.35,46.0,63.20,65,0,0,37,37087,0.000998,0.000547
9,SELL,SELL_AT_ASK_INCONSISTENT,AT_ASK,AGGRESSOR_BOOK_INCONSISTENCY,24,0.01933,1.235266e+03,1.299404,1.0025,3.0437,3.0437,12.5,32.85,34,0,0,0,37087,0.000647,0.000355


,book_partition,trade_aggressor_side,row_count,touch_or_through_count,inside_spread_trade_count,aggressor_book_inconsistency_count,outside_visible_book_count,wide_spread_state_match_count,local_lag_ms_median,local_lag_ms_p95,sequence_lag_median,sequence_lag_p95,touch_or_through_share,inside_spread_trade_share,aggressor_book_inconsistency_share,outside_visible_book_share,wide_spread_state_match_share
0,CALIBRATION,BUY,7267,7212,0,55,3833,0,55.4163,96.37943,33.0,153.00,0.992432,0.0,0.007568,0.527453,0.0
1,CALIBRATION,SELL,6266,6231,0,35,2955,0,49.8469,93.1921,33.0,165.00,0.994414,0.0,0.005586,0.471593,0.0
2,DEVELOPMENT,BUY,15194,14878,30,286,7028,30,54.042,96.227465,29.0,156.00,0.979202,0.001974,0.018823,0.462551,0.001974
3,DEVELOPMENT,SELL,18626,18410,37,179,8680,65,52.6492,95.3267,37.0,212.00,0.988403,0.001986,0.00961,0.466015,0.00349
4,ENGINEERING_HOLDOUT,BUY,2361,2358,0,3,852,0,53.6958,96.6491,14.0,110.00,0.998729,0.0,0.001271,0.360864,0.0
5,ENGINEERING_HOLDOUT,SELL,7772,7606,0,166,2842,0,57.4659,96.0061,16.0,160.00,0.978641,0.0,0.021359,0.365672,0.0
6,VALIDATION,BUY,5774,5689,0,85,2871,0,49.3745,97.8372,39.0,178.35,0.985279,0.0,0.014721,0.497229,0.0
7,VALIDATION,SELL,4423,4414,0,9,1865,0,40.0029,98.8936,26.0,167.00,0.997965,0.0,0.002035,0.42166,0.0


,gate,status,severity,detail
0,price_book_row_audit_row_count,PASS,BLOCKING,observed=67683 expected=67683
1,price_tick_size_expected,PASS,BLOCKING,inferred_tick_size=0.01
2,price_book_classification_exhaustive,PASS,BLOCKING,unclassified_count=0
3,price_book_no_invalid_price_or_book,PASS,BLOCKING,invalid_rows=0
4,price_book_inside_spread_trades,FAIL,WARNING,inside_spread_trade_count=67
5,price_book_aggressor_inconsistencies,FAIL,WARNING,aggressor_book_inconsistency_count=818
6,price_book_outside_visible_book_trades,FAIL,WARNING,outside_visible_book_count=30926
7,price_book_wide_spread_state_matches,FAIL,WARNING,wide_spread_state_match_count=95


{'price_vs_book_consistency_audit': {'path': 'D:\\Clown Project\\V0.1\\artifacts\\audit_tables\\03_CAUSAL_TRADE_BOOK_ALIGNMENT\\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__03_CAUSAL_TRADE_BOOK_ALIGNMENT__price_vs_book_consistency_audit.csv',
  'row_count': 38,
  'size_bytes': 6733,
  'sha256': '7338515cecedffa1c902091eab07b047e7f3172073ae95b7d3cc46084e5edae7'},
 'aggressor_vs_book_audit': {'path': 'D:\\Clown Project\\V0.1\\artifacts\\audit_tables\\03_CAUSAL_TRADE_BOOK_ALIGNMENT\\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__03_CAUSAL_TRADE_BOOK_ALIGNMENT__aggressor_vs_book_audit.csv',
  'row_count': 10,
  'size_bytes': 2448,
  'sha256': '868164419f218c7dd4c2387f3031595d75334303f4bb76e50db79e2a5a2e91d5'}}

In [12]:
# ============================================================
# Cell 10 — Partition alignment summary and boundary audit
# ============================================================

# This cell builds:
#   1. partition_alignment_summary
#   2. boundary_alignment_audit
#
# Trade partition is assigned from trade collector sequence using the observed
# chronological partition boundaries in the reconstructed book-state table.
#
# The matched book partition is preserved separately.
#
# Boundary-crossing matches are allowed:
# A trade near the start of a partition may causally match the latest book state
# from the prior partition. That is not leakage if LOCAL_STRICT holds.


# ------------------------------------------------------------
# Derive partition sequence intervals from book states
# ------------------------------------------------------------

require_columns(
    book_states,
    [
        "book_collector_sequence",
        "book_partition",
        "book_local_receipt_time_ns",
    ],
    "book_states",
)

require_columns(
    local_strict_all_alignment,
    [
        "trade_collector_sequence",
        "trade_local_receipt_time_ns",
        "book_collector_sequence",
        "book_local_receipt_time_ns",
        "book_partition",
        "is_local_strict_match",
        "sequence_lag",
        "local_observation_lag_ms",
        "staleness_bucket",
        "stale_match_flag",
        "very_stale_match_flag",
    ],
    "local_strict_all_alignment",
)

book_partition_ranges = (
    book_states
    .groupby("book_partition", dropna=False)
    .agg(
        book_partition_sequence_min=("book_collector_sequence", "min"),
        book_partition_sequence_max=("book_collector_sequence", "max"),
        book_partition_local_time_min=("book_local_receipt_time_ns", "min"),
        book_partition_local_time_max=("book_local_receipt_time_ns", "max"),
        book_state_count=("book_collector_sequence", "size"),
    )
    .reset_index()
    .rename(columns={"book_partition": "partition"})
)

book_partition_ranges = (
    book_partition_ranges
    .sort_values("book_partition_sequence_min")
    .reset_index(drop=True)
)

book_partition_ranges["next_partition"] = book_partition_ranges["partition"].shift(-1)
book_partition_ranges["next_partition_sequence_min"] = book_partition_ranges["book_partition_sequence_min"].shift(-1)

book_partition_ranges["trade_partition_sequence_start"] = (
    book_partition_ranges["book_partition_sequence_min"]
)

book_partition_ranges["trade_partition_sequence_end_exclusive"] = (
    book_partition_ranges["next_partition_sequence_min"]
)

book_partition_ranges.loc[
    book_partition_ranges["trade_partition_sequence_end_exclusive"].isna(),
    "trade_partition_sequence_end_exclusive",
] = EXPECTED_LAST_RECONSTRUCTION_DEPTH_SEQUENCE + 1

book_partition_ranges["trade_partition_sequence_start"] = (
    book_partition_ranges["trade_partition_sequence_start"].astype("int64")
)

book_partition_ranges["trade_partition_sequence_end_exclusive"] = (
    book_partition_ranges["trade_partition_sequence_end_exclusive"].astype("int64")
)


# ------------------------------------------------------------
# Assign trade partition from collector sequence
# ------------------------------------------------------------

partition_intervals = [
    {
        "partition": str(row["partition"]),
        "start": int(row["trade_partition_sequence_start"]),
        "end_exclusive": int(row["trade_partition_sequence_end_exclusive"]),
    }
    for _, row in book_partition_ranges.iterrows()
]

partitioned_alignment = local_strict_all_alignment.copy()
partitioned_alignment["trade_partition"] = pd.NA

for interval in partition_intervals:
    mask = (
        (partitioned_alignment["trade_collector_sequence"].astype("int64") >= interval["start"])
        & (partitioned_alignment["trade_collector_sequence"].astype("int64") < interval["end_exclusive"])
    )

    partitioned_alignment.loc[mask, "trade_partition"] = interval["partition"]

partitioned_alignment["trade_partition"] = partitioned_alignment["trade_partition"].astype("string")
partitioned_alignment["matched_book_partition"] = partitioned_alignment["book_partition"].astype("string")

unassigned_trade_partition_count = int(partitioned_alignment["trade_partition"].isna().sum())

# Keep the main alignment frame enriched for later cells.
local_strict_all_alignment["trade_partition"] = partitioned_alignment["trade_partition"]
local_strict_all_alignment["matched_book_partition"] = partitioned_alignment["matched_book_partition"]

local_strict_matched_alignment = (
    local_strict_all_alignment[
        local_strict_all_alignment["is_local_strict_match"]
    ]
    .copy()
    .reset_index(drop=True)
)

# Keep price audit enriched too if it exists from Cell 09.
if "price_vs_book_row_audit" in globals():
    price_vs_book_row_audit = price_vs_book_row_audit.merge(
        local_strict_all_alignment[
            [
                "trade_id",
                "trade_partition",
                "matched_book_partition",
            ]
        ],
        how="left",
        on="trade_id",
        validate="one_to_one",
    )


# ------------------------------------------------------------
# Boundary-crossing diagnostics
# ------------------------------------------------------------

partitioned_alignment["partition_crossing_match_flag"] = (
    partitioned_alignment["is_local_strict_match"]
    & partitioned_alignment["trade_partition"].notna()
    & partitioned_alignment["matched_book_partition"].notna()
    & (partitioned_alignment["trade_partition"] != partitioned_alignment["matched_book_partition"])
)

local_strict_all_alignment["partition_crossing_match_flag"] = (
    partitioned_alignment["partition_crossing_match_flag"]
)

boundary_crossing_rows = (
    partitioned_alignment[
        partitioned_alignment["partition_crossing_match_flag"]
    ]
    .copy()
    .reset_index(drop=True)
)

boundary_detail_columns = [
    "trade_id",
    "trade_collector_sequence",
    "book_collector_sequence",
    "sequence_lag",
    "trade_local_receipt_time_ns",
    "book_local_receipt_time_ns",
    "local_observation_lag_ms",
    "trade_exchange_trade_time_ms",
    "book_exchange_event_time_ms",
    "exchange_time_lag_ms",
    "trade_aggressor_side",
    "trade_price",
    "trade_quantity",
    "trade_partition",
    "matched_book_partition",
    "staleness_bucket",
    "stale_match_flag",
    "very_stale_match_flag",
    "local_strict_sequence_condition",
    "local_strict_time_condition",
]

boundary_crossing_detail = boundary_crossing_rows[
    [column for column in boundary_detail_columns if column in boundary_crossing_rows.columns]
].copy()


# ------------------------------------------------------------
# Partition alignment summary
# ------------------------------------------------------------

partition_summary_base = (
    partitioned_alignment
    .groupby("trade_partition", dropna=False)
    .agg(
        raw_trade_count=("trade_id", "size"),
        matched_count=("is_local_strict_match", "sum"),
        unmatched_count=("is_local_strict_match", lambda x: int((~x).sum())),
        buy_count=("trade_aggressor_side", lambda x: int((x == "BUY").sum())),
        sell_count=("trade_aggressor_side", lambda x: int((x == "SELL").sum())),
        total_quantity=("trade_quantity", "sum"),
        total_notional=("trade_notional", "sum"),
        trade_sequence_min=("trade_collector_sequence", "min"),
        trade_sequence_max=("trade_collector_sequence", "max"),
        trade_local_time_min=("trade_local_receipt_time_ns", "min"),
        trade_local_time_max=("trade_local_receipt_time_ns", "max"),
        matched_book_sequence_min=("book_collector_sequence", "min"),
        matched_book_sequence_max=("book_collector_sequence", "max"),
        sequence_lag_median=("sequence_lag", "median"),
        sequence_lag_p95=("sequence_lag", lambda x: safe_quantile(x, 0.95)),
        sequence_lag_max=("sequence_lag", "max"),
        local_lag_ms_median=("local_observation_lag_ms", "median"),
        local_lag_ms_p95=("local_observation_lag_ms", lambda x: safe_quantile(x, 0.95)),
        local_lag_ms_p99=("local_observation_lag_ms", lambda x: safe_quantile(x, 0.99)),
        local_lag_ms_max=("local_observation_lag_ms", "max"),
        stale_match_count=("stale_match_flag", "sum"),
        very_stale_match_count=("very_stale_match_flag", "sum"),
        partition_crossing_match_count=("partition_crossing_match_flag", "sum"),
    )
    .reset_index()
)

partition_summary_base["matched_share"] = (
    partition_summary_base["matched_count"] / partition_summary_base["raw_trade_count"]
)

partition_summary_base["unmatched_share"] = (
    partition_summary_base["unmatched_count"] / partition_summary_base["raw_trade_count"]
)

partition_summary_base["stale_match_share"] = (
    partition_summary_base["stale_match_count"] / partition_summary_base["raw_trade_count"]
)

partition_summary_base["very_stale_match_share"] = (
    partition_summary_base["very_stale_match_count"] / partition_summary_base["raw_trade_count"]
)

partition_summary_base["partition_crossing_match_share"] = (
    partition_summary_base["partition_crossing_match_count"] / partition_summary_base["raw_trade_count"]
)


# ------------------------------------------------------------
# Add price/book warning counts by trade partition
# ------------------------------------------------------------

if "price_vs_book_row_audit" in globals():
    price_partition_warning_summary = (
        price_vs_book_row_audit
        .groupby("trade_partition", dropna=False)
        .agg(
            touch_or_through_count=("visible_book_touch_or_through_flag", "sum"),
            inside_spread_trade_count=("inside_spread_trade_flag", "sum"),
            aggressor_book_inconsistency_count=("aggressor_book_inconsistency_flag", "sum"),
            outside_visible_book_count=("outside_visible_book_flag", "sum"),
            wide_spread_state_match_count=("wide_spread_state_match_flag", "sum"),
        )
        .reset_index()
    )

    partition_alignment_summary = partition_summary_base.merge(
        price_partition_warning_summary,
        how="left",
        on="trade_partition",
        validate="one_to_one",
    )

    for count_column in [
        "touch_or_through_count",
        "inside_spread_trade_count",
        "aggressor_book_inconsistency_count",
        "outside_visible_book_count",
        "wide_spread_state_match_count",
    ]:
        partition_alignment_summary[count_column] = (
            partition_alignment_summary[count_column].fillna(0).astype(int)
        )
        partition_alignment_summary[count_column.replace("_count", "_share")] = (
            partition_alignment_summary[count_column]
            / partition_alignment_summary["raw_trade_count"]
        )
else:
    partition_alignment_summary = partition_summary_base.copy()


# ------------------------------------------------------------
# Boundary alignment audit summary
# ------------------------------------------------------------

boundary_alignment_audit = (
    partitioned_alignment
    .groupby(
        [
            "trade_partition",
            "matched_book_partition",
            "partition_crossing_match_flag",
        ],
        dropna=False,
    )
    .agg(
        row_count=("trade_id", "size"),
        matched_count=("is_local_strict_match", "sum"),
        buy_count=("trade_aggressor_side", lambda x: int((x == "BUY").sum())),
        sell_count=("trade_aggressor_side", lambda x: int((x == "SELL").sum())),
        trade_sequence_min=("trade_collector_sequence", "min"),
        trade_sequence_max=("trade_collector_sequence", "max"),
        book_sequence_min=("book_collector_sequence", "min"),
        book_sequence_max=("book_collector_sequence", "max"),
        sequence_lag_min=("sequence_lag", "min"),
        sequence_lag_median=("sequence_lag", "median"),
        sequence_lag_max=("sequence_lag", "max"),
        local_lag_ms_min=("local_observation_lag_ms", "min"),
        local_lag_ms_median=("local_observation_lag_ms", "median"),
        local_lag_ms_max=("local_observation_lag_ms", "max"),
        stale_match_count=("stale_match_flag", "sum"),
        very_stale_match_count=("very_stale_match_flag", "sum"),
    )
    .reset_index()
    .sort_values(
        [
            "partition_crossing_match_flag",
            "trade_sequence_min",
        ],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

boundary_alignment_audit["row_share"] = (
    boundary_alignment_audit["row_count"] / len(partitioned_alignment)
)

boundary_alignment_audit["stale_match_share"] = (
    boundary_alignment_audit["stale_match_count"] / boundary_alignment_audit["row_count"]
)

boundary_alignment_audit["very_stale_match_share"] = (
    boundary_alignment_audit["very_stale_match_count"] / boundary_alignment_audit["row_count"]
)


# ------------------------------------------------------------
# Boundary gate checks
# ------------------------------------------------------------

boundary_future_sequence_violations = int(
    (
        boundary_crossing_rows["book_collector_sequence"].astype("Int64")
        >= boundary_crossing_rows["trade_collector_sequence"].astype("Int64")
    ).sum()
) if not boundary_crossing_rows.empty else 0

boundary_future_local_time_violations = int(
    (
        boundary_crossing_rows["book_local_receipt_time_ns"].astype("Int64")
        > boundary_crossing_rows["trade_local_receipt_time_ns"].astype("Int64")
    ).sum()
) if not boundary_crossing_rows.empty else 0

partition_alignment_gates: list[GateResult] = []

partition_alignment_gates.append(
    make_gate(
        gate="trade_partition_assignment_complete",
        passed=unassigned_trade_partition_count == 0,
        severity="BLOCKING",
        detail=f"unassigned_trade_partition_count={unassigned_trade_partition_count}",
    )
)

partition_alignment_gates.append(
    make_gate(
        gate="partition_alignment_summary_row_sum",
        passed=int(partition_alignment_summary["raw_trade_count"].sum()) == EXPECTED_RAW_TRADE_RECORDS,
        severity="BLOCKING",
        detail=(
            f"observed_sum={int(partition_alignment_summary['raw_trade_count'].sum())} "
            f"expected={EXPECTED_RAW_TRADE_RECORDS}"
        ),
    )
)

partition_alignment_gates.append(
    make_gate(
        gate="boundary_crossing_local_strict_sequence_valid",
        passed=boundary_future_sequence_violations == 0,
        severity="BLOCKING",
        detail=f"boundary_future_sequence_violations={boundary_future_sequence_violations}",
    )
)

partition_alignment_gates.append(
    make_gate(
        gate="boundary_crossing_local_strict_time_valid",
        passed=boundary_future_local_time_violations == 0,
        severity="BLOCKING",
        detail=f"boundary_future_local_time_violations={boundary_future_local_time_violations}",
    )
)

partition_alignment_gates.append(
    make_gate(
        gate="partition_crossing_matches_present",
        passed=int(partitioned_alignment["partition_crossing_match_flag"].sum()) == 0,
        severity="WARNING",
        detail=f"partition_crossing_match_count={int(partitioned_alignment['partition_crossing_match_flag'].sum())}",
    )
)

partition_alignment_gate_frame = gate_results_to_frame(partition_alignment_gates)

fail_if_blocking_gate_failed(partition_alignment_gate_frame)


# ------------------------------------------------------------
# Write outputs and verify read-back
# ------------------------------------------------------------

partition_alignment_summary_metadata = write_csv_and_verify(
    partition_alignment_summary,
    PARTITION_ALIGNMENT_SUMMARY_PATH,
    expected_rows=len(partition_alignment_summary),
)

boundary_alignment_audit_metadata = write_csv_and_verify(
    boundary_alignment_audit,
    BOUNDARY_ALIGNMENT_AUDIT_PATH,
    expected_rows=len(boundary_alignment_audit),
)

partition_boundary_output_metadata = {
    "partition_alignment_summary": partition_alignment_summary_metadata,
    "boundary_alignment_audit": boundary_alignment_audit_metadata,
}


# ------------------------------------------------------------
# Compact display
# ------------------------------------------------------------

display(book_partition_ranges)
display(partition_alignment_summary)
display(boundary_alignment_audit)
display(boundary_crossing_detail.head(50))
display(partition_alignment_gate_frame)

partition_boundary_output_metadata

,partition,book_partition_sequence_min,book_partition_sequence_max,book_partition_local_time_min,book_partition_local_time_max,book_state_count,next_partition,next_partition_sequence_min,trade_partition_sequence_start,trade_partition_sequence_end_exclusive
0,DEVELOPMENT,10,51839,1783665468432309400,1783667269290574400,18010,CALIBRATION,51840,10,51840
1,CALIBRATION,51840,72575,1783667269391572100,1783667989674233200,7204,VALIDATION,72577,51840,72577
2,VALIDATION,72577,88126,1783667989776171900,1783668524962184400,5353,ENGINEERING_HOLDOUT,88127,72577,88127
3,ENGINEERING_HOLDOUT,88127,103677,1783668525057534700,1783669066749750800,5418,None,<NA>,88127,103678


,trade_partition,raw_trade_count,matched_count,unmatched_count,buy_count,sell_count,total_quantity,total_notional,trade_sequence_min,trade_sequence_max,trade_local_time_min,trade_local_time_max,matched_book_sequence_min,matched_book_sequence_max,sequence_lag_median,sequence_lag_p95,sequence_lag_max,local_lag_ms_median,local_lag_ms_p95,local_lag_ms_p99,local_lag_ms_max,stale_match_count,very_stale_match_count,partition_crossing_match_count,matched_share,unmatched_share,stale_match_share,very_stale_match_share,partition_crossing_match_share,touch_or_through_count,inside_spread_trade_count,aggressor_book_inconsistency_count,outside_visible_book_count,wide_spread_state_match_count,touch_or_through_share,inside_spread_trade_share,aggressor_book_inconsistency_share,outside_visible_book_share,wide_spread_state_match_share
0,CALIBRATION,13533,13533,0,7267,6266,47.88290,3.058916e+06,51852,72576,1783667270546982000,1783667989690751200,51851,72575,33.0,158.0,271,53.9729,95.0314,100.2771,108.0506,0,0,0,1.0,0.0,0.0,0.0,0.0,13443,0,90,6788,0,0.99335,0.0,0.00665,0.501589,0.0
1,DEVELOPMENT,33820,33820,0,15194,18626,165.51400,1.057147e+07,14,51838,1783665468766951600,1783667269232205000,13,51837,33.0,186.0,461,53.4924,95.67201,102.879,113.3191,0,0,0,1.0,0.0,0.0,0.0,0.0,33288,67,465,15708,95,0.98427,0.001981,0.013749,0.464459,0.002809
2,ENGINEERING_HOLDOUT,10133,10133,0,2361,7772,63.83986,4.081884e+06,88129,103676,1783668525205920700,1783669066707304900,88128,103659,16.0,138.0,312,56.8191,96.35148,99.4821,101.7121,0,0,0,1.0,0.0,0.0,0.0,0.0,9964,0,169,3694,0,0.983322,0.0,0.016678,0.364551,0.0
3,VALIDATION,10197,10197,0,5774,4423,51.96171,3.323676e+06,72578,88125,1783667989786206500,1783668524924691300,72577,88124,34.0,174.2,317,47.7944,98.3169,102.7102,107.9806,0,0,0,1.0,0.0,0.0,0.0,0.0,10103,0,94,4736,0,0.990782,0.0,0.009218,0.46445,0.0


,trade_partition,matched_book_partition,partition_crossing_match_flag,row_count,matched_count,buy_count,sell_count,trade_sequence_min,trade_sequence_max,book_sequence_min,book_sequence_max,sequence_lag_min,sequence_lag_median,sequence_lag_max,local_lag_ms_min,local_lag_ms_median,local_lag_ms_max,stale_match_count,very_stale_match_count,row_share,stale_match_share,very_stale_match_share
0,DEVELOPMENT,DEVELOPMENT,False,33820,33820,15194,18626,14,51838,13,51837,1,33.0,461,0.0,53.4924,113.3191,0,0,0.499682,0.0,0.0
1,CALIBRATION,CALIBRATION,False,13533,13533,7267,6266,51852,72576,51851,72575,1,33.0,271,0.0,53.9729,108.0506,0,0,0.199947,0.0,0.0
2,VALIDATION,VALIDATION,False,10197,10197,5774,4423,72578,88125,72577,88124,1,34.0,317,0.0,47.7944,107.9806,0,0,0.150658,0.0,0.0
3,ENGINEERING_HOLDOUT,ENGINEERING_HOLDOUT,False,10133,10133,2361,7772,88129,103676,88128,103659,1,16.0,312,0.0,56.8191,101.7121,0,0,0.149713,0.0,0.0


,trade_id,trade_collector_sequence,book_collector_sequence,sequence_lag,trade_local_receipt_time_ns,book_local_receipt_time_ns,local_observation_lag_ms,trade_exchange_trade_time_ms,book_exchange_event_time_ms,exchange_time_lag_ms,trade_aggressor_side,trade_price,trade_quantity,trade_partition,matched_book_partition,staleness_bucket,stale_match_flag,very_stale_match_flag,local_strict_sequence_condition,local_strict_time_condition


,gate,status,severity,detail
0,trade_partition_assignment_complete,PASS,BLOCKING,unassigned_trade_partition_count=0
1,partition_alignment_summary_row_sum,PASS,BLOCKING,observed_sum=67683 expected=67683
2,boundary_crossing_local_strict_sequence_valid,PASS,BLOCKING,boundary_future_sequence_violations=0
3,boundary_crossing_local_strict_time_valid,PASS,BLOCKING,boundary_future_local_time_violations=0
4,partition_crossing_matches_present,PASS,WARNING,partition_crossing_match_count=0


{'partition_alignment_summary': {'path': 'D:\\Clown Project\\V0.1\\artifacts\\audit_tables\\03_CAUSAL_TRADE_BOOK_ALIGNMENT\\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__03_CAUSAL_TRADE_BOOK_ALIGNMENT__partition_alignment_summary.csv',
  'row_count': 4,
  'size_bytes': 2057,
  'sha256': 'fd46ee83adc83df1880a2e291d4abcd4f84cfe904eb28c7196205e6c7dc881af'},
 'boundary_alignment_audit': {'path': 'D:\\Clown Project\\V0.1\\artifacts\\audit_tables\\03_CAUSAL_TRADE_BOOK_ALIGNMENT\\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__03_CAUSAL_TRADE_BOOK_ALIGNMENT__boundary_alignment_audit.csv',
  'row_count': 4,
  'size_bytes': 964,
  'sha256': '1c398ef8006cbf6d5e8ad7471b8048271e3e12e9006e28eb0c0163195fcedecc'}}

In [13]:
# ============================================================
# Cell 11 — EXCHANGE_STRICT alignment sensitivity
# ============================================================

# This cell builds a sensitivity-only alignment using exchange timestamps:
#
#   book_exchange_event_time_ms < trade_exchange_trade_time_ms
#
# Among eligible book states, it selects the latest book exchange event time.
#
# This is NOT the primary authority.
# This table is used only to quantify how much the answer changes if one ignores
# collector-local causality and aligns only by exchange timestamps.


# ------------------------------------------------------------
# Prepare compact trade and book inputs
# ------------------------------------------------------------

exchange_trade_frame = local_strict_all_alignment[
    [
        "trade_id",
        "trade_collector_sequence",
        "trade_local_receipt_time_ns",
        "trade_local_receipt_time",
        "trade_exchange_trade_time_ms",
        "trade_exchange_trade_time",
        "trade_aggressor_side",
        "trade_price",
        "trade_quantity",
        "trade_notional",
        "book_collector_sequence",
        "book_local_receipt_time_ns",
        "book_exchange_event_time_ms",
        "book_best_bid",
        "book_best_ask",
        "book_spread",
        "book_midpoint",
        "book_microprice",
        "book_partition",
        "trade_partition",
        "matched_book_partition",
    ]
].copy()

exchange_trade_frame = exchange_trade_frame.rename(
    columns={
        "book_collector_sequence": "local_strict_book_collector_sequence",
        "book_local_receipt_time_ns": "local_strict_book_local_receipt_time_ns",
        "book_exchange_event_time_ms": "local_strict_book_exchange_event_time_ms",
        "book_best_bid": "local_strict_book_best_bid",
        "book_best_ask": "local_strict_book_best_ask",
        "book_spread": "local_strict_book_spread",
        "book_midpoint": "local_strict_book_midpoint",
        "book_microprice": "local_strict_book_microprice",
        "book_partition": "local_strict_book_partition",
    }
)

exchange_book_frame = book_states[
    [
        "book_state_id",
        "book_collector_sequence",
        "book_local_receipt_time_ns",
        "book_local_receipt_time",
        "book_exchange_event_time_ms",
        "book_exchange_event_time",
        "book_best_bid",
        "book_best_ask",
        "book_spread",
        "book_midpoint",
        "book_microprice",
        "book_partition",
    ]
].copy()

exchange_book_frame = exchange_book_frame.sort_values(
    [
        "book_exchange_event_time_ms",
        "book_collector_sequence",
    ],
    kind="mergesort",
).reset_index(drop=True)

exchange_book_time_array = (
    exchange_book_frame["book_exchange_event_time_ms"]
    .astype("int64")
    .to_numpy()
)

trade_exchange_time_array = (
    exchange_trade_frame["trade_exchange_trade_time_ms"]
    .astype("int64")
    .to_numpy()
)

require(
    np.all(np.diff(exchange_book_time_array) >= 0),
    "exchange_book_time_array must be non-decreasing after sorting",
)


# ------------------------------------------------------------
# EXCHANGE_STRICT vectorized selection
# ------------------------------------------------------------

# side="left" enforces strict inequality:
# candidate book exchange time must be < trade exchange trade time.
exchange_upper_positions = np.searchsorted(
    exchange_book_time_array,
    trade_exchange_time_array,
    side="left",
)

exchange_candidate_positions = exchange_upper_positions - 1
exchange_candidate_has_book = exchange_candidate_positions >= 0

exchange_matched_book_rows = (
    exchange_book_frame
    .reindex(exchange_candidate_positions)
    .reset_index(drop=True)
    .add_prefix("exchange_strict_")
)

exchange_strict_alignment_sensitivity = pd.concat(
    [
        exchange_trade_frame.reset_index(drop=True),
        exchange_matched_book_rows,
    ],
    axis=1,
)

exchange_strict_alignment_sensitivity["sensitivity_policy"] = SENSITIVITY_ALIGNMENT_POLICY
exchange_strict_alignment_sensitivity["exchange_candidate_position"] = exchange_candidate_positions
exchange_strict_alignment_sensitivity["exchange_upper_position"] = exchange_upper_positions
exchange_strict_alignment_sensitivity["exchange_candidate_has_book"] = exchange_candidate_has_book


# ------------------------------------------------------------
# EXCHANGE_STRICT validity and comparison flags
# ------------------------------------------------------------

exchange_strict_alignment_sensitivity["is_exchange_strict_match"] = (
    exchange_strict_alignment_sensitivity["exchange_candidate_has_book"]
    & exchange_strict_alignment_sensitivity["exchange_strict_book_exchange_event_time_ms"].notna()
    & (
        exchange_strict_alignment_sensitivity["exchange_strict_book_exchange_event_time_ms"].astype("Int64")
        < exchange_strict_alignment_sensitivity["trade_exchange_trade_time_ms"].astype("Int64")
    )
)

exchange_strict_alignment_sensitivity["exchange_strict_unmatched_reason"] = "MATCHED_EXCHANGE_STRICT"

exchange_strict_alignment_sensitivity.loc[
    ~exchange_strict_alignment_sensitivity["is_exchange_strict_match"],
    "exchange_strict_unmatched_reason",
] = "NO_PRIOR_EXCHANGE_TIME_BOOK_STATE"

exchange_strict_alignment_sensitivity["exchange_strict_exchange_lag_ms"] = (
    exchange_strict_alignment_sensitivity["trade_exchange_trade_time_ms"].astype("Int64")
    - exchange_strict_alignment_sensitivity["exchange_strict_book_exchange_event_time_ms"].astype("Int64")
)

exchange_strict_alignment_sensitivity["exchange_strict_sequence_lag"] = (
    exchange_strict_alignment_sensitivity["trade_collector_sequence"].astype("Int64")
    - exchange_strict_alignment_sensitivity["exchange_strict_book_collector_sequence"].astype("Int64")
)

exchange_strict_alignment_sensitivity["exchange_strict_local_lag_ns"] = (
    exchange_strict_alignment_sensitivity["trade_local_receipt_time_ns"].astype("Int64")
    - exchange_strict_alignment_sensitivity["exchange_strict_book_local_receipt_time_ns"].astype("Int64")
)

exchange_strict_alignment_sensitivity["exchange_strict_local_lag_ms"] = (
    pd.to_numeric(
        exchange_strict_alignment_sensitivity["exchange_strict_local_lag_ns"],
        errors="coerce",
    )
    / 1_000_000.0
)

exchange_strict_alignment_sensitivity["same_book_as_local_strict"] = (
    exchange_strict_alignment_sensitivity["is_exchange_strict_match"]
    & (
        exchange_strict_alignment_sensitivity["exchange_strict_book_collector_sequence"].astype("Int64")
        == exchange_strict_alignment_sensitivity["local_strict_book_collector_sequence"].astype("Int64")
    )
)

exchange_strict_alignment_sensitivity["different_book_from_local_strict"] = (
    exchange_strict_alignment_sensitivity["is_exchange_strict_match"]
    & ~exchange_strict_alignment_sensitivity["same_book_as_local_strict"]
)

exchange_strict_alignment_sensitivity["exchange_strict_sequence_noncausal_flag"] = (
    exchange_strict_alignment_sensitivity["is_exchange_strict_match"]
    & (
        exchange_strict_alignment_sensitivity["exchange_strict_book_collector_sequence"].astype("Int64")
        >= exchange_strict_alignment_sensitivity["trade_collector_sequence"].astype("Int64")
    )
)

exchange_strict_alignment_sensitivity["exchange_strict_local_time_noncausal_flag"] = (
    exchange_strict_alignment_sensitivity["is_exchange_strict_match"]
    & (
        exchange_strict_alignment_sensitivity["exchange_strict_book_local_receipt_time_ns"].astype("Int64")
        > exchange_strict_alignment_sensitivity["trade_local_receipt_time_ns"].astype("Int64")
    )
)

exchange_strict_alignment_sensitivity["exchange_strict_any_local_noncausal_flag"] = (
    exchange_strict_alignment_sensitivity["exchange_strict_sequence_noncausal_flag"]
    | exchange_strict_alignment_sensitivity["exchange_strict_local_time_noncausal_flag"]
)

exchange_strict_alignment_sensitivity["exchange_vs_local_sequence_delta"] = (
    exchange_strict_alignment_sensitivity["exchange_strict_book_collector_sequence"].astype("Int64")
    - exchange_strict_alignment_sensitivity["local_strict_book_collector_sequence"].astype("Int64")
)

exchange_strict_alignment_sensitivity["exchange_vs_local_book_exchange_time_delta_ms"] = (
    exchange_strict_alignment_sensitivity["exchange_strict_book_exchange_event_time_ms"].astype("Int64")
    - exchange_strict_alignment_sensitivity["local_strict_book_exchange_event_time_ms"].astype("Int64")
)

exchange_strict_alignment_sensitivity["exchange_vs_local_book_local_time_delta_ms"] = (
    pd.to_numeric(
        (
            exchange_strict_alignment_sensitivity["exchange_strict_book_local_receipt_time_ns"].astype("Int64")
            - exchange_strict_alignment_sensitivity["local_strict_book_local_receipt_time_ns"].astype("Int64")
        ),
        errors="coerce",
    )
    / 1_000_000.0
)

exchange_strict_alignment_sensitivity["exchange_strict_comparison_class"] = "UNCLASSIFIED"

exchange_strict_alignment_sensitivity.loc[
    ~exchange_strict_alignment_sensitivity["is_exchange_strict_match"],
    "exchange_strict_comparison_class",
] = "EXCHANGE_STRICT_UNMATCHED"

exchange_strict_alignment_sensitivity.loc[
    exchange_strict_alignment_sensitivity["same_book_as_local_strict"],
    "exchange_strict_comparison_class",
] = "SAME_AS_LOCAL_STRICT"

exchange_strict_alignment_sensitivity.loc[
    exchange_strict_alignment_sensitivity["different_book_from_local_strict"]
    & ~exchange_strict_alignment_sensitivity["exchange_strict_any_local_noncausal_flag"],
    "exchange_strict_comparison_class",
] = "DIFFERENT_BUT_LOCALLY_CAUSAL"

exchange_strict_alignment_sensitivity.loc[
    exchange_strict_alignment_sensitivity["different_book_from_local_strict"]
    & exchange_strict_alignment_sensitivity["exchange_strict_any_local_noncausal_flag"],
    "exchange_strict_comparison_class",
] = "DIFFERENT_AND_LOCALLY_NONCAUSAL"


# ------------------------------------------------------------
# Compact sensitivity summary
# ------------------------------------------------------------

exchange_strict_sensitivity_summary = (
    exchange_strict_alignment_sensitivity
    .groupby("exchange_strict_comparison_class", dropna=False)
    .agg(
        row_count=("trade_id", "size"),
        buy_count=("trade_aggressor_side", lambda x: int((x == "BUY").sum())),
        sell_count=("trade_aggressor_side", lambda x: int((x == "SELL").sum())),
        exchange_lag_ms_min=("exchange_strict_exchange_lag_ms", "min"),
        exchange_lag_ms_median=("exchange_strict_exchange_lag_ms", "median"),
        exchange_lag_ms_max=("exchange_strict_exchange_lag_ms", "max"),
        exchange_strict_sequence_lag_min=("exchange_strict_sequence_lag", "min"),
        exchange_strict_sequence_lag_median=("exchange_strict_sequence_lag", "median"),
        exchange_strict_sequence_lag_max=("exchange_strict_sequence_lag", "max"),
        exchange_strict_local_lag_ms_min=("exchange_strict_local_lag_ms", "min"),
        exchange_strict_local_lag_ms_median=("exchange_strict_local_lag_ms", "median"),
        exchange_strict_local_lag_ms_max=("exchange_strict_local_lag_ms", "max"),
        exchange_vs_local_sequence_delta_min=("exchange_vs_local_sequence_delta", "min"),
        exchange_vs_local_sequence_delta_median=("exchange_vs_local_sequence_delta", "median"),
        exchange_vs_local_sequence_delta_max=("exchange_vs_local_sequence_delta", "max"),
        sequence_noncausal_count=("exchange_strict_sequence_noncausal_flag", "sum"),
        local_time_noncausal_count=("exchange_strict_local_time_noncausal_flag", "sum"),
        any_local_noncausal_count=("exchange_strict_any_local_noncausal_flag", "sum"),
    )
    .reset_index()
    .sort_values("row_count", ascending=False)
    .reset_index(drop=True)
)

exchange_strict_sensitivity_summary["row_share"] = (
    exchange_strict_sensitivity_summary["row_count"]
    / len(exchange_strict_alignment_sensitivity)
)


# ------------------------------------------------------------
# Partition sensitivity summary
# ------------------------------------------------------------

exchange_strict_partition_summary = (
    exchange_strict_alignment_sensitivity
    .groupby(["trade_partition", "exchange_strict_comparison_class"], dropna=False)
    .agg(
        row_count=("trade_id", "size"),
        buy_count=("trade_aggressor_side", lambda x: int((x == "BUY").sum())),
        sell_count=("trade_aggressor_side", lambda x: int((x == "SELL").sum())),
        same_as_local_count=("same_book_as_local_strict", "sum"),
        different_from_local_count=("different_book_from_local_strict", "sum"),
        sequence_noncausal_count=("exchange_strict_sequence_noncausal_flag", "sum"),
        local_time_noncausal_count=("exchange_strict_local_time_noncausal_flag", "sum"),
        any_local_noncausal_count=("exchange_strict_any_local_noncausal_flag", "sum"),
        exchange_vs_local_sequence_delta_median=("exchange_vs_local_sequence_delta", "median"),
        exchange_vs_local_sequence_delta_min=("exchange_vs_local_sequence_delta", "min"),
        exchange_vs_local_sequence_delta_max=("exchange_vs_local_sequence_delta", "max"),
    )
    .reset_index()
    .sort_values(["trade_partition", "row_count"], ascending=[True, False])
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Gates
# ------------------------------------------------------------

exchange_strict_matched_count = int(
    exchange_strict_alignment_sensitivity["is_exchange_strict_match"].sum()
)

exchange_strict_unmatched_count = int(
    (~exchange_strict_alignment_sensitivity["is_exchange_strict_match"]).sum()
)

exchange_strict_exchange_time_violations = int(
    (
        exchange_strict_alignment_sensitivity["is_exchange_strict_match"]
        & (
            exchange_strict_alignment_sensitivity["exchange_strict_book_exchange_event_time_ms"].astype("Int64")
            >= exchange_strict_alignment_sensitivity["trade_exchange_trade_time_ms"].astype("Int64")
        )
    ).sum()
)

exchange_strict_unclassified_count = int(
    exchange_strict_alignment_sensitivity["exchange_strict_comparison_class"].eq("UNCLASSIFIED").sum()
)

exchange_strict_local_noncausal_count = int(
    exchange_strict_alignment_sensitivity["exchange_strict_any_local_noncausal_flag"].sum()
)

exchange_strict_different_count = int(
    exchange_strict_alignment_sensitivity["different_book_from_local_strict"].sum()
)

exchange_strict_gates: list[GateResult] = []

exchange_strict_gates.append(
    make_gate(
        gate="exchange_strict_row_count",
        passed=len(exchange_strict_alignment_sensitivity) == EXPECTED_RAW_TRADE_RECORDS,
        severity="BLOCKING",
        detail=f"observed={len(exchange_strict_alignment_sensitivity)} expected={EXPECTED_RAW_TRADE_RECORDS}",
    )
)

exchange_strict_gates.append(
    make_gate(
        gate="exchange_strict_exchange_time_valid",
        passed=exchange_strict_exchange_time_violations == 0,
        severity="BLOCKING",
        detail=f"exchange_time_violations={exchange_strict_exchange_time_violations}",
    )
)

exchange_strict_gates.append(
    make_gate(
        gate="exchange_strict_classification_exhaustive",
        passed=exchange_strict_unclassified_count == 0,
        severity="BLOCKING",
        detail=f"unclassified_count={exchange_strict_unclassified_count}",
    )
)

exchange_strict_gates.append(
    make_gate(
        gate="exchange_strict_unmatched_count",
        passed=exchange_strict_unmatched_count == 0,
        severity="WARNING",
        detail=f"matched={exchange_strict_matched_count} unmatched={exchange_strict_unmatched_count}",
    )
)

exchange_strict_gates.append(
    make_gate(
        gate="exchange_strict_differs_from_local_strict",
        passed=exchange_strict_different_count == 0,
        severity="WARNING",
        detail=f"different_book_from_local_strict_count={exchange_strict_different_count}",
    )
)

exchange_strict_gates.append(
    make_gate(
        gate="exchange_strict_locally_noncausal_matches",
        passed=exchange_strict_local_noncausal_count == 0,
        severity="WARNING",
        detail=f"locally_noncausal_count={exchange_strict_local_noncausal_count}",
    )
)

exchange_strict_gate_frame = gate_results_to_frame(exchange_strict_gates)

fail_if_blocking_gate_failed(exchange_strict_gate_frame)


# ------------------------------------------------------------
# Write sensitivity output and verify read-back
# ------------------------------------------------------------

exchange_strict_sensitivity_output = exchange_strict_alignment_sensitivity[
    [
        "sensitivity_policy",
        "trade_id",
        "trade_collector_sequence",
        "trade_local_receipt_time_ns",
        "trade_local_receipt_time",
        "trade_exchange_trade_time_ms",
        "trade_exchange_trade_time",
        "trade_aggressor_side",
        "trade_price",
        "trade_quantity",
        "trade_partition",

        "local_strict_book_collector_sequence",
        "local_strict_book_local_receipt_time_ns",
        "local_strict_book_exchange_event_time_ms",
        "local_strict_book_best_bid",
        "local_strict_book_best_ask",
        "local_strict_book_partition",

        "exchange_strict_book_state_id",
        "exchange_strict_book_collector_sequence",
        "exchange_strict_book_local_receipt_time_ns",
        "exchange_strict_book_local_receipt_time",
        "exchange_strict_book_exchange_event_time_ms",
        "exchange_strict_book_exchange_event_time",
        "exchange_strict_book_best_bid",
        "exchange_strict_book_best_ask",
        "exchange_strict_book_spread",
        "exchange_strict_book_midpoint",
        "exchange_strict_book_microprice",
        "exchange_strict_book_partition",

        "is_exchange_strict_match",
        "exchange_strict_unmatched_reason",
        "exchange_strict_exchange_lag_ms",
        "exchange_strict_sequence_lag",
        "exchange_strict_local_lag_ns",
        "exchange_strict_local_lag_ms",
        "same_book_as_local_strict",
        "different_book_from_local_strict",
        "exchange_strict_sequence_noncausal_flag",
        "exchange_strict_local_time_noncausal_flag",
        "exchange_strict_any_local_noncausal_flag",
        "exchange_vs_local_sequence_delta",
        "exchange_vs_local_book_exchange_time_delta_ms",
        "exchange_vs_local_book_local_time_delta_ms",
        "exchange_strict_comparison_class",
    ]
].copy()

exchange_strict_sensitivity_metadata = write_csv_and_verify(
    exchange_strict_sensitivity_output,
    EXCHANGE_STRICT_SENSITIVITY_PATH,
    expected_rows=len(exchange_strict_sensitivity_output),
)

exchange_strict_output_metadata = {
    "exchange_strict_alignment_sensitivity": exchange_strict_sensitivity_metadata,
}


# ------------------------------------------------------------
# Compact display
# ------------------------------------------------------------

display(exchange_strict_sensitivity_summary)
display(exchange_strict_partition_summary)
display(exchange_strict_gate_frame)

exchange_strict_output_metadata

,exchange_strict_comparison_class,row_count,buy_count,sell_count,exchange_lag_ms_min,exchange_lag_ms_median,exchange_lag_ms_max,exchange_strict_sequence_lag_min,exchange_strict_sequence_lag_median,exchange_strict_sequence_lag_max,exchange_strict_local_lag_ms_min,exchange_strict_local_lag_ms_median,exchange_strict_local_lag_ms_max,exchange_vs_local_sequence_delta_min,exchange_vs_local_sequence_delta_median,exchange_vs_local_sequence_delta_max,sequence_noncausal_count,local_time_noncausal_count,any_local_noncausal_count,row_share
0,SAME_AS_LOCAL_STRICT,66173,30028,36145,1,49.0,102,1,30.0,461,0.0,54.0308,113.3191,0,0.0,0,0,0,0,0.97769
1,DIFFERENT_BUT_LOCALLY_CAUSAL,1510,568,942,84,99.0,103,2,43.5,388,92.015,104.0237,117.8238,-385,-2.0,-1,0,0,0,0.02231


,trade_partition,exchange_strict_comparison_class,row_count,buy_count,sell_count,same_as_local_count,different_from_local_count,sequence_noncausal_count,local_time_noncausal_count,any_local_noncausal_count,exchange_vs_local_sequence_delta_median,exchange_vs_local_sequence_delta_min,exchange_vs_local_sequence_delta_max
0,CALIBRATION,SAME_AS_LOCAL_STRICT,13337,7187,6150,13337,0,0,0,0,0.0,0,0
1,CALIBRATION,DIFFERENT_BUT_LOCALLY_CAUSAL,196,80,116,0,196,0,0,0,-1.0,-204,-1
2,DEVELOPMENT,SAME_AS_LOCAL_STRICT,32959,14807,18152,32959,0,0,0,0,0.0,0,0
3,DEVELOPMENT,DIFFERENT_BUT_LOCALLY_CAUSAL,861,387,474,0,861,0,0,0,-2.0,-385,-1
4,ENGINEERING_HOLDOUT,SAME_AS_LOCAL_STRICT,9814,2357,7457,9814,0,0,0,0,0.0,0,0
5,ENGINEERING_HOLDOUT,DIFFERENT_BUT_LOCALLY_CAUSAL,319,4,315,0,319,0,0,0,-7.0,-313,-1
6,VALIDATION,SAME_AS_LOCAL_STRICT,10063,5677,4386,10063,0,0,0,0,0.0,0,0
7,VALIDATION,DIFFERENT_BUT_LOCALLY_CAUSAL,134,97,37,0,134,0,0,0,-119.0,-133,-1


,gate,status,severity,detail
0,exchange_strict_row_count,PASS,BLOCKING,observed=67683 expected=67683
1,exchange_strict_exchange_time_valid,PASS,BLOCKING,exchange_time_violations=0
2,exchange_strict_classification_exhaustive,PASS,BLOCKING,unclassified_count=0
3,exchange_strict_unmatched_count,PASS,WARNING,matched=67683 unmatched=0
4,exchange_strict_differs_from_local_strict,FAIL,WARNING,different_book_from_local_strict_count=1510
5,exchange_strict_locally_noncausal_matches,PASS,WARNING,locally_noncausal_count=0


{'exchange_strict_alignment_sensitivity': {'path': 'D:\\Clown Project\\V0.1\\artifacts\\audit_tables\\03_CAUSAL_TRADE_BOOK_ALIGNMENT\\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__03_CAUSAL_TRADE_BOOK_ALIGNMENT__exchange_strict_alignment_sensitivity.csv',
  'row_count': 67683,
  'size_bytes': 35771219,
  'sha256': '18d45cb0ae6ced11c8c877f737c44ba48bff7a61303ee1accf28d85f7fa499f5'}}

In [14]:
# ============================================================
# Cell 12 — V0.0 synchronization reference reconciliation
# ============================================================

# This cell searches for V0.0 trade-book synchronization / alignment reference
# artifacts and compares them against the V0.1 LOCAL_STRICT alignment when a
# suitable reference file exists.
#
# V0.0 references are provenance only.
# They must not replace V0.1 reconstruction or LOCAL_STRICT alignment.
#
# Missing V0.0 synchronization reference files are warnings, not blockers.


# ------------------------------------------------------------
# Candidate discovery
# ------------------------------------------------------------

V00_PROCESSED_ROOT = V00_ROOT / "data" / "processed"
V00_REFERENCE_SEARCH_ROOTS = [
    V00_PROCESSED_ROOT / "trades",
    V00_PROCESSED_ROOT / "book",
    V00_PROCESSED_ROOT / "events",
    V00_PROCESSED_ROOT / "point_process",
    V00_PROCESSED_ROOT,
]

V00_SYNC_REFERENCE_NAME_TOKENS = [
    "sync",
    "synchron",
    "align",
    "trade_book",
    "trades_book",
    "book_trade",
    "matched",
    "causal",
]

V00_SYNC_REFERENCE_EXTENSIONS = {
    ".csv",
    ".parquet",
    ".pq",
    ".json",
    ".jsonl",
}


def discover_v00_sync_reference_candidates() -> pd.DataFrame:
    """Find likely V0.0 trade-book synchronization reference files."""
    rows = []
    seen_paths: set[str] = set()

    for root in V00_REFERENCE_SEARCH_ROOTS:
        root = Path(root)

        if not root.exists():
            continue

        for path in root.rglob("*"):
            if not path.is_file():
                continue

            if path.suffix.lower() not in V00_SYNC_REFERENCE_EXTENSIONS:
                continue

            path_str = str(path)
            path_name_lower = path.name.lower()
            path_full_lower = path_str.lower()

            if path_str in seen_paths:
                continue

            if SOURCE_RUN_PREFIX.lower() not in path_full_lower:
                continue

            token_hits = [
                token
                for token in V00_SYNC_REFERENCE_NAME_TOKENS
                if token in path_name_lower or token in path_full_lower
            ]

            if not token_hits:
                continue

            seen_paths.add(path_str)

            rows.append(
                {
                    "path": path_str,
                    "filename": path.name,
                    "suffix": path.suffix.lower(),
                    "parent": str(path.parent),
                    "size_bytes": path.stat().st_size,
                    "modified_time_utc": datetime.fromtimestamp(
                        path.stat().st_mtime,
                        tz=timezone.utc,
                    ).isoformat(),
                    "token_hits": ",".join(token_hits),
                    "candidate_score": len(token_hits),
                }
            )

    if not rows:
        return pd.DataFrame(
            columns=[
                "path",
                "filename",
                "suffix",
                "parent",
                "size_bytes",
                "modified_time_utc",
                "token_hits",
                "candidate_score",
            ]
        )

    return (
        pd.DataFrame(rows)
        .sort_values(
            ["candidate_score", "size_bytes", "filename"],
            ascending=[False, False, True],
        )
        .reset_index(drop=True)
    )


def load_reference_frame(path: Path) -> pd.DataFrame:
    """Load a reference artifact into a DataFrame."""
    path = require_file(path)
    suffix = path.suffix.lower()

    if suffix in {".csv"}:
        return pd.read_csv(path, low_memory=False)

    if suffix in {".parquet", ".pq"}:
        return pd.read_parquet(path)

    if suffix == ".jsonl":
        records = read_jsonl_records(path, expected_rows=None)
        return pd.DataFrame(records)

    if suffix == ".json":
        payload = read_json_file(path)

        if isinstance(payload, dict):
            for key in ["records", "data", "rows", "items"]:
                value = payload.get(key)
                if isinstance(value, list):
                    return pd.DataFrame(value)

            return pd.json_normalize(payload)

        return pd.DataFrame()

    raise ValueError(f"Unsupported reference suffix: {suffix}")


v00_sync_reference_candidates = discover_v00_sync_reference_candidates()


# ------------------------------------------------------------
# Select first loadable reference with trade-like rows
# ------------------------------------------------------------

v00_sync_reference_selection_rows = []
v00_sync_reference_frame = pd.DataFrame()
v00_sync_reference_path: Path | None = None
v00_sync_reference_load_status = "NO_REFERENCE_CANDIDATES_FOUND"

for _, candidate in v00_sync_reference_candidates.iterrows():
    candidate_path = Path(candidate["path"])

    try:
        candidate_frame = load_reference_frame(candidate_path)
        load_error = None
    except Exception as exc:
        candidate_frame = pd.DataFrame()
        load_error = repr(exc)

    candidate_columns = list(candidate_frame.columns)
    candidate_row_count = int(len(candidate_frame))

    has_trade_id_like = any(
        normalize_column_name(column) in {
            "trade_id",
            "t",
            "raw_trade_id",
            "tradeid",
        }
        for column in candidate_columns
    )

    has_trade_sequence_like = any(
        normalize_column_name(column) in {
            "trade_collector_sequence",
            "collector_sequence",
            "raw_trade_collector_sequence",
            "trade_sequence",
        }
        for column in candidate_columns
    )

    has_book_sequence_like = any(
        "book" in normalize_column_name(column)
        and "sequence" in normalize_column_name(column)
        for column in candidate_columns
    )

    useful_reference = (
        load_error is None
        and candidate_row_count > 0
        and (has_trade_id_like or has_trade_sequence_like)
    )

    v00_sync_reference_selection_rows.append(
        {
            "path": str(candidate_path),
            "filename": candidate_path.name,
            "suffix": candidate_path.suffix.lower(),
            "load_status": "LOADED" if load_error is None else "LOAD_FAILED",
            "load_error": load_error,
            "row_count": candidate_row_count,
            "column_count": len(candidate_columns),
            "has_trade_id_like": has_trade_id_like,
            "has_trade_sequence_like": has_trade_sequence_like,
            "has_book_sequence_like": has_book_sequence_like,
            "selected": useful_reference and v00_sync_reference_path is None,
        }
    )

    if useful_reference and v00_sync_reference_path is None:
        v00_sync_reference_frame = candidate_frame.copy()
        v00_sync_reference_path = candidate_path
        v00_sync_reference_load_status = "REFERENCE_SELECTED"

v00_sync_reference_selection_audit = pd.DataFrame(v00_sync_reference_selection_rows)

if v00_sync_reference_path is None and not v00_sync_reference_candidates.empty:
    v00_sync_reference_load_status = "NO_LOADABLE_TRADE_LIKE_REFERENCE_SELECTED"


# ------------------------------------------------------------
# Reference reconciliation helpers
# ------------------------------------------------------------

V00_TRADE_ID_ALIASES = [
    "trade_id",
    "raw_trade_id",
    "binance_trade_id",
    "t",
    "tradeid",
]

V00_TRADE_COLLECTOR_SEQUENCE_ALIASES = [
    "trade_collector_sequence",
    "raw_trade_collector_sequence",
    "collector_sequence_trade",
    "trade_sequence",
    "collector_sequence",
    "sequence",
]

V00_TRADE_LOCAL_RECEIPT_NS_ALIASES = [
    "trade_local_receipt_time_ns",
    "raw_trade_local_receipt_time_ns",
    "local_receipt_time_ns_trade",
    "trade_receipt_time_ns",
    "local_receipt_time_ns",
]

V00_TRADE_EXCHANGE_TIME_MS_ALIASES = [
    "trade_exchange_trade_time_ms",
    "exchange_trade_time_ms",
    "trade_time_ms",
    "T",
]

V00_AGGRESSOR_SIDE_ALIASES = [
    "trade_aggressor_side",
    "aggressor_side",
    "side",
    "trade_side",
    "signed_side",
]

V00_BUYER_IS_MAKER_ALIASES = [
    "trade_buyer_is_maker",
    "buyer_is_maker",
    "m",
]

V00_TRADE_PRICE_ALIASES = [
    "trade_price",
    "price",
    "p",
]

V00_TRADE_QUANTITY_ALIASES = [
    "trade_quantity",
    "quantity",
    "qty",
    "q",
]

V00_BOOK_COLLECTOR_SEQUENCE_ALIASES = [
    "book_collector_sequence",
    "matched_book_collector_sequence",
    "visible_book_collector_sequence",
    "book_sequence",
    "matched_book_sequence",
    "book_state_collector_sequence",
]

V00_BOOK_LOCAL_RECEIPT_NS_ALIASES = [
    "book_local_receipt_time_ns",
    "matched_book_local_receipt_time_ns",
    "visible_book_local_receipt_time_ns",
    "book_receipt_time_ns",
]

V00_BOOK_EXCHANGE_EVENT_TIME_MS_ALIASES = [
    "book_exchange_event_time_ms",
    "matched_book_exchange_event_time_ms",
    "visible_book_exchange_event_time_ms",
    "book_event_time_ms",
]

V00_BOOK_BEST_BID_ALIASES = [
    "book_best_bid",
    "best_bid",
    "best_bid_price",
    "bid_price_1",
    "bid_1_price",
]

V00_BOOK_BEST_ASK_ALIASES = [
    "book_best_ask",
    "best_ask",
    "best_ask_price",
    "ask_price_1",
    "ask_1_price",
]

V00_BOOK_MIDPOINT_ALIASES = [
    "book_midpoint",
    "midpoint",
    "mid_price",
    "mid",
]

V00_BOOK_SPREAD_ALIASES = [
    "book_spread",
    "spread",
    "bid_ask_spread",
]


def maybe_resolve_reference_column(frame: pd.DataFrame, aliases: list[str]) -> str | None:
    """Resolve a reference column if available."""
    if frame.empty:
        return None

    return resolve_column(
        frame,
        aliases,
        "v00_sync_reference_frame",
        required=False,
    )


def canonicalize_v00_sync_reference(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Create a canonical V0.0 reference comparison frame and a column-resolution audit."""
    if frame.empty:
        return pd.DataFrame(), pd.DataFrame(
            columns=["logical_column", "source_column", "available"]
        )

    resolution = {
        "v00_trade_id": maybe_resolve_reference_column(frame, V00_TRADE_ID_ALIASES),
        "v00_trade_collector_sequence": maybe_resolve_reference_column(frame, V00_TRADE_COLLECTOR_SEQUENCE_ALIASES),
        "v00_trade_local_receipt_time_ns": maybe_resolve_reference_column(frame, V00_TRADE_LOCAL_RECEIPT_NS_ALIASES),
        "v00_trade_exchange_trade_time_ms": maybe_resolve_reference_column(frame, V00_TRADE_EXCHANGE_TIME_MS_ALIASES),
        "v00_aggressor_side": maybe_resolve_reference_column(frame, V00_AGGRESSOR_SIDE_ALIASES),
        "v00_buyer_is_maker": maybe_resolve_reference_column(frame, V00_BUYER_IS_MAKER_ALIASES),
        "v00_trade_price": maybe_resolve_reference_column(frame, V00_TRADE_PRICE_ALIASES),
        "v00_trade_quantity": maybe_resolve_reference_column(frame, V00_TRADE_QUANTITY_ALIASES),
        "v00_book_collector_sequence": maybe_resolve_reference_column(frame, V00_BOOK_COLLECTOR_SEQUENCE_ALIASES),
        "v00_book_local_receipt_time_ns": maybe_resolve_reference_column(frame, V00_BOOK_LOCAL_RECEIPT_NS_ALIASES),
        "v00_book_exchange_event_time_ms": maybe_resolve_reference_column(frame, V00_BOOK_EXCHANGE_EVENT_TIME_MS_ALIASES),
        "v00_book_best_bid": maybe_resolve_reference_column(frame, V00_BOOK_BEST_BID_ALIASES),
        "v00_book_best_ask": maybe_resolve_reference_column(frame, V00_BOOK_BEST_ASK_ALIASES),
        "v00_book_midpoint": maybe_resolve_reference_column(frame, V00_BOOK_MIDPOINT_ALIASES),
        "v00_book_spread": maybe_resolve_reference_column(frame, V00_BOOK_SPREAD_ALIASES),
    }

    canonical = pd.DataFrame(index=frame.index)

    for logical_column, source_column in resolution.items():
        if source_column is None:
            canonical[logical_column] = pd.NA
        else:
            canonical[logical_column] = frame[source_column]

    for column in [
        "v00_trade_id",
        "v00_trade_collector_sequence",
        "v00_trade_local_receipt_time_ns",
        "v00_trade_exchange_trade_time_ms",
        "v00_book_collector_sequence",
        "v00_book_local_receipt_time_ns",
        "v00_book_exchange_event_time_ms",
    ]:
        canonical[column] = coerce_int_series(canonical[column])

    for column in [
        "v00_trade_price",
        "v00_trade_quantity",
        "v00_book_best_bid",
        "v00_book_best_ask",
        "v00_book_midpoint",
        "v00_book_spread",
    ]:
        canonical[column] = coerce_float_series(canonical[column])

    if canonical["v00_aggressor_side"].isna().all() and canonical["v00_buyer_is_maker"].notna().any():
        buyer_is_maker = coerce_bool_series(canonical["v00_buyer_is_maker"])
        canonical["v00_aggressor_side"] = pd.Series(pd.NA, index=canonical.index, dtype="string")
        canonical.loc[buyer_is_maker == False, "v00_aggressor_side"] = "BUY"
        canonical.loc[buyer_is_maker == True, "v00_aggressor_side"] = "SELL"
    else:
        canonical["v00_aggressor_side"] = canonical["v00_aggressor_side"].astype("string").str.upper()

    canonical["v00_reference_row_number"] = np.arange(1, len(canonical) + 1)

    resolution_audit = pd.DataFrame(
        [
            {
                "logical_column": logical_column,
                "source_column": source_column,
                "available": source_column is not None,
            }
            for logical_column, source_column in resolution.items()
        ]
    )

    return canonical, resolution_audit


v00_sync_reference_canonical, v00_sync_column_resolution_audit = canonicalize_v00_sync_reference(
    v00_sync_reference_frame
)


# ------------------------------------------------------------
# Reconcile against V0.1 LOCAL_STRICT alignment
# ------------------------------------------------------------

v00_reconciliation_findings = []
v00_field_reconciliation_rows = []
v00_reference_reconciliation_summary_rows = []

v01_for_v00_reconciliation = local_strict_all_alignment[
    [
        "trade_id",
        "trade_collector_sequence",
        "trade_local_receipt_time_ns",
        "trade_exchange_trade_time_ms",
        "trade_aggressor_side",
        "trade_price",
        "trade_quantity",
        "book_collector_sequence",
        "book_local_receipt_time_ns",
        "book_exchange_event_time_ms",
        "book_best_bid",
        "book_best_ask",
        "book_midpoint",
        "book_spread",
        "is_local_strict_match",
    ]
].copy()

if v00_sync_reference_path is None:
    v00_reconciliation_findings.append(
        {
            "finding_class": "REFERENCE_NOT_FOUND",
            "severity": "WARNING",
            "blocking": False,
            "detail": (
                "No loadable V0.0 trade-book synchronization reference artifact was selected. "
                "V0.1 LOCAL_STRICT alignment remains authoritative."
            ),
        }
    )

    v00_reference_reconciliation_summary_rows.append(
        {
            "metric": "v00_reference_status",
            "value": v00_sync_reference_load_status,
        }
    )

else:
    v00_reference_hash = sha256_file(v00_sync_reference_path)

    reference_join_key = None

    if (
        "v00_trade_id" in v00_sync_reference_canonical.columns
        and v00_sync_reference_canonical["v00_trade_id"].notna().any()
        and not v00_sync_reference_canonical["v00_trade_id"].dropna().duplicated().any()
    ):
        reference_join_key = "trade_id"

        v00_join_ready = v00_sync_reference_canonical.rename(
            columns={"v00_trade_id": "trade_id"}
        )

        v01_join_ready = v01_for_v00_reconciliation.copy()

    elif (
        "v00_trade_collector_sequence" in v00_sync_reference_canonical.columns
        and v00_sync_reference_canonical["v00_trade_collector_sequence"].notna().any()
        and not v00_sync_reference_canonical["v00_trade_collector_sequence"].dropna().duplicated().any()
    ):
        reference_join_key = "trade_collector_sequence"

        v00_join_ready = v00_sync_reference_canonical.rename(
            columns={"v00_trade_collector_sequence": "trade_collector_sequence"}
        )

        v01_join_ready = v01_for_v00_reconciliation.copy()

    else:
        reference_join_key = None

    v00_reference_reconciliation_summary_rows.extend(
        [
            {"metric": "v00_reference_status", "value": v00_sync_reference_load_status},
            {"metric": "v00_reference_path", "value": str(v00_sync_reference_path)},
            {"metric": "v00_reference_sha256", "value": v00_reference_hash},
            {"metric": "v00_reference_row_count", "value": int(len(v00_sync_reference_canonical))},
            {"metric": "v01_local_strict_row_count", "value": int(len(v01_for_v00_reconciliation))},
            {"metric": "reference_join_key", "value": reference_join_key},
        ]
    )

    if reference_join_key is None:
        v00_reconciliation_findings.append(
            {
                "finding_class": "REFERENCE_SCHEMA_SCOPE_DIFFERENCE",
                "severity": "WARNING",
                "blocking": False,
                "detail": (
                    "V0.0 reference was loadable but has no unique trade_id or trade_collector_sequence "
                    "available for row-level reconciliation."
                ),
            }
        )

    else:
        joined_reference = v01_join_ready.merge(
            v00_join_ready,
            how="outer",
            on=reference_join_key,
            indicator=True,
            suffixes=("_v01", "_v00"),
            validate="one_to_one",
        )

        matched_key_rows = int((joined_reference["_merge"] == "both").sum())
        v01_only_rows = int((joined_reference["_merge"] == "left_only").sum())
        v00_only_rows = int((joined_reference["_merge"] == "right_only").sum())

        v00_reference_reconciliation_summary_rows.extend(
            [
                {"metric": "matched_key_rows", "value": matched_key_rows},
                {"metric": "v01_only_rows", "value": v01_only_rows},
                {"metric": "v00_only_rows", "value": v00_only_rows},
            ]
        )

        if v01_only_rows > 0 or v00_only_rows > 0:
            v00_reconciliation_findings.append(
                {
                    "finding_class": "REFERENCE_ROW_SCOPE_DIFFERENCE",
                    "severity": "WARNING",
                    "blocking": False,
                    "detail": f"v01_only_rows={v01_only_rows}; v00_only_rows={v00_only_rows}",
                }
            )

        field_pairs = [
            {
                "field": "trade_collector_sequence",
                "v01_column": "trade_collector_sequence",
                "v00_column": "v00_trade_collector_sequence",
                "comparison_type": "integer_exact",
            },
            {
                "field": "trade_local_receipt_time_ns",
                "v01_column": "trade_local_receipt_time_ns",
                "v00_column": "v00_trade_local_receipt_time_ns",
                "comparison_type": "integer_exact",
            },
            {
                "field": "trade_exchange_trade_time_ms",
                "v01_column": "trade_exchange_trade_time_ms",
                "v00_column": "v00_trade_exchange_trade_time_ms",
                "comparison_type": "integer_exact",
            },
            {
                "field": "trade_aggressor_side",
                "v01_column": "trade_aggressor_side",
                "v00_column": "v00_aggressor_side",
                "comparison_type": "string_exact",
            },
            {
                "field": "trade_price",
                "v01_column": "trade_price",
                "v00_column": "v00_trade_price",
                "comparison_type": "float_close",
            },
            {
                "field": "trade_quantity",
                "v01_column": "trade_quantity",
                "v00_column": "v00_trade_quantity",
                "comparison_type": "float_close",
            },
            {
                "field": "matched_book_collector_sequence",
                "v01_column": "book_collector_sequence",
                "v00_column": "v00_book_collector_sequence",
                "comparison_type": "integer_exact",
            },
            {
                "field": "matched_book_local_receipt_time_ns",
                "v01_column": "book_local_receipt_time_ns",
                "v00_column": "v00_book_local_receipt_time_ns",
                "comparison_type": "integer_exact",
            },
            {
                "field": "matched_book_exchange_event_time_ms",
                "v01_column": "book_exchange_event_time_ms",
                "v00_column": "v00_book_exchange_event_time_ms",
                "comparison_type": "integer_exact",
            },
            {
                "field": "book_best_bid",
                "v01_column": "book_best_bid",
                "v00_column": "v00_book_best_bid",
                "comparison_type": "float_close",
            },
            {
                "field": "book_best_ask",
                "v01_column": "book_best_ask",
                "v00_column": "v00_book_best_ask",
                "comparison_type": "float_close",
            },
            {
                "field": "book_midpoint",
                "v01_column": "book_midpoint",
                "v00_column": "v00_book_midpoint",
                "comparison_type": "float_close",
            },
            {
                "field": "book_spread",
                "v01_column": "book_spread",
                "v00_column": "v00_book_spread",
                "comparison_type": "float_close",
            },
        ]

        both_rows = joined_reference["_merge"] == "both"

        for pair in field_pairs:
            field = pair["field"]
            v01_column = pair["v01_column"]
            v00_column = pair["v00_column"]
            comparison_type = pair["comparison_type"]

            if v01_column not in joined_reference.columns:
                v01_available = False
            else:
                v01_available = joined_reference.loc[both_rows, v01_column].notna().any()

            if v00_column not in joined_reference.columns:
                v00_available = False
            else:
                v00_available = joined_reference.loc[both_rows, v00_column].notna().any()

            if not v01_available or not v00_available:
                comparison_class = "REFERENCE_SCHEMA_SCOPE_DIFFERENCE"
                comparable_count = 0
                match_count = 0
                mismatch_count = 0
                null_pair_count = 0

            else:
                left = joined_reference.loc[both_rows, v01_column]
                right = joined_reference.loc[both_rows, v00_column]

                comparable_mask = left.notna() & right.notna()
                null_pair_mask = left.isna() & right.isna()

                comparable_count = int(comparable_mask.sum())
                null_pair_count = int(null_pair_mask.sum())

                if comparison_type == "float_close":
                    matches = np.isclose(
                        pd.to_numeric(left[comparable_mask], errors="coerce"),
                        pd.to_numeric(right[comparable_mask], errors="coerce"),
                        rtol=0.0,
                        atol=1e-9,
                    )
                    match_count = int(np.sum(matches))
                else:
                    match_count = int(
                        (
                            left[comparable_mask].astype("string")
                            == right[comparable_mask].astype("string")
                        ).sum()
                    )

                mismatch_count = comparable_count - match_count

                if mismatch_count == 0:
                    comparison_class = "EXACT_OR_CLOSE_MATCH"
                elif field in {
                    "book_midpoint",
                    "book_spread",
                }:
                    comparison_class = "DERIVED_METRIC_CONVENTION_DIFFERENCE"
                elif field in {
                    "matched_book_collector_sequence",
                    "matched_book_local_receipt_time_ns",
                    "matched_book_exchange_event_time_ms",
                }:
                    comparison_class = "ALIGNMENT_POLICY_DIFFERENCE"
                else:
                    comparison_class = "FIELD_VALUE_DIFFERENCE"

            v00_field_reconciliation_rows.append(
                {
                    "field": field,
                    "v01_column": v01_column,
                    "v00_column": v00_column,
                    "comparison_type": comparison_type,
                    "v01_available": v01_available,
                    "v00_available": v00_available,
                    "comparison_class": comparison_class,
                    "matched_key_rows": matched_key_rows,
                    "comparable_count": comparable_count,
                    "match_count": match_count,
                    "mismatch_count": mismatch_count,
                    "null_pair_count": null_pair_count,
                    "mismatch_share_of_comparable": (
                        mismatch_count / comparable_count
                        if comparable_count
                        else np.nan
                    ),
                }
            )

            if comparison_class not in {
                "EXACT_OR_CLOSE_MATCH",
                "REFERENCE_SCHEMA_SCOPE_DIFFERENCE",
            }:
                v00_reconciliation_findings.append(
                    {
                        "finding_class": comparison_class,
                        "severity": "WARNING",
                        "blocking": False,
                        "detail": (
                            f"{field}: mismatch_count={mismatch_count}; "
                            f"comparable_count={comparable_count}"
                        ),
                    }
                )


# ------------------------------------------------------------
# Final reconciliation tables
# ------------------------------------------------------------

v00_sync_reconciliation_summary = pd.DataFrame(
    v00_reference_reconciliation_summary_rows
)

if v00_sync_reconciliation_summary.empty:
    v00_sync_reconciliation_summary = pd.DataFrame(
        [
            {
                "metric": "v00_reference_status",
                "value": v00_sync_reference_load_status,
            }
        ]
    )

v00_sync_field_reconciliation = pd.DataFrame(v00_field_reconciliation_rows)

if v00_sync_field_reconciliation.empty:
    v00_sync_field_reconciliation = pd.DataFrame(
        columns=[
            "field",
            "v01_column",
            "v00_column",
            "comparison_type",
            "v01_available",
            "v00_available",
            "comparison_class",
            "matched_key_rows",
            "comparable_count",
            "match_count",
            "mismatch_count",
            "null_pair_count",
            "mismatch_share_of_comparable",
        ]
    )

v00_sync_reference_findings = pd.DataFrame(v00_reconciliation_findings)

if v00_sync_reference_findings.empty:
    v00_sync_reference_findings = pd.DataFrame(
        [
            {
                "finding_class": "REFERENCE_RECONCILIATION_COMPLETE",
                "severity": "INFO",
                "blocking": False,
                "detail": "No V0.0 reference-blocking findings. V0.1 LOCAL_STRICT remains authoritative.",
            }
        ]
    )


# ------------------------------------------------------------
# Gates
# ------------------------------------------------------------

v00_reference_reconciliation_gates: list[GateResult] = []

v00_reference_reconciliation_gates.append(
    make_gate(
        gate="v00_reference_not_used_as_authority",
        passed=True,
        severity="BLOCKING",
        detail="V0.0 synchronization references are used only for reconciliation/provenance.",
    )
)

v00_reference_reconciliation_gates.append(
    make_gate(
        gate="v01_local_strict_alignment_still_authority",
        passed=len(local_strict_all_alignment) == EXPECTED_RAW_TRADE_RECORDS,
        severity="BLOCKING",
        detail=f"v01_alignment_rows={len(local_strict_all_alignment)} expected={EXPECTED_RAW_TRADE_RECORDS}",
    )
)

v00_reference_reconciliation_gates.append(
    make_gate(
        gate="v00_reference_selected",
        passed=v00_sync_reference_path is not None,
        severity="WARNING",
        detail=v00_sync_reference_load_status,
    )
)

v00_reference_reconciliation_gates.append(
    make_gate(
        gate="v00_reference_findings_nonblocking",
        passed=not bool(v00_sync_reference_findings["blocking"].fillna(False).any()),
        severity="BLOCKING",
        detail=f"finding_count={len(v00_sync_reference_findings)}",
    )
)

v00_reference_reconciliation_gate_frame = gate_results_to_frame(
    v00_reference_reconciliation_gates
)

fail_if_blocking_gate_failed(v00_reference_reconciliation_gate_frame)


# ------------------------------------------------------------
# Write reconciliation outputs and verify read-back
# ------------------------------------------------------------

v00_sync_reconciliation_summary_metadata = write_csv_and_verify(
    v00_sync_reconciliation_summary,
    V00_SYNC_RECONCILIATION_SUMMARY_PATH,
    expected_rows=len(v00_sync_reconciliation_summary),
)

v00_sync_field_reconciliation_metadata = write_csv_and_verify(
    v00_sync_field_reconciliation,
    V00_SYNC_FIELD_RECONCILIATION_PATH,
    expected_rows=len(v00_sync_field_reconciliation),
)

v00_sync_reference_findings_metadata = write_csv_and_verify(
    v00_sync_reference_findings,
    V00_SYNC_REFERENCE_FINDINGS_PATH,
    expected_rows=len(v00_sync_reference_findings),
)

v00_reference_reconciliation_output_metadata = {
    "v00_sync_reconciliation_summary": v00_sync_reconciliation_summary_metadata,
    "v00_sync_field_reconciliation": v00_sync_field_reconciliation_metadata,
    "v00_sync_reference_findings": v00_sync_reference_findings_metadata,
}


# ------------------------------------------------------------
# Compact display
# ------------------------------------------------------------

display(v00_sync_reference_candidates.head(20))
display(v00_sync_reference_selection_audit.head(20))
display(v00_sync_column_resolution_audit)
display(v00_sync_reconciliation_summary)
display(v00_sync_field_reconciliation)
display(v00_sync_reference_findings)
display(v00_reference_reconciliation_gate_frame)

v00_reference_reconciliation_output_metadata

,path,filename,suffix,parent,size_bytes,modified_time_utc,token_hits,candidate_score
0,D:\Clown Project\V0.0\data\processed\hawkes\BTCUSDT_spot_20260710T063746Z_c8b5bf12_causal_intensity_event_replay.parquet,BTCUSDT_spot_20260710T063746Z_c8b5bf12_causal_intensity_event_replay.parquet,.parquet,D:\Clown Project\V0.0\data\processed\hawkes,2572953,2026-07-10T10:15:13.369755+00:00,causal,1
1,D:\Clown Project\V0.0\data\processed\hawkes\BTCUSDT_spot_20260710T063746Z_c8b5bf12_post_event_causal_intensities_repaired.parquet,BTCUSDT_spot_20260710T063746Z_c8b5bf12_post_event_causal_intensities_repaired.parquet,.parquet,D:\Clown Project\V0.0\data\processed\hawkes,2135124,2026-07-10T11:37:35.113713+00:00,causal,1


,path,filename,suffix,load_status,load_error,row_count,column_count,has_trade_id_like,has_trade_sequence_like,has_book_sequence_like,selected
0,D:\Clown Project\V0.0\data\processed\hawkes\BTCUSDT_spot_20260710T063746Z_c8b5bf12_causal_intensity_event_replay.parquet,BTCUSDT_spot_20260710T063746Z_c8b5bf12_causal_intensity_event_replay.parquet,.parquet,LOAD_FAILED,"ImportError(""Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.\nA suitable version of pyarrow or fastparquet is required for parquet su...",0,0,False,False,False,False
1,D:\Clown Project\V0.0\data\processed\hawkes\BTCUSDT_spot_20260710T063746Z_c8b5bf12_post_event_causal_intensities_repaired.parquet,BTCUSDT_spot_20260710T063746Z_c8b5bf12_post_event_causal_intensities_repaired.parquet,.parquet,LOAD_FAILED,"ImportError(""Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.\nA suitable version of pyarrow or fastparquet is required for parquet su...",0,0,False,False,False,False


,logical_column,source_column,available


,metric,value
0,v00_reference_status,NO_LOADABLE_TRADE_LIKE_REFERENCE_SELECTED


,field,v01_column,v00_column,comparison_type,v01_available,v00_available,comparison_class,matched_key_rows,comparable_count,match_count,mismatch_count,null_pair_count,mismatch_share_of_comparable


,finding_class,severity,blocking,detail
0,REFERENCE_NOT_FOUND,WARNING,False,No loadable V0.0 trade-book synchronization reference artifact was selected. V0.1 LOCAL_STRICT alignment remains authoritative.


,gate,status,severity,detail
0,v00_reference_not_used_as_authority,PASS,BLOCKING,V0.0 synchronization references are used only for reconciliation/provenance.
1,v01_local_strict_alignment_still_authority,PASS,BLOCKING,v01_alignment_rows=67683 expected=67683
2,v00_reference_selected,FAIL,WARNING,NO_LOADABLE_TRADE_LIKE_REFERENCE_SELECTED
3,v00_reference_findings_nonblocking,PASS,BLOCKING,finding_count=1


{'v00_sync_reconciliation_summary': {'path': 'D:\\Clown Project\\V0.1\\artifacts\\reconciliation\\03_CAUSAL_TRADE_BOOK_ALIGNMENT\\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__03_CAUSAL_TRADE_BOOK_ALIGNMENT__v0_0_synchronization_reconciliation_summary.csv',
  'row_count': 1,
  'size_bytes': 78,
  'sha256': '2b82787caf564752d60cb72444b26430d231d6181ae2a39a0ea35ab6e7363a5e'},
 'v00_sync_field_reconciliation': {'path': 'D:\\Clown Project\\V0.1\\artifacts\\reconciliation\\03_CAUSAL_TRADE_BOOK_ALIGNMENT\\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__03_CAUSAL_TRADE_BOOK_ALIGNMENT__v0_0_synchronization_field_reconciliation.csv',
  'row_count': 0,
  'size_bytes': 196,
  'sha256': 'a02ad9e4ef3ddba79eb63014bdbf9887bd82969054d5dea29377aeb6abd33af3'},
 'v00_sync_reference_findings': {'path': 'D:\\Clown Project\\V0.1\\artifacts\\reconciliation\\03_CAUSAL_TRADE_BOOK_ALIGNMENT\\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e

In [15]:
# ============================================================
# Cell 13 — Write authoritative LOCAL_STRICT alignment outputs
# ============================================================

# This cell writes the two primary Notebook 03 data outputs:
#
#   1. all-trades LOCAL_STRICT alignment table
#   2. matched LOCAL_STRICT trade-book table
#
# These are the authoritative outputs for Notebook 04.
#
# Audit tables were already written in prior cells. This cell writes the actual
# aligned data tables and verifies read-back row counts and hashes.


# ------------------------------------------------------------
# Enrich authoritative alignment with row-level price/book diagnostics
# ------------------------------------------------------------

alignment_output = local_strict_all_alignment.copy()

price_book_enrichment_columns = [
    "trade_id",
    "price_tick_size",
    "trade_price_tick",
    "book_best_bid_tick",
    "book_best_ask_tick",
    "book_spread_ticks",
    "trade_minus_bid_ticks",
    "trade_minus_ask_ticks",
    "trade_minus_midpoint",
    "trade_minus_microprice",
    "price_vs_book_location",
    "aggressor_book_class",
    "visible_book_touch_or_through_flag",
    "inside_spread_trade_flag",
    "aggressor_book_inconsistency_flag",
    "outside_visible_book_flag",
    "wide_spread_state_match_flag",
    "price_book_warning_class",
]

if "price_vs_book_row_audit" in globals():
    require_columns(
        price_vs_book_row_audit,
        ["trade_id"],
        "price_vs_book_row_audit",
    )

    enrichment_available_columns = [
        column
        for column in price_book_enrichment_columns
        if column in price_vs_book_row_audit.columns
    ]

    price_book_enrichment = (
        price_vs_book_row_audit[enrichment_available_columns]
        .drop_duplicates(subset=["trade_id"])
        .copy()
    )

    require(
        len(price_book_enrichment) == len(local_strict_matched_alignment),
        (
            "price_book_enrichment row count must equal matched alignment row count: "
            f"observed={len(price_book_enrichment)} expected={len(local_strict_matched_alignment)}"
        ),
    )

    alignment_output = alignment_output.merge(
        price_book_enrichment,
        how="left",
        on="trade_id",
        validate="one_to_one",
    )
else:
    for column in price_book_enrichment_columns:
        if column != "trade_id":
            alignment_output[column] = pd.NA


# ------------------------------------------------------------
# Stable output schema selection
# ------------------------------------------------------------

authoritative_first_columns = [
    "source_run_prefix",
    "v0_1_run_id",
    "alignment_policy",

    "trade_id",
    "trade_raw_row_number",
    "trade_collector_sequence",
    "trade_local_receipt_time_ns",
    "trade_local_receipt_time",
    "trade_exchange_event_time_ms",
    "trade_exchange_event_time",
    "trade_exchange_trade_time_ms",
    "trade_exchange_trade_time",
    "trade_event_type",
    "trade_symbol",
    "trade_price",
    "trade_quantity",
    "trade_notional",
    "trade_buyer_is_maker",
    "trade_aggressor_side",
    "trade_partition",

    "is_local_strict_match",
    "alignment_status",
    "unmatched_reason",

    "book_state_id",
    "book_row_number",
    "book_collector_sequence",
    "book_local_receipt_time_ns",
    "book_local_receipt_time",
    "book_exchange_event_time_ms",
    "book_exchange_event_time",
    "book_first_update_id",
    "book_final_update_id",
    "book_previous_final_update_id",
    "book_best_bid",
    "book_best_ask",
    "book_best_bid_size",
    "book_best_ask_size",
    "book_spread",
    "book_midpoint",
    "book_microprice",
    "book_l1_imbalance",
    "book_top10_imbalance",
    "book_partition",
    "matched_book_partition",

    "sequence_lag",
    "local_observation_lag_ns",
    "local_observation_lag_ms",
    "exchange_time_lag_ms",
    "same_local_timestamp_flag",
    "zero_local_lag_flag",
    "above_nominal_book_interval_flag",
    "stale_match_flag",
    "very_stale_match_flag",
    "staleness_bucket",

    "partition_crossing_match_flag",

    "price_tick_size",
    "trade_price_tick",
    "book_best_bid_tick",
    "book_best_ask_tick",
    "book_spread_ticks",
    "trade_minus_bid_ticks",
    "trade_minus_ask_ticks",
    "trade_minus_midpoint",
    "trade_minus_microprice",
    "price_vs_book_location",
    "aggressor_book_class",
    "visible_book_touch_or_through_flag",
    "inside_spread_trade_flag",
    "aggressor_book_inconsistency_flag",
    "outside_visible_book_flag",
    "wide_spread_state_match_flag",
    "price_book_warning_class",

    "local_strict_sequence_condition",
    "local_strict_time_condition",
    "candidate_has_book",
    "alignment_candidate_position",
    "sequence_upper_position",
    "time_upper_position",
    "next_book_position",
    "next_book_would_be_eligible",
]

top10_columns = [
    column
    for column in alignment_output.columns
    if column.startswith("top10_")
]

source_and_trace_columns = [
    column
    for column in [
        "trade_source_file",
        "book_source_file",
        "trade_payload_path",
        "trade_payload_score",
        "trade_collector_sequence_source_path",
        "trade_local_receipt_time_ns_source_path",
        "trade_local_receipt_time_text",
        "trade_local_receipt_time_text_source_path",
        "trade_buyer_order_id",
        "trade_seller_order_id",
        "trade_ignore_flag",
    ]
    if column in alignment_output.columns
]

ordered_columns = (
    [column for column in authoritative_first_columns if column in alignment_output.columns]
    + [column for column in top10_columns if column not in authoritative_first_columns]
    + [column for column in source_and_trace_columns if column not in authoritative_first_columns and column not in top10_columns]
)

remaining_columns = [
    column
    for column in alignment_output.columns
    if column not in ordered_columns
]

ordered_columns = ordered_columns + remaining_columns

alignment_output = alignment_output[ordered_columns].copy()


# ------------------------------------------------------------
# Matched-only authoritative output
# ------------------------------------------------------------

matched_alignment_output = (
    alignment_output[
        alignment_output["is_local_strict_match"]
    ]
    .copy()
    .reset_index(drop=True)
)

all_alignment_output = alignment_output.reset_index(drop=True).copy()


# ------------------------------------------------------------
# Output gates before writing
# ------------------------------------------------------------

authoritative_output_gates: list[GateResult] = []

authoritative_output_gates.append(
    make_gate(
        gate="authoritative_all_alignment_row_count",
        passed=len(all_alignment_output) == EXPECTED_RAW_TRADE_RECORDS,
        severity="BLOCKING",
        detail=f"observed={len(all_alignment_output)} expected={EXPECTED_RAW_TRADE_RECORDS}",
    )
)

authoritative_output_gates.append(
    make_gate(
        gate="authoritative_matched_alignment_row_count",
        passed=len(matched_alignment_output) == int(local_strict_all_alignment["is_local_strict_match"].sum()),
        severity="BLOCKING",
        detail=(
            f"observed={len(matched_alignment_output)} "
            f"expected={int(local_strict_all_alignment['is_local_strict_match'].sum())}"
        ),
    )
)

authoritative_output_gates.append(
    make_gate(
        gate="authoritative_trade_id_unique_all_alignment",
        passed=not all_alignment_output["trade_id"].duplicated().any(),
        severity="BLOCKING",
        detail=f"duplicate_trade_id_count={int(all_alignment_output['trade_id'].duplicated().sum())}",
    )
)

authoritative_output_gates.append(
    make_gate(
        gate="authoritative_trade_sequence_unique_all_alignment",
        passed=not all_alignment_output["trade_collector_sequence"].duplicated().any(),
        severity="BLOCKING",
        detail=f"duplicate_trade_sequence_count={int(all_alignment_output['trade_collector_sequence'].duplicated().sum())}",
    )
)

authoritative_output_gates.append(
    make_gate(
        gate="authoritative_all_alignment_has_match_policy",
        passed=all_alignment_output["alignment_policy"].eq(PRIMARY_ALIGNMENT_POLICY).all(),
        severity="BLOCKING",
        detail=f"policy={PRIMARY_ALIGNMENT_POLICY}",
    )
)

authoritative_output_gates.append(
    make_gate(
        gate="authoritative_matched_rows_obey_sequence_causality",
        passed=(
            matched_alignment_output["book_collector_sequence"].astype("Int64")
            < matched_alignment_output["trade_collector_sequence"].astype("Int64")
        ).all(),
        severity="BLOCKING",
        detail="Requires book_collector_sequence < trade_collector_sequence.",
    )
)

authoritative_output_gates.append(
    make_gate(
        gate="authoritative_matched_rows_obey_local_time_causality",
        passed=(
            matched_alignment_output["book_local_receipt_time_ns"].astype("Int64")
            <= matched_alignment_output["trade_local_receipt_time_ns"].astype("Int64")
        ).all(),
        severity="BLOCKING",
        detail="Requires book_local_receipt_time_ns <= trade_local_receipt_time_ns.",
    )
)

authoritative_output_gates.append(
    make_gate(
        gate="authoritative_matched_rows_have_nonnegative_local_lag",
        passed=(matched_alignment_output["local_observation_lag_ns"].astype("Int64") >= 0).all(),
        severity="BLOCKING",
        detail="Requires local_observation_lag_ns >= 0.",
    )
)

authoritative_output_gates.append(
    make_gate(
        gate="authoritative_output_contains_trade_partition",
        passed=all_alignment_output["trade_partition"].notna().all(),
        severity="BLOCKING",
        detail=f"null_trade_partition={int(all_alignment_output['trade_partition'].isna().sum())}",
    )
)

authoritative_output_gates.append(
    make_gate(
        gate="authoritative_output_contains_price_book_classification",
        passed=matched_alignment_output["aggressor_book_class"].notna().all(),
        severity="BLOCKING",
        detail=f"null_aggressor_book_class={int(matched_alignment_output['aggressor_book_class'].isna().sum())}",
    )
)

authoritative_output_gate_frame = gate_results_to_frame(authoritative_output_gates)

fail_if_blocking_gate_failed(authoritative_output_gate_frame)


# ------------------------------------------------------------
# Write authoritative outputs and verify read-back
# ------------------------------------------------------------

local_strict_all_alignment_metadata = write_csv_and_verify(
    all_alignment_output,
    LOCAL_STRICT_ALL_ALIGNMENT_PATH,
    expected_rows=len(all_alignment_output),
)

local_strict_matched_alignment_metadata = write_csv_and_verify(
    matched_alignment_output,
    LOCAL_STRICT_MATCHED_ALIGNMENT_PATH,
    expected_rows=len(matched_alignment_output),
)

authoritative_alignment_output_metadata = {
    "local_strict_all_trades_alignment": local_strict_all_alignment_metadata,
    "local_strict_matched_trade_book": local_strict_matched_alignment_metadata,
}


# ------------------------------------------------------------
# Verify core columns after read-back
# ------------------------------------------------------------

all_alignment_readback = pd.read_csv(
    LOCAL_STRICT_ALL_ALIGNMENT_PATH,
    low_memory=False,
    usecols=[
        "trade_id",
        "trade_collector_sequence",
        "book_collector_sequence",
        "trade_local_receipt_time_ns",
        "book_local_receipt_time_ns",
        "is_local_strict_match",
        "alignment_policy",
    ],
)

matched_alignment_readback = pd.read_csv(
    LOCAL_STRICT_MATCHED_ALIGNMENT_PATH,
    low_memory=False,
    usecols=[
        "trade_id",
        "trade_collector_sequence",
        "book_collector_sequence",
        "trade_local_receipt_time_ns",
        "book_local_receipt_time_ns",
        "is_local_strict_match",
        "alignment_policy",
    ],
)

readback_gates: list[GateResult] = []

readback_gates.append(
    make_gate(
        gate="all_alignment_readback_row_count",
        passed=len(all_alignment_readback) == EXPECTED_RAW_TRADE_RECORDS,
        severity="BLOCKING",
        detail=f"observed={len(all_alignment_readback)} expected={EXPECTED_RAW_TRADE_RECORDS}",
    )
)

readback_gates.append(
    make_gate(
        gate="matched_alignment_readback_row_count",
        passed=len(matched_alignment_readback) == len(matched_alignment_output),
        severity="BLOCKING",
        detail=f"observed={len(matched_alignment_readback)} expected={len(matched_alignment_output)}",
    )
)

readback_gates.append(
    make_gate(
        gate="all_alignment_readback_policy",
        passed=all_alignment_readback["alignment_policy"].eq(PRIMARY_ALIGNMENT_POLICY).all(),
        severity="BLOCKING",
        detail=f"policy={PRIMARY_ALIGNMENT_POLICY}",
    )
)

readback_gates.append(
    make_gate(
        gate="matched_alignment_readback_policy",
        passed=matched_alignment_readback["alignment_policy"].eq(PRIMARY_ALIGNMENT_POLICY).all(),
        severity="BLOCKING",
        detail=f"policy={PRIMARY_ALIGNMENT_POLICY}",
    )
)

readback_gates.append(
    make_gate(
        gate="matched_alignment_readback_all_matched",
        passed=matched_alignment_readback["is_local_strict_match"].astype(bool).all(),
        severity="BLOCKING",
        detail="Matched output must contain matched rows only.",
    )
)

readback_gate_frame = gate_results_to_frame(readback_gates)

fail_if_blocking_gate_failed(readback_gate_frame)


# ------------------------------------------------------------
# Compact display
# ------------------------------------------------------------

authoritative_output_summary = pd.DataFrame(
    [
        {
            "output_name": "local_strict_all_trades_alignment",
            "path": local_strict_all_alignment_metadata["path"],
            "row_count": local_strict_all_alignment_metadata["row_count"],
            "column_count": int(all_alignment_output.shape[1]),
            "size_bytes": local_strict_all_alignment_metadata["size_bytes"],
            "sha256": local_strict_all_alignment_metadata["sha256"],
        },
        {
            "output_name": "local_strict_matched_trade_book",
            "path": local_strict_matched_alignment_metadata["path"],
            "row_count": local_strict_matched_alignment_metadata["row_count"],
            "column_count": int(matched_alignment_output.shape[1]),
            "size_bytes": local_strict_matched_alignment_metadata["size_bytes"],
            "sha256": local_strict_matched_alignment_metadata["sha256"],
        },
    ]
)

display(authoritative_output_summary)
display(authoritative_output_gate_frame)
display(readback_gate_frame)

authoritative_alignment_output_metadata

,output_name,path,row_count,column_count,size_bytes,sha256
0,local_strict_all_trades_alignment,D:\Clown Project\V0.1\data\processed\trade_book_alignment\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__03_CAUSAL_TRADE_BOOK_A...,67683,148,131540683,a18f66ca6a97a66b8f42a4d61b2aa34d26bcee5ae5a8d2054e576c88beede7a3
1,local_strict_matched_trade_book,D:\Clown Project\V0.1\data\processed\trade_book_alignment\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__03_CAUSAL_TRADE_BOOK_A...,67683,148,131540683,a18f66ca6a97a66b8f42a4d61b2aa34d26bcee5ae5a8d2054e576c88beede7a3


,gate,status,severity,detail
0,authoritative_all_alignment_row_count,PASS,BLOCKING,observed=67683 expected=67683
1,authoritative_matched_alignment_row_count,PASS,BLOCKING,observed=67683 expected=67683
2,authoritative_trade_id_unique_all_alignment,PASS,BLOCKING,duplicate_trade_id_count=0
3,authoritative_trade_sequence_unique_all_alignment,PASS,BLOCKING,duplicate_trade_sequence_count=0
4,authoritative_all_alignment_has_match_policy,PASS,BLOCKING,policy=LOCAL_STRICT
5,authoritative_matched_rows_obey_sequence_causality,PASS,BLOCKING,Requires book_collector_sequence < trade_collector_sequence.
6,authoritative_matched_rows_obey_local_time_causality,PASS,BLOCKING,Requires book_local_receipt_time_ns <= trade_local_receipt_time_ns.
7,authoritative_matched_rows_have_nonnegative_local_lag,PASS,BLOCKING,Requires local_observation_lag_ns >= 0.
8,authoritative_output_contains_trade_partition,PASS,BLOCKING,null_trade_partition=0
9,authoritative_output_contains_price_book_classification,PASS,BLOCKING,null_aggressor_book_class=0


,gate,status,severity,detail
0,all_alignment_readback_row_count,PASS,BLOCKING,observed=67683 expected=67683
1,matched_alignment_readback_row_count,PASS,BLOCKING,observed=67683 expected=67683
2,all_alignment_readback_policy,PASS,BLOCKING,policy=LOCAL_STRICT
3,matched_alignment_readback_policy,PASS,BLOCKING,policy=LOCAL_STRICT
4,matched_alignment_readback_all_matched,PASS,BLOCKING,Matched output must contain matched rows only.


{'local_strict_all_trades_alignment': {'path': 'D:\\Clown Project\\V0.1\\data\\processed\\trade_book_alignment\\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__03_CAUSAL_TRADE_BOOK_ALIGNMENT__local_strict_all_trades_alignment.csv',
  'row_count': 67683,
  'size_bytes': 131540683,
  'sha256': 'a18f66ca6a97a66b8f42a4d61b2aa34d26bcee5ae5a8d2054e576c88beede7a3'},
 'local_strict_matched_trade_book': {'path': 'D:\\Clown Project\\V0.1\\data\\processed\\trade_book_alignment\\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__03_CAUSAL_TRADE_BOOK_ALIGNMENT__local_strict_matched_trade_book.csv',
  'row_count': 67683,
  'size_bytes': 131540683,
  'sha256': 'a18f66ca6a97a66b8f42a4d61b2aa34d26bcee5ae5a8d2054e576c88beede7a3'}}

In [16]:
# ============================================================
# Cell 14 — Notebook 03 manifest and Notebook 04 handoff
# ============================================================

# This cell writes:
#
#   1. Notebook 03 output manifest
#   2. Notebook 03 to Notebook 04 handoff
#
# Notebook 04 is authorized only if every BLOCKING gate in Notebook 03 passed.
# WARNING gates are preserved as warnings and do not block the handoff.


# ------------------------------------------------------------
# Collect all gate frames
# ------------------------------------------------------------

gate_frame_names = [
    "input_contract_gate_frame",
    "upstream_payload_gate_frame",
    "raw_trade_gate_frame",
    "book_state_gate_frame",
    "local_strict_alignment_gate_frame",
    "lag_staleness_sequence_gate_frame",
    "price_book_gate_frame",
    "partition_alignment_gate_frame",
    "exchange_strict_gate_frame",
    "v00_reference_reconciliation_gate_frame",
    "authoritative_output_gate_frame",
    "readback_gate_frame",
]

available_gate_frames = []

for frame_name in gate_frame_names:
    if frame_name in globals():
        frame = globals()[frame_name].copy()
        frame.insert(0, "gate_frame_name", frame_name)
        available_gate_frames.append(frame)

require(
    len(available_gate_frames) == len(gate_frame_names),
    (
        "Missing one or more expected gate frames: "
        f"missing={[name for name in gate_frame_names if name not in globals()]}"
    ),
)

notebook_03_all_gates = pd.concat(
    available_gate_frames,
    ignore_index=True,
)

require_columns(
    notebook_03_all_gates,
    ["gate_frame_name", "gate", "status", "severity", "detail"],
    "notebook_03_all_gates",
)

blocking_gate_failures = notebook_03_all_gates[
    (notebook_03_all_gates["severity"] == "BLOCKING")
    & (notebook_03_all_gates["status"] != "PASS")
].copy()

warning_gate_failures = notebook_03_all_gates[
    (notebook_03_all_gates["severity"] == "WARNING")
    & (notebook_03_all_gates["status"] != "PASS")
].copy()

info_gate_rows = notebook_03_all_gates[
    notebook_03_all_gates["severity"] == "INFO"
].copy()

require(
    blocking_gate_failures.empty,
    f"Notebook 03 cannot write handoff because blocking gates failed: {len(blocking_gate_failures)}",
)


# ------------------------------------------------------------
# Terminal status
# ------------------------------------------------------------

alignment_warning_gate_names = {
    "local_strict_unmatched_count",
    "local_strict_stale_match_count",
    "no_unmatched_rows",
    "price_book_inside_spread_trades",
    "price_book_aggressor_inconsistencies",
    "price_book_outside_visible_book_trades",
    "price_book_wide_spread_state_matches",
    "partition_crossing_matches_present",
    "exchange_strict_unmatched_count",
    "exchange_strict_differs_from_local_strict",
    "exchange_strict_locally_noncausal_matches",
}

reference_warning_gate_names = {
    "payload_hash_contract_traceable::notebook_01_raw_data_audit",
    "payload_hash_contract_traceable::notebook_02_output_manifest",
    "payload_hash_contract_traceable::notebook_02_to_03_handoff",
    "book_top10_wide_enrichment",
    "v00_reference_selected",
}

failed_warning_gate_set = set(warning_gate_failures["gate"].astype(str))

has_alignment_warnings = bool(failed_warning_gate_set & alignment_warning_gate_names)
has_reference_warnings = bool(failed_warning_gate_set & reference_warning_gate_names)

if not warning_gate_failures.empty and has_alignment_warnings:
    NOTEBOOK_03_TERMINAL_STATUS = "PASS_WITH_ALIGNMENT_WARNINGS"
elif not warning_gate_failures.empty and has_reference_warnings:
    NOTEBOOK_03_TERMINAL_STATUS = "PASS_WITH_REFERENCE_WARNINGS"
else:
    NOTEBOOK_03_TERMINAL_STATUS = "PASS"

NOTEBOOK_04_AUTHORIZED = blocking_gate_failures.empty


# ------------------------------------------------------------
# Collect output metadata
# ------------------------------------------------------------

output_metadata_sources = {
    "authoritative_alignment_outputs": authoritative_alignment_output_metadata,
    "lag_staleness_sequence_outputs": lag_staleness_sequence_output_metadata,
    "price_book_outputs": price_book_output_metadata,
    "partition_boundary_outputs": partition_boundary_output_metadata,
    "exchange_strict_outputs": exchange_strict_output_metadata,
    "v00_reference_reconciliation_outputs": v00_reference_reconciliation_output_metadata,
}

manifest_output_artifacts = []

for group_name, metadata_group in output_metadata_sources.items():
    for artifact_name, metadata in metadata_group.items():
        manifest_output_artifacts.append(
            {
                "group": group_name,
                "artifact_name": artifact_name,
                "path": metadata["path"],
                "row_count": int(metadata["row_count"]),
                "size_bytes": int(metadata["size_bytes"]),
                "sha256": metadata["sha256"],
            }
        )

manifest_output_artifacts_frame = pd.DataFrame(manifest_output_artifacts)

require(
    not manifest_output_artifacts_frame.empty,
    "Manifest output artifact list is empty.",
)

require(
    manifest_output_artifacts_frame["path"].is_unique,
    "Manifest output artifact paths must be unique.",
)

for path_string in manifest_output_artifacts_frame["path"]:
    require_file(Path(path_string))


# ------------------------------------------------------------
# Input artifact metadata
# ------------------------------------------------------------

manifest_input_artifacts = []

for _, row in input_contract_audit.iterrows():
    manifest_input_artifacts.append(
        {
            "logical_name": row["logical_name"],
            "role": row["role"],
            "path": row["path"],
            "size_bytes": None if pd.isna(row["size_bytes"]) else int(row["size_bytes"]),
            "observed_file_sha256": row["observed_file_sha256"],
            "expected_sha256": row["expected_sha256"],
            "hash_contract_status": row["hash_contract_status"],
            "overall_status": row["overall_status"],
        }
    )


# ------------------------------------------------------------
# Counts and diagnostic summaries for handoff
# ------------------------------------------------------------

price_book_warning_counts = {
    "inside_spread_trade_count": int(price_vs_book_row_audit["inside_spread_trade_flag"].sum()),
    "aggressor_book_inconsistency_count": int(price_vs_book_row_audit["aggressor_book_inconsistency_flag"].sum()),
    "outside_visible_book_count": int(price_vs_book_row_audit["outside_visible_book_flag"].sum()),
    "wide_spread_state_match_count": int(price_vs_book_row_audit["wide_spread_state_match_flag"].sum()),
}

exchange_strict_sensitivity_counts = {
    "same_as_local_strict_count": int(exchange_strict_alignment_sensitivity["same_book_as_local_strict"].sum()),
    "different_from_local_strict_count": int(exchange_strict_alignment_sensitivity["different_book_from_local_strict"].sum()),
    "locally_noncausal_count": int(exchange_strict_alignment_sensitivity["exchange_strict_any_local_noncausal_flag"].sum()),
}

alignment_handoff_counts = {
    "raw_trade_count": int(len(raw_trades)),
    "book_state_count": int(len(book_states)),
    "local_strict_all_alignment_rows": int(len(all_alignment_output)),
    "local_strict_matched_rows": int(len(matched_alignment_output)),
    "local_strict_unmatched_rows": int(len(unmatched_trades)),
    "buy_trade_count": int((all_alignment_output["trade_aggressor_side"] == "BUY").sum()),
    "sell_trade_count": int((all_alignment_output["trade_aggressor_side"] == "SELL").sum()),
    "zero_local_lag_count": int(all_alignment_output["zero_local_lag_flag"].sum()),
    "above_nominal_book_interval_count": int(all_alignment_output["above_nominal_book_interval_flag"].sum()),
    "stale_match_count": int(all_alignment_output["stale_match_flag"].sum()),
    "very_stale_match_count": int(all_alignment_output["very_stale_match_flag"].sum()),
    "partition_crossing_match_count": int(all_alignment_output["partition_crossing_match_flag"].sum()),
    **price_book_warning_counts,
    **exchange_strict_sensitivity_counts,
}

local_lag_handoff_summary = {
    "local_observation_lag_ms_min": safe_min(all_alignment_output["local_observation_lag_ms"]),
    "local_observation_lag_ms_p01": safe_quantile(all_alignment_output["local_observation_lag_ms"], 0.01),
    "local_observation_lag_ms_p05": safe_quantile(all_alignment_output["local_observation_lag_ms"], 0.05),
    "local_observation_lag_ms_p25": safe_quantile(all_alignment_output["local_observation_lag_ms"], 0.25),
    "local_observation_lag_ms_p50": safe_quantile(all_alignment_output["local_observation_lag_ms"], 0.50),
    "local_observation_lag_ms_p75": safe_quantile(all_alignment_output["local_observation_lag_ms"], 0.75),
    "local_observation_lag_ms_p95": safe_quantile(all_alignment_output["local_observation_lag_ms"], 0.95),
    "local_observation_lag_ms_p99": safe_quantile(all_alignment_output["local_observation_lag_ms"], 0.99),
    "local_observation_lag_ms_max": safe_max(all_alignment_output["local_observation_lag_ms"]),
}

event_stream_authority = {
    "primary_event_time_field_for_notebook_04": "trade_local_receipt_time_ns",
    "secondary_exchange_trade_time_field": "trade_exchange_trade_time_ms",
    "primary_ordering_field": "trade_collector_sequence",
    "trade_identity_field": "trade_id",
    "aggressor_side_field": "trade_aggressor_side",
    "price_field": "trade_price",
    "quantity_field": "trade_quantity",
    "notional_field": "trade_notional",
    "matched_book_sequence_field": "book_collector_sequence",
    "matched_book_local_time_field": "book_local_receipt_time_ns",
    "alignment_policy": PRIMARY_ALIGNMENT_POLICY,
    "event_stream_instruction": (
        "Notebook 04 may aggregate trades into event streams, including individual prints "
        "and same-millisecond same-side bursts, using this LOCAL_STRICT matched table. "
        "Notebook 04 must not use EXCHANGE_STRICT as primary authority."
    ),
}


# ------------------------------------------------------------
# Build manifest payload
# ------------------------------------------------------------

notebook_03_completed_at_utc = utc_now_iso()

notebook_03_manifest_payload = {
    "project_name": PROJECT_NAME,
    "pipeline_version": PIPELINE_VERSION,
    "notebook_name": NOTEBOOK_NAME,
    "notebook_filename": NOTEBOOK_FILENAME,
    "source_run_prefix": SOURCE_RUN_PREFIX,
    "v0_1_run_id": V01_RUN_ID,
    "combined_prefix": COMBINED_PREFIX,
    "operating_mode": OPERATING_MODE,
    "primary_ordering_authority": PRIMARY_ORDERING_AUTHORITY,
    "primary_alignment_policy": PRIMARY_ALIGNMENT_POLICY,
    "sensitivity_alignment_policy": SENSITIVITY_ALIGNMENT_POLICY,
    "canonical_timezone": CANONICAL_TIMEZONE,
    "started_at_utc": runtime_record["started_at_utc"],
    "completed_at_utc": notebook_03_completed_at_utc,
    "terminal_status": NOTEBOOK_03_TERMINAL_STATUS,
    "notebook_04_authorized": NOTEBOOK_04_AUTHORIZED,

    "expected_counts": {
        "raw_trade_records": EXPECTED_RAW_TRADE_RECORDS,
        "raw_depth_records": EXPECTED_RAW_DEPTH_RECORDS,
        "reconstructed_book_states": EXPECTED_RECONSTRUCTED_BOOK_STATES,
        "top10_wide_rows": EXPECTED_TOP10_WIDE_ROWS,
        "top10_long_rows": EXPECTED_TOP10_LONG_ROWS,
    },

    "observed_counts": alignment_handoff_counts,
    "local_lag_summary": local_lag_handoff_summary,

    "input_artifacts": manifest_input_artifacts,
    "output_artifacts": manifest_output_artifacts,

    "gate_summary": {
        "gate_frame_count": len(available_gate_frames),
        "total_gate_count": int(len(notebook_03_all_gates)),
        "blocking_gate_count": int((notebook_03_all_gates["severity"] == "BLOCKING").sum()),
        "blocking_gate_failure_count": int(len(blocking_gate_failures)),
        "warning_gate_count": int((notebook_03_all_gates["severity"] == "WARNING").sum()),
        "warning_gate_failure_count": int(len(warning_gate_failures)),
        "info_gate_count": int(len(info_gate_rows)),
    },

    "warning_findings": warning_gate_failures[
        ["gate_frame_name", "gate", "status", "severity", "detail"]
    ].to_dict(orient="records"),

    "reference_reconciliation_status": {
        "v00_reference_status": v00_sync_reference_load_status,
        "v00_reference_selected": v00_sync_reference_path is not None,
        "v00_reference_path": None if v00_sync_reference_path is None else str(v00_sync_reference_path),
        "v00_reference_findings": v00_sync_reference_findings.to_dict(orient="records"),
    },

    "authority_limits": {
        "authorized": [
            "LOCAL_STRICT trade-book alignment",
            "matched trade-book table for Notebook 04",
            "all raw trades accounted for",
            "causal sequence and local-receipt-time constraints",
            "lag and staleness diagnostics",
            "price-versus-book diagnostics",
            "aggressor-versus-book diagnostics",
            "partition alignment summary",
            "EXCHANGE_STRICT sensitivity only",
        ],
        "not_authorized": [
            "event aggregation",
            "Hawkes event arrays",
            "market-state feature tables",
            "Poisson baseline model",
            "Hawkes model estimation",
            "Hawkes superiority claim",
            "quote logic",
            "fill simulation",
            "inventory accounting",
            "cash accounting",
            "P&L",
            "Sharpe ratio",
            "maximum drawdown",
            "live deployment",
        ],
    },

    "runtime_record": runtime_record,
}


# ------------------------------------------------------------
# Build Notebook 04 handoff payload
# ------------------------------------------------------------

notebook_03_to_04_handoff_payload = {
    "project_name": PROJECT_NAME,
    "pipeline_version": PIPELINE_VERSION,
    "source_run_prefix": SOURCE_RUN_PREFIX,
    "v0_1_run_id": V01_RUN_ID,
    "combined_prefix": COMBINED_PREFIX,

    "producing_notebook": NOTEBOOK_NAME,
    "next_notebook": "04_EVENT_STREAM_CONSTRUCTION",
    "created_at_utc": notebook_03_completed_at_utc,

    "terminal_status": NOTEBOOK_03_TERMINAL_STATUS,
    "notebook_04_authorized": NOTEBOOK_04_AUTHORIZED,

    "primary_alignment_policy": PRIMARY_ALIGNMENT_POLICY,
    "primary_ordering_authority": PRIMARY_ORDERING_AUTHORITY,
    "canonical_timezone": CANONICAL_TIMEZONE,

    "authoritative_inputs_for_notebook_04": {
        "local_strict_matched_trade_book": local_strict_matched_alignment_metadata,
        "local_strict_all_trades_alignment": local_strict_all_alignment_metadata,
        "unmatched_trades": unmatched_trades_metadata,
        "match_lag_distribution": match_lag_distribution_metadata,
        "staleness_audit": staleness_audit_metadata,
        "sequence_order_audit": sequence_order_audit_metadata,
        "price_vs_book_consistency_audit": price_vs_book_consistency_metadata,
        "aggressor_vs_book_audit": aggressor_vs_book_metadata,
        "partition_alignment_summary": partition_alignment_summary_metadata,
        "boundary_alignment_audit": boundary_alignment_audit_metadata,
    },

    "sensitivity_only_inputs": {
        "exchange_strict_alignment_sensitivity": exchange_strict_sensitivity_metadata,
    },

    "counts": alignment_handoff_counts,
    "local_lag_summary": local_lag_handoff_summary,
    "event_stream_authority": event_stream_authority,

    "partition_summary": partition_alignment_summary.to_dict(orient="records"),
    "staleness_summary": staleness_audit.to_dict(orient="records"),
    "aggressor_summary": aggressor_vs_book_audit.to_dict(orient="records"),
    "price_book_warning_counts": price_book_warning_counts,
    "exchange_strict_sensitivity_counts": exchange_strict_sensitivity_counts,

    "required_notebook_04_gates": [
        "Read LOCAL_STRICT matched trade-book table from disk.",
        "Verify matched trade row count equals 67,683.",
        "Verify trade_id uniqueness.",
        "Verify trade_collector_sequence uniqueness and strict chronological order.",
        "Verify aggressor side is BUY or SELL for every row.",
        "Verify event aggregation conserves trade count.",
        "Verify event aggregation conserves total quantity and notional by side.",
        "Do not use EXCHANGE_STRICT as primary authority.",
        "Do not use V0.0 point-process reference files as authority.",
        "Write individual-print event stream and same-ms same-side burst event stream.",
    ],

    "warnings_to_carry_forward": warning_gate_failures[
        ["gate_frame_name", "gate", "status", "severity", "detail"]
    ].to_dict(orient="records"),

    "notebook_03_manifest_path": str(NOTEBOOK_03_OUTPUT_MANIFEST_PATH),
}


# ------------------------------------------------------------
# Write JSON outputs
# ------------------------------------------------------------

write_json_file(
    notebook_03_manifest_payload,
    NOTEBOOK_03_OUTPUT_MANIFEST_PATH,
)

notebook_03_manifest_metadata = summarize_path(NOTEBOOK_03_OUTPUT_MANIFEST_PATH)

write_json_file(
    notebook_03_to_04_handoff_payload,
    NOTEBOOK_03_TO_04_HANDOFF_PATH,
)

notebook_03_to_04_handoff_metadata = summarize_path(NOTEBOOK_03_TO_04_HANDOFF_PATH)


# ------------------------------------------------------------
# Read-back verification
# ------------------------------------------------------------

manifest_readback = read_json_file(NOTEBOOK_03_OUTPUT_MANIFEST_PATH)
handoff_readback = read_json_file(NOTEBOOK_03_TO_04_HANDOFF_PATH)

manifest_handoff_gates: list[GateResult] = []

manifest_handoff_gates.append(
    make_gate(
        gate="notebook_03_manifest_written",
        passed=Path(NOTEBOOK_03_OUTPUT_MANIFEST_PATH).is_file(),
        severity="BLOCKING",
        detail=str(NOTEBOOK_03_OUTPUT_MANIFEST_PATH),
    )
)

manifest_handoff_gates.append(
    make_gate(
        gate="notebook_03_handoff_written",
        passed=Path(NOTEBOOK_03_TO_04_HANDOFF_PATH).is_file(),
        severity="BLOCKING",
        detail=str(NOTEBOOK_03_TO_04_HANDOFF_PATH),
    )
)

manifest_handoff_gates.append(
    make_gate(
        gate="manifest_readback_identity",
        passed=(
            manifest_readback.get("source_run_prefix") == SOURCE_RUN_PREFIX
            and manifest_readback.get("v0_1_run_id") == V01_RUN_ID
            and manifest_readback.get("notebook_name") == NOTEBOOK_NAME
        ),
        severity="BLOCKING",
        detail="Manifest identity fields must match current run.",
    )
)

manifest_handoff_gates.append(
    make_gate(
        gate="handoff_readback_identity",
        passed=(
            handoff_readback.get("source_run_prefix") == SOURCE_RUN_PREFIX
            and handoff_readback.get("v0_1_run_id") == V01_RUN_ID
            and handoff_readback.get("producing_notebook") == NOTEBOOK_NAME
            and handoff_readback.get("next_notebook") == "04_EVENT_STREAM_CONSTRUCTION"
        ),
        severity="BLOCKING",
        detail="Handoff identity fields must match current run.",
    )
)

manifest_handoff_gates.append(
    make_gate(
        gate="handoff_authorizes_notebook_04",
        passed=bool(handoff_readback.get("notebook_04_authorized")) is True,
        severity="BLOCKING",
        detail=f"notebook_04_authorized={handoff_readback.get('notebook_04_authorized')}",
    )
)

manifest_handoff_gates.append(
    make_gate(
        gate="manifest_terminal_status_consistent",
        passed=manifest_readback.get("terminal_status") == NOTEBOOK_03_TERMINAL_STATUS,
        severity="BLOCKING",
        detail=f"terminal_status={NOTEBOOK_03_TERMINAL_STATUS}",
    )
)

manifest_handoff_gate_frame = gate_results_to_frame(manifest_handoff_gates)

fail_if_blocking_gate_failed(manifest_handoff_gate_frame)


# ------------------------------------------------------------
# Compact display
# ------------------------------------------------------------

notebook_03_final_summary = pd.DataFrame(
    [
        {
            "notebook_name": NOTEBOOK_NAME,
            "terminal_status": NOTEBOOK_03_TERMINAL_STATUS,
            "notebook_04_authorized": NOTEBOOK_04_AUTHORIZED,
            "blocking_gate_failures": int(len(blocking_gate_failures)),
            "warning_gate_failures": int(len(warning_gate_failures)),
            "raw_trade_count": int(len(raw_trades)),
            "matched_trade_book_rows": int(len(matched_alignment_output)),
            "unmatched_rows": int(len(unmatched_trades)),
            "manifest_sha256": notebook_03_manifest_metadata["sha256"],
            "handoff_sha256": notebook_03_to_04_handoff_metadata["sha256"],
        }
    ]
)

manifest_handoff_output_metadata = {
    "notebook_03_output_manifest": notebook_03_manifest_metadata,
    "notebook_03_to_04_handoff": notebook_03_to_04_handoff_metadata,
}

display(notebook_03_final_summary)
display(warning_gate_failures[["gate_frame_name", "gate", "status", "severity", "detail"]])
display(manifest_handoff_gate_frame)

manifest_handoff_output_metadata

,notebook_name,terminal_status,notebook_04_authorized,blocking_gate_failures,warning_gate_failures,raw_trade_count,matched_trade_book_rows,unmatched_rows,manifest_sha256,handoff_sha256
0,03_CAUSAL_TRADE_BOOK_ALIGNMENT,PASS_WITH_ALIGNMENT_WARNINGS,True,0,6,67683,67683,0,4daa7fa17c2f953e3b931b607e1b5b066f4e29e7dea7397d86b3c9e3b77406ca,6f4d48e4fbb56d9ac80b415b3c0fe6c4625526b555938b846ee2876626b6152e


,gate_frame_name,gate,status,severity,detail
98,price_book_gate_frame,price_book_inside_spread_trades,FAIL,WARNING,inside_spread_trade_count=67
99,price_book_gate_frame,price_book_aggressor_inconsistencies,FAIL,WARNING,aggressor_book_inconsistency_count=818
100,price_book_gate_frame,price_book_outside_visible_book_trades,FAIL,WARNING,outside_visible_book_count=30926
101,price_book_gate_frame,price_book_wide_spread_state_matches,FAIL,WARNING,wide_spread_state_match_count=95
111,exchange_strict_gate_frame,exchange_strict_differs_from_local_strict,FAIL,WARNING,different_book_from_local_strict_count=1510
115,v00_reference_reconciliation_gate_frame,v00_reference_selected,FAIL,WARNING,NO_LOADABLE_TRADE_LIKE_REFERENCE_SELECTED


,gate,status,severity,detail
0,notebook_03_manifest_written,PASS,BLOCKING,D:\Clown Project\V0.1\artifacts\manifests\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__03_CAUSAL_TRADE_BOOK_ALIGNMENT__notebo...
1,notebook_03_handoff_written,PASS,BLOCKING,D:\Clown Project\V0.1\artifacts\handoff\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__03_CAUSAL_TRADE_BOOK_ALIGNMENT__notebook...
2,manifest_readback_identity,PASS,BLOCKING,Manifest identity fields must match current run.
3,handoff_readback_identity,PASS,BLOCKING,Handoff identity fields must match current run.
4,handoff_authorizes_notebook_04,PASS,BLOCKING,notebook_04_authorized=True
5,manifest_terminal_status_consistent,PASS,BLOCKING,terminal_status=PASS_WITH_ALIGNMENT_WARNINGS


{'notebook_03_output_manifest': {'path': 'D:\\Clown Project\\V0.1\\artifacts\\manifests\\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__03_CAUSAL_TRADE_BOOK_ALIGNMENT__notebook_03_output_manifest.json',
  'exists': True,
  'size_bytes': 19576,
  'sha256': '4daa7fa17c2f953e3b931b607e1b5b066f4e29e7dea7397d86b3c9e3b77406ca'},
 'notebook_03_to_04_handoff': {'path': 'D:\\Clown Project\\V0.1\\artifacts\\handoff\\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__03_CAUSAL_TRADE_BOOK_ALIGNMENT__notebook_03_to_notebook_04_handoff.json',
  'exists': True,
  'size_bytes': 28856,
  'sha256': '6f4d48e4fbb56d9ac80b415b3c0fe6c4625526b555938b846ee2876626b6152e'}}